In [29]:
#1
!pip install -q spacy sentence-transformers gradio scikit-learn
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [30]:
#2
import re, json, os, glob, uuid, html, datetime
import numpy as np
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

nlp = spacy.load("en_core_web_sm")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

SAVE_DIR = "/content/ux_projects"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Gradio version:", gr.__version__)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Gradio version: 6.31.0


In [31]:
#3
DOMAINS = {
 "food_ordering": {
  "label": "Food ordering",
  "description": "App or website for ordering food from restaurants, browsing menus, paying and tracking delivery.",
  "keywords": ["food","restaurant","delivery","menu","meal","cuisine","dish","cafe","takeaway","canteen","swiggy","zomato"],
  "pages": ["Home","Restaurant List","Menu","Cart","Checkout","Order Tracking","Profile"],
  "features": {
    "Search & filters": "search restaurants and dishes, filter by cuisine, price and rating",
    "Menu browsing": "browse menus with photos, categories and prices",
    "Cart & customization": "add items to cart, customize toppings, edit quantities",
    "Online payment": "pay online with cards, UPI, wallets or cash on delivery",
    "Live order tracking": "track order status and delivery partner on a live map",
    "Ratings & reviews": "rate and review restaurants and dishes",
    "Saved addresses & favorites": "save delivery addresses and favorite restaurants",
    "Offers & promo codes": "apply coupons, discounts and promotional offers",
    "Reorder": "quickly reorder previous orders"},
  "journey": ["Open app","Find a restaurant or dish","View menu","Add items to cart","Checkout and pay","Track delivery","Rate the order"],
  "tips": ["Show delivery time and fees early, before the cart.","Keep the add-to-cart action one tap from the dish card.","Show real-time order status to reduce anxiety.","Make reordering and saved addresses effortless."]},
 "ecommerce": {
  "label": "E-commerce",
  "description": "Online store for browsing products, comparing, adding to cart, checking out and tracking orders.",
  "keywords": ["shop","store","ecommerce","e-commerce","product","cart","buy","sell","marketplace","fashion","retail","clothing","catalog"],
  "pages": ["Home","Product List","Product Detail","Cart","Checkout","Order History","Wishlist","Profile"],
  "features": {
    "Search & filters": "search products, filter and sort by price, brand, size and rating",
    "Product gallery": "high quality product images, zoom and variant selection",
    "Cart": "add to cart, update quantities, save for later",
    "Secure checkout": "secure checkout with multiple payment methods and guest checkout",
    "Reviews & ratings": "customer reviews and ratings on products",
    "Wishlist": "save products to a wishlist for later",
    "Coupons & discounts": "discount codes, offers and sales banners",
    "Order tracking": "track shipments and manage returns",
    "Recommendations": "personalized product recommendations"},
  "journey": ["Land on home","Search or browse","Compare products","View product detail","Add to cart","Checkout and pay","Track order"],
  "tips": ["Make filters and sorting prominent on listing pages.","Show price, shipping and returns clearly before checkout.","Offer guest checkout to reduce abandonment.","Use trust signals: reviews, secure payment badges."]},
 "education": {
  "label": "Education / e-learning",
  "description": "Learning platform with courses, video lessons, quizzes, progress tracking for students and teachers.",
  "keywords": ["course","learn","student","teacher","school","lesson","quiz","education","e-learning","elearning","tutor","classroom","exam","university"],
  "pages": ["Home","Course Catalog","Course Detail","Lesson Player","Quiz","Dashboard","Profile"],
  "features": {
    "Course search": "search and filter courses by topic, level and rating",
    "Video lessons": "watch video lessons with playback controls and notes",
    "Quizzes & assignments": "take quizzes, submit assignments and get feedback",
    "Progress tracking": "track learning progress, streaks and completion",
    "Certificates": "earn certificates after completing courses",
    "Discussion forums": "discuss with peers and instructors in forums or comments",
    "Instructor tools": "instructors upload content and view student analytics",
    "Reminders": "notifications and reminders for lessons and deadlines"},
  "journey": ["Sign up","Discover a course","Enroll","Watch lessons","Take quiz","Track progress","Get certificate"],
  "tips": ["Always show where the learner is and what is next.","Keep lesson pages distraction-free.","Use progress bars and small wins for motivation.","Support different learning paces; allow resume."]},
 "healthcare": {
  "label": "Healthcare",
  "description": "Healthcare app to find doctors, book appointments, consult online and manage medical records.",
  "keywords": ["doctor","patient","hospital","clinic","appointment","health","medical","medicine","telemedicine","pharmacy","prescription"],
  "pages": ["Home","Find Doctor","Doctor Profile","Book Appointment","Dashboard","Medical Records","Telehealth","Profile"],
  "features": {
    "Doctor search": "search doctors by specialty, location, availability and fees",
    "Appointment booking": "book, reschedule and cancel appointments with time slots",
    "Telemedicine": "video consultation and chat with doctors",
    "Medical records": "store and view medical records, reports and prescriptions securely",
    "Medication reminders": "reminders for medicines and follow-ups",
    "Privacy & security": "secure login and privacy of patient data",
    "Billing & payments": "pay consultation fees and view invoices",
    "Notifications": "appointment reminders and health alerts"},
  "journey": ["Search a doctor","View profile","Pick a time slot","Confirm booking","Attend consultation","Get prescription","Follow-up"],
  "tips": ["Use calm colors and clear, plain language.","Make emergency and contact actions obvious.","Show privacy and security reassurance on sensitive screens.","Design for all ages: large text and tap targets."]},
 "banking": {
  "label": "Banking / finance",
  "description": "Banking app for checking balances, transferring money, viewing transactions, managing cards and paying bills.",
  "keywords": ["bank","banking","loan","fintech","balance","transaction","money transfer","savings","credit card","debit card","finance","upi","wallet"],
  "pages": ["Login","Dashboard","Accounts","Transfer","Transactions","Cards","Bills & Payments","Profile"],
  "features": {
    "Secure authentication": "secure login with PIN, biometrics and multi-factor authentication",
    "Balance overview": "see account balances and summary at a glance",
    "Fund transfer": "transfer money to beneficiaries with confirmation steps",
    "Transaction history": "search, filter and export transaction history",
    "Bill payments": "pay bills and set up recurring payments",
    "Card management": "freeze cards, set limits and view card details",
    "Spending analytics": "charts and insights on spending by category",
    "Alerts": "real-time alerts for transactions and suspicious activity"},
  "journey": ["Secure login","View balance","Choose action","Enter details","Review and confirm","Authenticate (OTP/biometric)","See receipt"],
  "tips": ["Put security feedback and confirmations on every money action.","Show balance and recent activity first.","Use clear error messages and undo/limits for risky actions.","Keep flows short, but never skip a review step."]},
}

PAGE_COMPONENTS = {
 "Home": ["Top navigation bar","Search bar","Hero / featured section","Category shortcuts","Recommended items","Footer / bottom tab bar"],
 "Login": ["Logo","Email / phone field","Password / OTP field","Primary login button","Social login options","Sign-up link"],
 "Restaurant List": ["Search & filter bar","Restaurant cards (photo, rating, ETA)","Sort dropdown","Infinite scroll"],
 "Menu": ["Restaurant header (rating, ETA)","Category tabs","Dish cards with photo and price","Add-to-cart button","Sticky cart summary"],
 "Cart": ["Item list with quantity steppers","Promo code field","Price breakdown","Checkout button"],
 "Checkout": ["Address selector","Payment method selector","Order summary","Place order button"],
 "Order Tracking": ["Order status timeline","Live map","Delivery partner info","Help / support button"],
 "Profile": ["Avatar and name","Personal details","Saved addresses / preferences","Settings and logout"],
 "Product List": ["Filter sidebar","Product grid","Sort dropdown","Pagination"],
 "Product Detail": ["Image gallery","Title, price, rating","Variant selector","Add to cart / buy now","Description and reviews"],
 "Order History": ["Order list with status","Date filter","Reorder / track buttons"],
 "Wishlist": ["Saved product grid","Move to cart button","Remove button"],
 "Course Catalog": ["Search & category filters","Course cards (thumbnail, rating, level)","Sort dropdown"],
 "Course Detail": ["Course banner","Syllabus accordion","Instructor info","Enroll button","Reviews"],
 "Lesson Player": ["Video player","Lesson list sidebar","Notes tab","Next / previous controls","Progress bar"],
 "Dashboard": ["Welcome header","Key stats cards","Activity / progress chart","Quick actions","Notifications panel"],
 "Quiz": ["Question card","Answer options","Timer","Progress indicator","Submit button"],
 "Find Doctor": ["Search by name or specialty","Filters (location, availability, fee)","Doctor cards with rating"],
 "Doctor Profile": ["Photo and credentials","Specialties and experience","Availability calendar","Book appointment button","Patient reviews"],
 "Book Appointment": ["Date picker","Time slot grid","Patient details form","Confirm button"],
 "Medical Records": ["Records list by date","Upload / download controls","Prescriptions section","Privacy notice"],
 "Telehealth": ["Video call area","Chat panel","Call controls (mute, camera, end)","Shared notes / prescription"],
 "Accounts": ["Account cards with balances","Account details","Statement download"],
 "Transfer": ["From-account selector","Beneficiary selector / add new","Amount field","Review and confirm step","OTP / biometric confirmation"],
 "Transactions": ["Search and date filter","Transactions grouped by day","Category tags","Export button"],
 "Cards": ["Card visual","Freeze / unfreeze toggle","Limits and settings","Recent card transactions"],
 "Bills & Payments": ["Biller categories","Saved billers","Amount and due date","Pay now button","Payment history"],
}
DEFAULT_COMPONENTS = ["Header / navigation", "Main content area", "Footer"]
ALL_PAGES = list(PAGE_COMPONENTS)

STYLES = {
 "minimal":      {"palette": {"primary":"#111827","accent":"#2563EB","bg":"#FFFFFF","surface":"#F3F4F6","text":"#111827"}, "fonts":"Inter (headings + body)", "notes":"Lots of whitespace, few colors, simple icons."},
 "modern":       {"palette": {"primary":"#4F46E5","accent":"#06B6D4","bg":"#F8FAFC","surface":"#FFFFFF","text":"#0F172A"}, "fonts":"Poppins (headings) + Inter (body)", "notes":"Rounded cards, soft shadows, gradients used sparingly."},
 "playful":      {"palette": {"primary":"#FF6B6B","accent":"#FFD93D","bg":"#FFF9F0","surface":"#FFFFFF","text":"#2D2A32"}, "fonts":"Nunito (headings + body)", "notes":"Bright colors, friendly illustrations, rounded shapes."},
 "professional": {"palette": {"primary":"#0B3D91","accent":"#1B998B","bg":"#F5F7FA","surface":"#FFFFFF","text":"#1F2933"}, "fonts":"Source Sans 3 / Roboto", "notes":"Structured grids, restrained colors, clear hierarchy."},
 "dark":         {"palette": {"primary":"#8B5CF6","accent":"#22D3EE","bg":"#0F172A","surface":"#1E293B","text":"#E2E8F0"}, "fonts":"Inter (headings + body)", "notes":"Dark surfaces, high-contrast text, glowing accents."},
 "elegant":      {"palette": {"primary":"#1F1F1F","accent":"#B08D57","bg":"#FAF7F2","surface":"#FFFFFF","text":"#222222"}, "fonts":"Playfair Display (headings) + Lato (body)", "notes":"Serif headings, muted gold accents, generous spacing."},
}
STYLE_ALIASES = {"minimal":["minimal","minimalist","clean","simple"], "modern":["modern","sleek","trendy","contemporary"],
                 "playful":["playful","fun","colorful","colourful","friendly","cartoon"], "professional":["professional","corporate","formal","business","trustworthy"],
                 "dark":["dark","night"], "elegant":["elegant","luxury","premium","classy"]}
COLOR_HEX = {"red":"#DC2626","blue":"#2563EB","green":"#16A34A","orange":"#EA580C","purple":"#7C3AED","pink":"#DB2777",
             "yellow":"#EAB308","teal":"#0D9488","black":"#111827","brown":"#92400E","grey":"#6B7280","gray":"#6B7280"}

DOMAIN_EMB = embed([f"{d['description']} Keywords: {', '.join(d['keywords'])}" for d in DOMAINS.values()])
FEATURE_EMB = {k: embed([f"{n}: {desc}" for n, desc in d["features"].items()]) for k, d in DOMAINS.items()}
print("Knowledge base ready:", [d["label"] for d in DOMAINS.values()])

Knowledge base ready: ['Food ordering', 'E-commerce', 'Education / e-learning', 'Healthcare', 'Banking / finance']


In [32]:
#4
FEATURE_THRESHOLD = 0.45   # lower = detects more features, higher = stricter
DOMAIN_THRESHOLD  = 0.35

def new_state():
    return {"domain": None, "domain_confidence": 0.0, "platform": None, "target_users": None, "brand_name": None,
            "auth_needed": None, "payment_needed": None, "style": None, "colors": [], "accessibility": None,
            "features": [], "vague_terms": [], "skipped": []}

def classify_domain(text):
    """Embedding similarity + keyword boost. Returns (domain or None, score, all_scores)."""
    t = text.lower()
    sims = cosine_similarity(embed([text]), DOMAIN_EMB)[0]
    scores = {}
    for (name, d), s in zip(DOMAINS.items(), sims):
        hits = sum(1 for k in d["keywords"] if re.search(r"\b" + re.escape(k), t))
        scores[name] = float(s) + 0.15 * min(hits, 3)
    ranked = sorted(scores.values(), reverse=True)
    best = max(scores, key=scores.get)
    if ranked[0] < DOMAIN_THRESHOLD or (ranked[0] - ranked[1]) < 0.03:
        return None, ranked[0], scores
    return best, ranked[0], scores

def extract_platform(text):
    t = text.lower()
    both = re.search(r"\b(both|cross-platform|responsive|web and mobile|mobile and web|website and app)\b", t)
    mobile = re.search(r"\b(mobile|android|ios|iphone|smartphone|mobile app)\b", t)
    web = re.search(r"\b(website|web app|web|site|browser|desktop|web-based)\b", t)
    if both or (mobile and web): return "both"
    if mobile: return "mobile"
    if web: return "web"
    return None

PEOPLE = r"(students|customers|patients|doctors|teachers|users|shoppers|diners|people|clients|parents|kids|children|employees|managers|sellers|merchants|learners|travellers|travelers|members|owners|businesses|beginners|professionals|elderly|seniors|youth)"
def extract_target_users(text):
    m = re.search(r"\b(?:for|targeting|aimed at|used by)\s+((?:[\w-]+\s){0,3}?" + PEOPLE + r")\b", text, re.I)
    return m.group(1).strip() if m else None

def extract_brand(text):
    m = re.search(r"\b(?:called|named)\s+([A-Z][\w&-]*(?:\s[A-Z][\w&-]*)*)", text)
    if m: return m.group(1)
    for ent in nlp(text).ents:
        if ent.label_ == "ORG": return ent.text
    return None

def split_segments(text):
    segs = []
    for s in nlp(text).sents:
        for part in re.split(r",|;|\band\b|\bwith\b|\balso\b|\bplus\b", s.text):
            if len(part.split()) >= 2: segs.append(part.strip())
    return segs

def extract_features(text, domain):
    if not domain: return []
    segs = split_segments(text)
    if not segs: return []
    sims = cosine_similarity(embed(segs), FEATURE_EMB[domain])
    names = list(DOMAINS[domain]["features"])
    return [names[j] for j in range(len(names)) if sims[:, j].max() >= FEATURE_THRESHOLD]

def extract_style(text):
    t = text.lower()
    for style, words in STYLE_ALIASES.items():
        if any(re.search(r"\b" + w + r"\b", t) for w in words): return style
    return None

def extract_colors(text):
    t = text.lower()
    return [c for c in COLOR_HEX if re.search(r"\b" + c + r"\b", t)]

def extract_auth(text):
    t = text.lower()
    if re.search(r"\b(no login|without login|no sign ?up|guest only|no account)\b", t): return False
    if re.search(r"\b(login|log in|sign ?up|sign in|register|registration|authentication|user accounts?)\b", t): return True
    return None

def extract_payment(text):
    t = text.lower()
    if re.search(r"\b(no payments?|without payments?|free of charge)\b", t): return False
    if re.search(r"\b(payments?|pay online|checkout|upi|credit card|debit card|wallet|subscription)\b", t): return True
    return None

VAGUE_STYLE_WORDS = ["modern", "clean", "simple", "nice", "cool", "user-friendly", "user friendly", "attractive", "beautiful", "good-looking"]
def detect_vague_terms(text):
    t = text.lower()
    return [w for w in VAGUE_STYLE_WORDS if re.search(r"\b" + re.escape(w) + r"\b", t)]

def extract_all(text, st):
    """Fill whatever can be found in free text without overwriting confirmed values."""
    if st["domain"] is None:
        d, conf, _ = classify_domain(text)
        if d: st["domain"], st["domain_confidence"] = d, round(conf, 3)
    if st["platform"] is None: st["platform"] = extract_platform(text)
    if st["target_users"] is None: st["target_users"] = extract_target_users(text)
    if st["brand_name"] is None: st["brand_name"] = extract_brand(text)
    if st["auth_needed"] is None: st["auth_needed"] = extract_auth(text)
    if st["payment_needed"] is None: st["payment_needed"] = extract_payment(text)
    if st["style"] is None: st["style"] = extract_style(text)
    for c in extract_colors(text):
        if c not in st["colors"]: st["colors"].append(c)
    if re.search(r"\b(accessib|wcag|screen reader|color ?blind|high contrast)", text.lower()): st["accessibility"] = True
    for f in extract_features(text, st["domain"]):
        if f not in st["features"]: st["features"].append(f)
    for v in detect_vague_terms(text):
        if v not in st["vague_terms"]: st["vague_terms"].append(v)
    return st
print("Extraction ready")

Extraction ready


In [33]:
#5
REQUIRED_SLOTS = ["domain", "platform", "target_users", "auth_needed", "payment_needed", "style"]

def is_filled(st, slot):
    return slot in st["skipped"] or st.get(slot) not in (None, "", [])

def next_missing(st):
    for s in REQUIRED_SLOTS:
        if not is_filled(st, s): return s
    return None

def ask_text(st, slot):
    if slot == "domain":
        return "What kind of product is this? (food ordering, e-commerce, education, healthcare, or banking)"
    if slot == "platform":
        return "Which platform do you want: **mobile app**, **website**, or **both**?"
    if slot == "target_users":
        return "Who are the main users? (for example: students, patients, shoppers)"
    if slot == "auth_needed":
        return "Will users need to create an account and log in? (yes / no)"
    if slot == "payment_needed":
        return "Will the product handle payments? (yes / no)"
    if slot == "style":
        vague = st["vague_terms"][0] if st["vague_terms"] else None
        pre = f"You said \"{vague}\", which can mean different things. " if vague else ""
        return pre + "Which visual style fits best: **minimal, modern, playful, professional, dark, or elegant**?"

def parse_yes_no(t):
    t = t.lower()
    if re.search(r"\b(no|not|without|don't|dont|nope|skip|none|nah)\b", t): return False
    if re.search(r"\b(yes|yeah|yep|sure|need|required|include|definitely|ok|okay|of course|y)\b", t): return True
    return None

def update_from_answer(st, slot, text):
    """Apply the user's answer to the slot we asked about. Returns True if understood."""
    if is_filled(st, slot): return True        # extract_all may already have caught it
    t = text.strip()
    if slot == "domain":
        low = t.lower()
        direct = {"food": "food_ordering", "restaurant": "food_ordering", "commerce": "ecommerce", "shop": "ecommerce",
                  "educat": "education", "learn": "education", "health": "healthcare", "medical": "healthcare", "bank": "banking", "financ": "banking"}
        for k, v in direct.items():
            if k in low: st["domain"] = v; return True
        d, conf, _ = classify_domain(t)
        if d: st["domain"], st["domain_confidence"] = d, round(conf, 3); return True
        return False
    if slot == "platform":
        p = extract_platform(t)
        if p: st["platform"] = p; return True
        if re.search(r"\bapp\b", t.lower()): st["platform"] = "mobile"; return True
        return False
    if slot == "target_users":
        cleaned = re.sub(r"^(for|my users are|users are|they are|it is for)\s+", "", t, flags=re.I).strip(" .")
        if len(cleaned) >= 3: st["target_users"] = cleaned; return True
        return False
    if slot in ("auth_needed", "payment_needed"):
        v = parse_yes_no(t)
        if v is None: return False
        st[slot] = v; return True
    if slot == "style":
        s = extract_style(t)
        if s: st["style"] = s; return True
        return False
    return False

def snapshot(st):
    parts = []
    if st["domain"]: parts.append(f"product: {DOMAINS[st['domain']]['label']}")
    if st["platform"]: parts.append(f"platform: {st['platform']}")
    if st["target_users"]: parts.append(f"users: {st['target_users']}")
    if st["auth_needed"] is not None: parts.append(f"login: {'yes' if st['auth_needed'] else 'no'}")
    if st["payment_needed"] is not None: parts.append(f"payments: {'yes' if st['payment_needed'] else 'no'}")
    if st["style"]: parts.append(f"style: {st['style']}")
    return "Noted so far: " + ", ".join(parts) + "." if parts else ""
print("Missing-info logic ready")

Missing-info logic ready


In [34]:
#6
def recommend_pages(st):
    pages = list(DOMAINS[st["domain"]]["pages"]) if st["domain"] else ["Home"]
    if st["auth_needed"] and "Login" not in pages:
        pages.insert(1 if "Home" in pages else 0, "Login")
    if st["auth_needed"] is False and "Login" in pages:
        pages.remove("Login")
    return pages

def recommend_features(st):
    return list(DOMAINS[st["domain"]]["features"]) if st["domain"] else []

def get_palette(st):
    style = st["style"] or "modern"
    pal = dict(STYLES[style]["palette"])
    if st["colors"]: pal["primary"] = COLOR_HEX[st["colors"][0]]
    return pal

def match_name(query, names, thr=0.5):
    q = query.strip().lower()
    if not q or not names: return None
    for n in names:
        if q == n.lower(): return n
    for n in names:
        if q in n.lower() or n.lower() in q: return n
    sims = cosine_similarity(embed([query]), embed(names))[0]
    j = int(np.argmax(sims))
    return names[j] if sims[j] >= thr else None

def components_for(page):
    return PAGE_COMPONENTS.get(page, DEFAULT_COMPONENTS)

def ascii_wireframe(page, comps, width=48):
    inner = width - 2
    line = "+" + "-" * inner + "+"
    out = [line, "|" + page.upper().center(inner) + "|", line]
    for c in comps:
        out.append("| [" + c[:inner - 4].ljust(inner - 4) + "] |")
    out.append(line)
    return "\n".join(out)

def ux_analysis(p):
    st = p.state
    if not st["domain"]: return "No analysis yet: finish the requirements chat first."
    d = DOMAINS[st["domain"]]
    platform_tip = {"mobile": "Use a bottom tab bar, thumb-reachable actions, tap targets of at least 44px.",
                    "web": "Use a responsive grid, clear top navigation, full keyboard navigation.",
                    "both": "Design mobile-first, then scale up to desktop with a responsive grid."}.get(st["platform"], "Platform not specified; design mobile-first.")
    extra = []
    if st["auth_needed"]: extra.append("Support easy login (social/OTP) and a clear forgot-password path.")
    if st["payment_needed"]: extra.append("Show total cost early, offer multiple payment methods, confirm success clearly.")
    if st["accessibility"]: extra.append("Target WCAG AA: contrast 4.5:1, labels on inputs, visible focus states.")
    lines = [
      f"### UX analysis: {st['brand_name'] or d['label']}",
      f"**Product type:** {d['label']}  |  **Platform:** {st['platform'] or 'not set'}  |  **Users:** {st['target_users'] or 'not set'}",
      "", "**Core user journey**", " → ".join(d["journey"]), "",
      "**UX recommendations**", *[f"- {t}" for t in d["tips"]], f"- {platform_tip}", *[f"- {t}" for t in extra],
      "", f"**Selected pages ({len(p.pages)}):** " + ", ".join(p.pages or recommend_pages(st)),
      f"**Selected features ({len(p.features)}):** " + (", ".join(p.features) or "none yet")]
    return "\n".join(lines)

def build_spec_markdown(p):
    st = p.state
    if not st["domain"]: return "_Nothing to show yet. Start with the Requirements chat._"
    style = st["style"] or "modern"
    pal, sty = get_palette(st), STYLES[style]
    md = [f"# {st['brand_name'] or p.name}", "", ux_analysis(p), "", "---", f"## Visual design: {style}",
          f"- **Fonts:** {sty['fonts']}", f"- **Notes:** {sty['notes']}",
          "- **Palette:** " + ", ".join(f"{k} `{v}`" for k, v in pal.items()), "", "## Pages and components"]
    for pg in p.pages:
        md.append(f"### {pg}")
        md += [f"- {c}" for c in components_for(pg)]
    return "\n".join(md)

def export_html(p):
    st = p.state
    pal = get_palette(st)
    fonts = "Inter, Arial, sans-serif"
    css = f"""body{{margin:0;font-family:{fonts};background:{pal['bg']};color:{pal['text']}}}
header{{background:{pal['primary']};color:#fff;padding:20px 32px}} nav a{{color:#fff;margin-right:16px;font-size:14px}}
section{{max-width:900px;margin:24px auto;padding:20px;background:{pal['surface']};border-radius:12px;box-shadow:0 1px 4px rgba(0,0,0,.15)}}
h2{{color:{pal['primary']};margin-top:0}} .c{{border:2px dashed {pal['accent']};border-radius:8px;padding:14px;margin:8px 0;font-size:14px}}"""
    nav = "".join(f'<a href="#{html.escape(pg)}">{html.escape(pg)}</a>' for pg in p.pages)
    body = ""
    for pg in p.pages:
        comps = "".join(f'<div class="c">{html.escape(c)}</div>' for c in components_for(pg))
        body += f'<section id="{html.escape(pg)}"><h2>{html.escape(pg)}</h2>{comps}</section>'
    title = html.escape(st["brand_name"] or p.name)
    doc = f"<!doctype html><html><head><meta charset='utf-8'><title>{title}</title><style>{css}</style></head><body><header><h1>{title}</h1><nav>{nav}</nav></header>{body}</body></html>"
    path = os.path.join(SAVE_DIR, f"{p.id}_wireframe.html")
    open(path, "w", encoding="utf-8").write(doc)
    return path
print("Recommendation engine ready")

Recommendation engine ready


In [35]:
#7
STAGES = ["requirements", "pages", "features", "design"]

class Project:
    def __init__(self, name="Untitled project"):
        self.id = uuid.uuid4().hex[:8]
        self.name = name
        self.created = datetime.datetime.now().isoformat(timespec="seconds")
        self.state = new_state()
        self.stage = "requirements"
        self.histories = {s: [] for s in STAGES}   # one separate chat per stage: [user, bot] pairs
        self.pending = None
        self.retries = {}
        self.pages, self.features = [], []
        self.stage_done = {s: False for s in STAGES}

    def update_name(self):
        if self.state["brand_name"]: self.name = self.state["brand_name"]
        elif self.state["domain"] and self.name == "Untitled project":
            self.name = DOMAINS[self.state["domain"]]["label"] + " project"

    def to_dict(self): return dict(self.__dict__)

    @classmethod
    def from_dict(cls, d):
        p = cls(); p.__dict__.update(d); return p

    def save(self, path=None):
        path = path or os.path.join(SAVE_DIR, f"{self.id}.json")
        json.dump(self.to_dict(), open(path, "w"), indent=2)
        return path

    @classmethod
    def load(cls, path):
        return cls.from_dict(json.load(open(path)))

def list_saved():
    files = sorted(glob.glob(f"{SAVE_DIR}/*.json"), key=os.path.getmtime, reverse=True)
    out = []
    for f in files:
        try:
            d = json.load(open(f)); out.append((f"{d['name']} ({d['id']})", f))
        except Exception:
            pass
    return out
print("Project manager ready")

Project manager ready


In [36]:
#8
def list_md(items, empty="(none yet)"):
    return "\n".join(f"{i+1}. {x}" for i, x in enumerate(items)) if items else empty

DONE_RE = re.compile(r"^\s*(done|confirm|finish|finished|looks good|continue|next|ok done)\s*[.!]?\s*$", re.I)

# ---------- Stage 1: requirements ----------
def handle_requirements(p, text):
    st = p.state
    extract_all(text, st)
    prefix = ""
    if p.pending:
        slot = p.pending
        if update_from_answer(st, slot, text):
            p.retries.pop(slot, None)
        else:
            p.retries[slot] = p.retries.get(slot, 0) + 1
            if slot != "domain" and p.retries[slot] >= 2:
                st["skipped"].append(slot)
                prefix = f"No problem, I'll leave **{slot.replace('_', ' ')}** open for now.\n\n"
            else:
                return "Sorry, I didn't catch that. " + ask_text(st, slot)
    p.pending = None
    p.update_name()
    slot = next_missing(st)
    if slot:
        p.pending = slot
        return prefix + (snapshot(st) + "\n\n" if snapshot(st) else "") + ask_text(st, slot)
    p.stage_done["requirements"] = True
    p.pages = recommend_pages(st)
    feats = list(st["features"])
    p.features = feats if feats else recommend_features(st)[:5]
    return (prefix + "Requirements are complete.\n\n" + snapshot(st) + "\n\n" + ux_analysis(p) +
            "\n\nNext: switch to the **pages** chat to choose your pages.")

# ---------- Stage 2: pages ----------
def handle_pages(p, text):
    st = p.state
    if not st["domain"]: return "Please finish the Requirements chat first (at least the product type)."
    if not p.pages: p.pages = recommend_pages(st)
    low = text.strip().lower()
    if DONE_RE.match(low):
        p.stage_done["pages"] = True
        return "Pages confirmed:\n" + list_md(p.pages) + "\n\nNext: switch to the **features** chat."
    m = re.match(r"^(add|include)\s+(.+)", low)
    if m:
        q = m.group(2).strip(" .")
        name = match_name(q, ALL_PAGES, 0.6) or q.title()
        if name in p.pages: return f"**{name}** is already in your pages.\n\n" + list_md(p.pages)
        p.pages.append(name)
        return f"Added **{name}**.\n\n" + list_md(p.pages)
    m = re.match(r"^(remove|delete|drop)\s+(.+)", low)
    if m:
        name = match_name(m.group(2).strip(" ."), p.pages)
        if not name: return "I couldn't find that page.\n\n" + list_md(p.pages)
        p.pages.remove(name)
        return f"Removed **{name}**.\n\n" + list_md(p.pages)
    if re.match(r"^(list|show|pages)", low): return "Current pages:\n" + list_md(p.pages)
    if re.match(r"^(suggest|recommend|reset)", low):
        p.pages = recommend_pages(st); return "Reset to recommended pages:\n" + list_md(p.pages)
    return "Commands: `add <page>`, `remove <page>`, `list`, `reset`, `done`.\n\nCurrent pages:\n" + list_md(p.pages)

# ---------- Stage 3: features ----------
def handle_features(p, text):
    st = p.state
    if not st["domain"]: return "Please finish the Requirements chat first."
    catalog = DOMAINS[st["domain"]]["features"]
    low = text.strip().lower()
    def suggestions(): return [f for f in catalog if f not in p.features]
    if DONE_RE.match(low):
        p.stage_done["features"] = True
        return "Features confirmed:\n" + list_md(p.features) + "\n\nNext: switch to the **design** chat."
    m = re.match(r"^(add|include)\s+(.+)", low)
    if m:
        q = m.group(2).strip(" .")
        name = match_name(q, list(catalog), 0.45) or q.capitalize()
        if name in p.features: return f"**{name}** is already selected.\n\n" + list_md(p.features)
        p.features.append(name)
        return f"Added **{name}**.\n\n" + list_md(p.features)
    m = re.match(r"^(remove|delete|drop)\s+(.+)", low)
    if m:
        name = match_name(m.group(2).strip(" ."), p.features)
        if not name: return "I couldn't find that feature.\n\n" + list_md(p.features)
        p.features.remove(name)
        return f"Removed **{name}**.\n\n" + list_md(p.features)
    if re.match(r"^(suggest|recommend)", low):
        return "Suggested features you haven't picked yet:\n" + list_md(suggestions(), "(all added)")
    if re.match(r"^(list|show|features)", low): return "Selected features:\n" + list_md(p.features)
    # free text: detect features from description
    found = [f for f in extract_features(text, st["domain"]) if f not in p.features]
    if found:
        p.features += found
        return "Detected and added: " + ", ".join(found) + "\n\n" + list_md(p.features)
    return "Commands: `add <feature>`, `remove <feature>`, `suggest`, `list`, `done`.\n\nSelected:\n" + list_md(p.features)

# ---------- Stage 4: design ----------
def handle_design(p, text):
    st = p.state
    if not st["domain"]: return "Please finish the Requirements chat first."
    if not p.pages: p.pages = recommend_pages(st)
    low = text.strip().lower()
    if st["style"] is None: st["style"] = "modern"
    m = re.match(r"^wireframes?\s*(.*)", low)
    if m:
        arg = m.group(1).strip()
        if arg in ("", "all"):
            return "\n\n".join("```\n" + ascii_wireframe(pg, components_for(pg)) + "\n```" for pg in p.pages)
        pg = match_name(arg, p.pages)
        if not pg: return "Page not found. Your pages: " + ", ".join(p.pages)
        return "```\n" + ascii_wireframe(pg, components_for(pg)) + "\n```"
    m = re.match(r"^(style|theme)\s+(.+)", low)
    s = extract_style(m.group(2)) if m else extract_style(low)
    if s:
        st["style"] = s
        pal, sty = get_palette(st), STYLES[s]
        return (f"Style set to **{s}**.\n- Fonts: {sty['fonts']}\n- {sty['notes']}\n- Palette: " +
                ", ".join(f"{k} `{v}`" for k, v in pal.items()))
    cols = extract_colors(low)
    if cols:
        st["colors"] = cols
        return f"Primary color set to **{cols[0]}** (`{get_palette(st)['primary']}`)."
    if low in ("spec", "summary", "export", "done"):
        p.stage_done["design"] = True
        return build_spec_markdown(p) + "\n\nOpen the **Project** tab to download the spec and an HTML wireframe."
    style = st["style"]; pal = get_palette(st)
    return (f"Current design: **{style}**, fonts {STYLES[style]['fonts']}, primary `{pal['primary']}`.\n\n"
            "Try: `style dark` · `color green` · `wireframe cart` · `wireframes` · `spec`")

# ---------- Intros and dispatcher ----------
def stage_intro(p, stage):
    st = p.state
    if stage == "requirements":
        return ("Hi! Describe the website or app you want to build, in your own words. "
                "I'll ask follow-up questions for anything missing.")
    if not st["domain"]:
        return "Please finish the **requirements** chat first, then come back here."
    if stage == "pages":
        if not p.pages: p.pages = recommend_pages(st)
        return "Recommended pages for your project:\n" + list_md(p.pages) + "\n\nCommands: `add <page>`, `remove <page>`, `list`, `reset`, `done`."
    if stage == "features":
        if not p.features: p.features = list(st["features"]) or recommend_features(st)[:5]
        sug = [f for f in recommend_features(st) if f not in p.features]
        return ("Selected features:\n" + list_md(p.features) + "\n\nSuggestions you could add:\n" + list_md(sug, "(none)") +
                "\n\nCommands: `add <feature>`, `remove <feature>`, `suggest`, `list`, `done`.")
    if stage == "design":
        return ("Let's design it. Choose a style: " + ", ".join(STYLES) + ".\n"
                "Commands: `style dark` · `color green` · `wireframe <page>` · `wireframes` · `spec`")

def ensure_intro(p, stage):
    if not p.histories[stage]:
        p.histories[stage].append(["", stage_intro(p, stage)])

HANDLERS = {"requirements": handle_requirements, "pages": handle_pages, "features": handle_features, "design": handle_design}

def chat(p, text, stage=None):
    stage = stage or p.stage
    p.stage = stage
    ensure_intro(p, stage)
    reply = HANDLERS[stage](p, text)
    p.histories[stage].append([text, reply])
    p.update_name()
    p.save()  # auto-save so you can resume later
    return reply
print("Chat logic ready")

Chat logic ready


In [37]:
#12
# (id, title, tags, source, text)
_KB = [
 # ---- General usability (Nielsen's 10 heuristics) ----
 ("h_status","Show system status","all","Nielsen heuristic 1","Always tell users what is happening: loading indicators, progress steps, and confirmation messages after actions."),
 ("h_match","Speak the user's language","all","Nielsen heuristic 2","Use words and concepts familiar to your users, not internal jargon. Follow real-world conventions and logical order."),
 ("h_control","Give control and easy exits","all","Nielsen heuristic 3","Provide undo, cancel and back options so users can leave unwanted states without losing work."),
 ("h_consistency","Be consistent","all","Nielsen heuristic 4","Keep labels, icons, colors and interactions consistent across pages and follow platform conventions."),
 ("h_prevent","Prevent errors","all","Nielsen heuristic 5","Use constraints, sensible defaults and confirmation for destructive or irreversible actions."),
 ("h_recognition","Recognition over recall","all","Nielsen heuristic 6","Keep options and recent items visible so users do not have to remember information between screens."),
 ("h_flexibility","Shortcuts for experienced users","all","Nielsen heuristic 7","Offer accelerators such as saved items, search, favorites and quick actions for frequent users."),
 ("h_minimal","Minimalist design","all","Nielsen heuristic 8","Remove irrelevant information. Every extra element competes with the important ones."),
 ("h_errors","Help users recover from errors","all","Nielsen heuristic 9","Write error messages in plain language, say what went wrong and suggest how to fix it."),
 ("h_help","Help and documentation","all","Nielsen heuristic 10","Provide short, searchable, task-focused help and contextual hints near the place they are needed."),
 # ---- Accessibility ----
 ("a_contrast","Text contrast","all","WCAG 2.2 (1.4.3, 1.4.11)","Meet AA contrast: at least 4.5:1 for normal text and 3:1 for large text and UI components such as buttons and input borders."),
 ("a_keyboard","Keyboard access and focus","all","WCAG 2.2 (2.1.1, 2.4.7)","All functions must work with a keyboard, with a clearly visible focus indicator and a logical tab order."),
 ("a_labels","Labels and alt text","all","WCAG 2.2 (1.1.1, 3.3.2)","Give every form field a visible label and every meaningful image a text alternative so screen readers can describe it."),
 ("a_color","Do not rely on color alone","all","WCAG 2.2 (1.4.1)","Never use color as the only way to show status or errors; add icons, text or patterns as well."),
 ("a_targets","Touch target size","mobile","Apple HIG / Material Design","Make tap targets at least 44x44 pt (Apple) or 48x48 dp (Material) with enough spacing between them to avoid mis-taps."),
 # ---- Navigation and layout ----
 ("n_mobile","Mobile navigation","mobile","Material Design / Apple HIG","Use a bottom tab bar with 3 to 5 top-level destinations so the main actions sit within thumb reach."),
 ("n_web","Web navigation","web","Common web patterns","Use a clear top navigation with a small number of items, a visible search, and breadcrumbs for deep hierarchies."),
 ("n_responsive","Responsive layout","all","Responsive design practice","Design mobile-first, use a fluid grid with breakpoints, and decide content priority before scaling up to larger screens."),
 ("n_cta","Primary call to action","all","Visual hierarchy practice","Give each screen one clear primary action, with a verb label such as Place order, and style secondary actions less prominently."),
 ("n_hick","Limit choices","all","Hick's law","More options increase decision time. Group, prioritize and progressively disclose choices."),
 # ---- Visual design ----
 ("v_type","Typography","all","Typography practice","Use at most two font families, a consistent type scale, body text around 16px on web, and line lengths of roughly 45 to 75 characters."),
 ("v_space","Spacing and hierarchy","all","Layout practice","Use a consistent spacing scale such as multiples of 8px and group related items closely to show hierarchy."),
 ("v_color","Color palette","all","Color practice","Use one primary and one accent color on a neutral base, plus consistent semantic colors for success, warning and error."),
 ("v_dark","Dark theme","all","Dark mode practice","Avoid pure black and pure white, desaturate accent colors, and recheck contrast ratios on dark surfaces."),
 # ---- Components and patterns ----
 ("p_forms","Form design","all","Form usability practice","Use a single column, labels above fields, only the necessary fields, correct input types, and inline validation with helpful messages."),
 ("p_search","Search and filters","all","Search UX practice","Make search prominent, show applied filters clearly with a one-tap clear option, offer sorting, and show result counts."),
 ("p_states","Empty, loading and error states","all","Interaction design practice","Design empty, loading and error states deliberately: explain what is happening and offer a next action. Skeleton screens improve perceived speed."),
 ("p_onboard","Sign-up and onboarding","all","Onboarding practice","Delay registration until it is needed, keep sign-up short, offer social or OTP login, and explain value before asking for data."),
 ("p_login","Login screen","all","Authentication UX practice","Offer a show-password toggle, forgot-password link, OTP autofill and optional biometric login, with clear error messages."),
 ("p_notify","Notifications","all","Notification practice","Send only relevant, timely notifications and give users control over their type and frequency."),
 ("p_perf","Perceived performance","all","Nielsen response-time limits","Around 0.1 s feels instant and 1 s keeps the flow of thought; beyond that show progress feedback and load images lazily."),
 ("p_trust","Trust and security cues","all","Trust UX practice","Show security cues near sensitive actions, explain why data is requested, and confirm important actions clearly."),
 # ---- Food ordering ----
 ("f_menu","Menu and dish cards","food_ordering","Food delivery UX patterns","Show dish photos, price, dietary tags and ratings on the card, with a quick add button and a customization sheet for options."),
 ("f_cart","Cart and fees","food_ordering","Food delivery UX patterns","Keep a sticky cart summary visible and show delivery fee, taxes and estimated time before checkout to avoid surprises."),
 ("f_track","Order tracking","food_ordering","Food delivery UX patterns","Show a status timeline (confirmed, preparing, on the way, delivered), a live map, ETA updates and a contact or help option."),
 ("f_reorder","Reorder and saved addresses","food_ordering","Food delivery UX patterns","Offer one-tap reorder, favorites and saved addresses to make repeat orders fast."),
 # ---- E-commerce ----
 ("e_product","Product detail page","ecommerce","E-commerce UX patterns","Use a large zoomable gallery, clear price and stock status, variant selectors, a prominent add to cart button, and returns info near the button."),
 ("e_checkout","Checkout flow","ecommerce","E-commerce UX patterns","Offer guest checkout, show a progress indicator and total cost early, support multiple payment methods and include a final review step."),
 ("e_cart","Shopping cart","ecommerce","E-commerce UX patterns","Allow editing quantities and removing items in the cart, keep the cart saved, and show shipping estimates and a promo code field."),
 # ---- Education ----
 ("d_progress","Learning progress","education","E-learning UX patterns","Show progress bars, let learners resume where they stopped, and break lessons into short units with visible next steps."),
 ("d_quiz","Quizzes","education","E-learning UX patterns","Give immediate feedback with explanations, allow review of answers, and make any timer clear and accessible."),
 ("d_video","Video lessons","education","E-learning UX patterns","Provide captions or transcripts, playback speed control, keyboard shortcuts and a notes area beside the video."),
 # ---- Healthcare ----
 ("m_booking","Appointment booking","healthcare","Healthcare UX patterns","Show doctor availability and fees upfront, use a simple date and time slot picker, send reminders and make rescheduling easy."),
 ("m_privacy","Privacy and sensitive data","healthcare","Healthcare UX patterns","Explain how health data is used, ask for explicit consent, use session timeouts, and use plain language with large readable text."),
 ("m_tele","Telehealth sessions","healthcare","Healthcare UX patterns","Offer a device and connection check before the call, a chat or phone fallback, and simple clear call controls."),
 # ---- Banking ----
 ("b_transfer","Money transfer flow","banking","Banking UX patterns","Use a short step flow with beneficiary verification, a clear review screen, OTP or biometric confirmation, and a receipt after completion."),
 ("b_security","Security feedback","banking","Banking UX patterns","Support multi-factor authentication, session timeout, masked account numbers and alerts for sensitive activity."),
 ("b_dash","Banking dashboard","banking","Banking UX patterns","Show balance first, then recent transactions and quick actions, with simple spending insights."),
 ("b_txn","Transaction list","banking","Banking UX patterns","Offer search, date filters and categories, distinguish pending from completed items, and allow export of statements."),
]
KB = [{"id": i, "title": t, "tags": tg.split(","), "source": s, "text": x} for (i, t, tg, s, x) in _KB]
KB_EMB = embed([f"{d['title']}. {d['text']}" for d in KB])
KB_BY_ID = {d["id"]: d for d in KB}
print(f"Knowledge base loaded: {len(KB)} guidelines")

Knowledge base loaded: 49 guidelines


In [38]:
#13
def retrieve(query, domain=None, platform=None, k=3, min_sim=0.25):
    sims = cosine_similarity(embed([query]), KB_EMB)[0].copy()
    for i, d in enumerate(KB):
        tags = d["tags"]
        if domain and domain in tags: sims[i] += 0.06                       # matches this project's domain
        if domain and "all" not in tags and domain not in tags and not (set(tags) & {"mobile", "web"}):
            sims[i] -= 0.08                                                  # belongs to a different domain
        if "mobile" in tags and platform == "web": sims[i] -= 0.08           # platform mismatch
        if "web" in tags and platform == "mobile": sims[i] -= 0.08
    order = np.argsort(-sims)[:k]
    return [(KB[i], float(sims[i])) for i in order if sims[i] >= min_sim]

def retrieve_for_page(p, page, k=3):
    st = p.state
    q = f"{page} page: " + ", ".join(components_for(page))
    return retrieve(q, st["domain"], st["platform"], k=k, min_sim=0.2)

def rag_answer(p, question):
    st = p.state
    hits = retrieve(question, st["domain"], st["platform"], k=3)
    if not hits:
        return ("I don't have guidance on that in my knowledge base yet. Try rephrasing, "
                "or ask about a page, a pattern (forms, navigation, checkout) or accessibility.")
    lines = [f"**UX guidance for:** _{question}_", ""]
    for n, (d, s) in enumerate(hits, 1):
        lines.append(f"{n}. **{d['title']}**: {d['text']} _({d['source']})_")
    if st["domain"]:
        plat = f", {st['platform']}" if st["platform"] else ""
        lines.append(f"\n_Ranked for your {DOMAINS[st['domain']]['label']} project{plat}._")
    return "\n".join(lines)

def add_guideline(id, title, text, source="Custom", tags=("all",)):
    """Add your own guideline at runtime (e.g. from a UX book or course notes)."""
    global KB_EMB
    d = {"id": id, "title": title, "tags": list(tags), "source": source, "text": text}
    KB.append(d); KB_BY_ID[id] = d
    KB_EMB = embed([f"{x['title']}. {x['text']}" for x in KB])
print("Retriever ready")

Retriever ready


In [39]:
#38
STACKS = {
 "react": {"label": "React + Vite + TypeScript + Tailwind CSS", "router": "React Router",
           "state": "Zustand for client state and TanStack Query for server state",
           "test": "Vitest + React Testing Library for unit and component tests, Playwright for end-to-end tests",
           "mock": "MSW (Mock Service Worker) with seed data", "a11y": "eslint-plugin-jsx-a11y and axe-core (via @axe-core/playwright)",
           "tree": "src/\n  app/ (providers, routes)\n  components/ui/ (design-system components)\n  features/<feature>/ (components, hooks, api per feature)\n  pages/ (one folder per page)\n  lib/ (api client, utils)\n  mocks/ (MSW handlers and seed data)\n  styles/ (tokens.css, global.css)\n  types/ (shared types)\n  tests/"},
 "nextjs": {"label": "Next.js (App Router) + TypeScript + Tailwind CSS", "router": "file-based App Router",
           "state": "TanStack Query for server state and Zustand for client state",
           "test": "Vitest + React Testing Library, Playwright for end-to-end tests", "mock": "Route Handlers returning mock data from seed files",
           "a11y": "eslint-plugin-jsx-a11y and axe-core (via @axe-core/playwright)",
           "tree": "app/ (routes and layouts, one folder per page)\ncomponents/ui/ (design-system components)\nfeatures/<feature>/\nlib/ (api client, utils)\nmocks/ (seed data)\nstyles/ (tokens)\ntypes/\ntests/"},
 "react-native": {"label": "React Native (Expo) + TypeScript", "router": "Expo Router",
           "state": "Zustand for client state and TanStack Query for server state", "test": "Jest + React Native Testing Library, Maestro for end-to-end flows",
           "mock": "MSW or a local mock service with seed data", "a11y": "accessibilityLabel/accessibilityRole props plus manual VoiceOver and TalkBack checks",
           "tree": "app/ (Expo Router screens, one per page)\nsrc/components/ui/ (design-system components)\nsrc/features/<feature>/\nsrc/lib/ (api client, utils)\nsrc/mocks/ (seed data)\nsrc/theme/ (tokens)\nsrc/types/\n__tests__/"},
 "flutter": {"label": "Flutter (Dart)", "router": "go_router", "state": "Riverpod",
           "test": "flutter_test for widget tests and integration_test for flows", "mock": "repository classes with fake in-memory data",
           "a11y": "Semantics widgets plus manual TalkBack and VoiceOver checks",
           "tree": "lib/\n  app/ (router, theme)\n  core/ (tokens, widgets/ui)\n  features/<feature>/ (data, domain, presentation)\n  data/ (repositories, fake data)\ntest/\nintegration_test/"},
 "vanilla": {"label": "HTML + CSS + vanilla JavaScript (no framework)", "router": "a small hash-based router", "state": "a simple store module",
           "test": "Playwright for end-to-end tests", "mock": "local JSON files behind a data module",
           "a11y": "axe-core run through Playwright plus manual keyboard checks",
           "tree": "index.html\nsrc/\n  pages/ (one module per page)\n  components/ (reusable UI functions)\n  data/ (JSON seed data, api.js)\n  styles/ (tokens.css, base.css)\n  router.js\ntests/"},
}

PAGE_META = {   # page: (route, purpose, primary action, likely next pages, edge cases)
 "Home": ("/", "Entry point that gets users to the main task in one or two taps", "Search or tap a card or category to open an item",
          ("Menu", "Product Detail", "Course Detail", "Doctor Profile", "Restaurant List", "Product List", "Course Catalog", "Find Doctor"),
          ["search returns no results", "slow network: show skeleton cards", "first-time visitor versus returning user"]),
 "Login": ("/login", "Authenticate returning users and link to account creation", "Submit credentials", ("Home", "Dashboard"),
           ["wrong credentials: inline error that does not reveal which field was wrong", "show or hide password", "forgot-password link", "button disabled while submitting"]),
 "Restaurant List": ("/restaurants", "Browse and filter restaurants", "Open a restaurant", ("Menu",),
           ["no results: suggest clearing filters", "closed restaurants shown but not orderable", "pagination or infinite scroll"]),
 "Menu": ("/restaurants/:id", "Show one restaurant's menu and let users add dishes", "Add dishes, then open the cart", ("Cart",),
           ["dish unavailable", "restaurant closed", "adding from a second restaurant while the cart is not empty: ask to replace the cart"]),
 "Cart": ("/cart", "Review and edit the order before paying", "Proceed to checkout", ("Checkout",),
           ["empty cart: empty state with a link back", "quantity never below 1", "invalid promo code message", "totals recalculate instantly"]),
 "Checkout": ("/checkout", "Collect address and payment details, then place the order", "Place order", ("Order Tracking", "Order History", "Dashboard"),
           ["inline validation errors", "payment failure with retry and no duplicate order", "disable the button after the first click", "guest versus signed-in user"]),
 "Order Tracking": ("/orders/:id", "Show live order progress", "Check status or contact support", ("Order History", "Home"),
           ["status changes while the page is open", "order cancelled or delayed", "map unavailable: still show the timeline"]),
 "Profile": ("/profile", "Manage personal details, saved items and settings", "Edit details or log out", ("Home",),
           ["unsaved changes warning", "logout confirmation", "invalid email or phone"]),
 "Product List": ("/products", "Browse, search and filter products", "Open a product", ("Product Detail",),
           ["no results", "many filters applied", "pagination or infinite scroll"]),
 "Product Detail": ("/products/:id", "Present one product with everything needed to decide", "Add to cart", ("Cart",),
           ["out of stock", "variant must be chosen before adding", "image fails to load"]),
 "Order History": ("/orders", "List past orders and their status", "Open or reorder an order", ("Order Tracking", "Cart"),
           ["no orders yet", "cancelled or refunded orders", "long lists need pagination"]),
 "Wishlist": ("/wishlist", "Hold products the user wants later", "Move an item to the cart", ("Cart", "Product Detail"),
           ["empty wishlist", "item becomes unavailable"]),
 "Course Catalog": ("/courses", "Browse and filter courses", "Open a course", ("Course Detail",),
           ["no results", "free versus paid badges", "pagination"]),
 "Course Detail": ("/courses/:id", "Explain a course so the learner can decide to enroll", "Enroll", ("Lesson Player",),
           ["already enrolled: show Continue instead", "paid course without payment method", "long syllabus collapsed by default"]),
 "Lesson Player": ("/learn/:courseId/:lessonId", "Play lessons and track progress", "Watch, then continue or take the quiz", ("Quiz", "Dashboard"),
           ["video fails to load: retry", "resume from last position", "lesson marked complete automatically"]),
 "Quiz": ("/learn/:courseId/quiz/:quizId", "Check understanding with questions and instant feedback", "Submit answers", ("Dashboard",),
           ["leaving mid-quiz: confirm and save progress", "timer expiry if timed", "no answer selected"]),
 "Dashboard": ("/dashboard", "Give a personal overview and quick actions", "Open an item from recent activity or quick actions", ("Home",),
           ["new user with no data", "each widget loads independently", "charts have a table alternative"]),
 "Find Doctor": ("/doctors", "Search and filter doctors", "Open a doctor's profile", ("Doctor Profile",),
           ["no results", "doctor not available today", "filters for fee and availability"]),
 "Doctor Profile": ("/doctors/:id", "Show credentials, availability and reviews", "Book appointment", ("Book Appointment",),
           ["no availability: offer notify-me", "long review lists", "video versus in-clinic options"]),
 "Book Appointment": ("/book/:doctorId", "Pick a slot and confirm the booking", "Confirm booking", ("Dashboard", "Home"),
           ["slot taken meanwhile: refresh and explain", "past slots disabled", "time zone shown clearly"]),
 "Medical Records": ("/records", "Store and view records securely", "Open or download a record", ("Dashboard",),
           ["no records yet", "upload failure", "session timeout on sensitive data"]),
 "Telehealth": ("/consult/:appointmentId", "Run the video consultation", "Join, talk, end call", ("Dashboard",),
           ["camera or microphone permission denied", "connection drops: fall back to chat or phone", "device check before joining"]),
 "Accounts": ("/accounts", "Show accounts and balances", "Open an account's transactions", ("Transactions",),
           ["balance hidden or shown toggle", "pending versus available balance", "loading skeleton"]),
 "Transfer": ("/transfer", "Move money safely in a short guided flow", "Review and confirm the transfer", ("Transactions", "Dashboard"),
           ["insufficient funds", "daily limit exceeded", "new beneficiary verification", "OTP or biometric failure", "double submission"]),
 "Transactions": ("/transactions", "Search and review transaction history", "Filter and open a transaction", ("Accounts",),
           ["pending versus completed", "empty search results", "export failure"]),
 "Cards": ("/cards", "View and control cards", "Freeze or unfreeze, change limits", ("Transactions",),
           ["freeze confirmation", "limit above the allowed maximum", "lost-card guidance"]),
 "Bills & Payments": ("/bills", "Pay bills and manage billers", "Pay now", ("Transactions",),
           ["payment fails", "due date passed", "duplicate payment warning"]),
}

DOMAIN_ENTITIES = {
 "food_ordering": ["User(id, name, email, phone)", "Address(id, user_id, label, line1, city, postal_code)", "Restaurant(id, name, cuisines, rating, eta_minutes, delivery_fee, is_open, image)",
                   "MenuItem(id, restaurant_id, name, description, price, category, dietary_tags, options, available)", "CartItem(menu_item_id, quantity, selected_options)",
                   "Order(id, user_id, restaurant_id, items, status, total, address_id, payment_method, created_at)", "Review(id, order_id, rating, comment)"],
 "ecommerce": ["User(id, name, email, phone)", "Address(id, user_id, line1, city, postal_code)", "Product(id, name, description, price, images, category, variants, stock, rating)",
               "CartItem(product_id, variant, quantity)", "WishlistItem(product_id)", "Order(id, items, status, total, shipping_address, payment_method, created_at)", "Review(id, product_id, rating, comment)"],
 "education": ["User(id, name, email, role)", "Course(id, title, description, level, price, rating, instructor_id, syllabus)", "Lesson(id, course_id, title, video_url, duration_minutes, position)",
               "Quiz(id, lesson_id, questions)", "Question(id, text, options, correct_option, explanation)", "Enrollment(user_id, course_id, progress_percent, last_lesson_id)", "Certificate(id, user_id, course_id, issued_at)"],
 "healthcare": ["User(id, name, email, phone)", "Doctor(id, name, specialty, years_experience, fee, rating, languages)", "AvailabilitySlot(id, doctor_id, start, end, mode)",
                "Appointment(id, patient_id, doctor_id, slot_id, status, mode, reason)", "MedicalRecord(id, patient_id, type, title, date, file_url)",
                "Prescription(id, appointment_id, medicines, instructions)", "Invoice(id, appointment_id, amount, status)"],
 "banking": ["User(id, name, email, phone)", "Account(id, type, masked_number, balance, available_balance, currency)", "Transaction(id, account_id, amount, direction, category, merchant, status, date)",
             "Beneficiary(id, name, masked_account, bank, verified)", "Transfer(id, from_account_id, beneficiary_id, amount, note, status)", "Card(id, masked_number, type, status, daily_limit)",
             "Biller(id, name, category, due_date, amount_due)"],
}

FEATURE_NOTES = {
 "Search & filters": "Debounced search plus filters (cuisine, brand, price, rating), sorting and result counts; filter state kept in the URL",
 "Menu browsing": "Category tabs and dish cards with photo, price and dietary tags",
 "Cart & customization": "Cart store with item options (for example toppings), quantity editing and persistence across reloads",
 "Online payment": "PaymentService interface with a mock provider; success, failure and retry flows",
 "Live order tracking": "Simulated real-time status updates (timers or a WebSocket stub), ETA and a map placeholder",
 "Ratings & reviews": "Star rating and short review input after an order; averages shown on cards",
 "Saved addresses & favorites": "Address book and favorite items or restaurants stored per user",
 "Offers & promo codes": "Promo validation service that returns a discount or an error reason",
 "Reorder": "One-tap reorder from order history into the cart",
 "Product gallery": "Zoomable image gallery with thumbnails and alt text",
 "Cart": "Cart with quantity editing, save for later and shipping estimate",
 "Secure checkout": "Checkout with guest option, validation and a final review step",
 "Reviews & ratings": "Rating summary, review list and review form",
 "Wishlist": "Add or remove products from lists and detail pages",
 "Coupons & discounts": "Coupon validation and visible discount lines in the totals",
 "Order tracking": "Order status timeline and shipment details",
 "Recommendations": "Simple similar-items and popular-now rows driven by seed-data rules",
 "Course search": "Search and filter by topic, level, rating and price",
 "Video lessons": "Video player with captions toggle, speed control and resume position",
 "Quizzes & assignments": "Quiz runner with instant feedback and explanations; assignment submission stub",
 "Progress tracking": "Per-course progress bars, completed-lesson tracking and streaks",
 "Certificates": "Certificate view generated on course completion, downloadable",
 "Discussion forums": "Threaded comments per lesson with posting and replying",
 "Instructor tools": "Instructor view to upload content (stub) and see student analytics",
 "Reminders": "In-app reminders for lessons and deadlines",
 "Doctor search": "Search and filter doctors by specialty, location, availability and fee",
 "Appointment booking": "Slot picker with booking, rescheduling and cancellation",
 "Telemedicine": "Video consultation room (mock stream) with chat and call controls",
 "Medical records": "Records list, upload and download stubs, clear privacy messaging",
 "Medication reminders": "Medicine schedule with in-app reminders",
 "Privacy & security": "Consent capture, session timeout and masked sensitive data",
 "Billing & payments": "Fee display, mock payment and invoice list",
 "Notifications": "In-app notification center with read and unread state",
 "Secure authentication": "PIN, biometric or MFA flow (mock) and session timeout",
 "Balance overview": "Account summary cards with a show or hide balance toggle",
 "Fund transfer": "Guided transfer: details, review, OTP or biometric confirmation, receipt",
 "Transaction history": "Searchable, filterable list showing pending versus completed, with export",
 "Bill payments": "Saved billers, due dates, pay now and recurring payment setup",
 "Card management": "Freeze and unfreeze, limits and recent card transactions",
 "Spending analytics": "Charts by category and period, each with a table alternative",
 "Alerts": "Transaction and security alerts list with severity",
}
FEATURE_PAGES = {
 "Search & filters": ["Home", "Restaurant List", "Product List"], "Menu browsing": ["Menu"], "Cart & customization": ["Menu", "Cart"], "Online payment": ["Checkout"],
 "Live order tracking": ["Order Tracking"], "Ratings & reviews": ["Menu", "Order History"], "Saved addresses & favorites": ["Checkout", "Profile"], "Offers & promo codes": ["Cart", "Checkout"],
 "Reorder": ["Order History"], "Product gallery": ["Product Detail"], "Cart": ["Cart", "Product Detail"], "Secure checkout": ["Checkout"], "Reviews & ratings": ["Product Detail"],
 "Wishlist": ["Wishlist", "Product List", "Product Detail"], "Coupons & discounts": ["Cart", "Checkout"], "Order tracking": ["Order History"], "Recommendations": ["Home", "Product Detail"],
 "Course search": ["Home", "Course Catalog"], "Video lessons": ["Lesson Player"], "Quizzes & assignments": ["Quiz"], "Progress tracking": ["Dashboard", "Lesson Player", "Course Detail"],
 "Certificates": ["Dashboard"], "Discussion forums": ["Lesson Player"], "Instructor tools": ["Dashboard"], "Reminders": ["Dashboard"],
 "Doctor search": ["Find Doctor", "Home"], "Appointment booking": ["Book Appointment", "Doctor Profile"], "Telemedicine": ["Telehealth"], "Medical records": ["Medical Records"],
 "Medication reminders": ["Dashboard"], "Privacy & security": ["Login", "Medical Records", "Profile"], "Billing & payments": ["Book Appointment", "Dashboard"], "Notifications": ["Dashboard"],
 "Secure authentication": ["Login"], "Balance overview": ["Dashboard", "Accounts"], "Fund transfer": ["Transfer"], "Transaction history": ["Transactions"], "Bill payments": ["Bills & Payments"],
 "Card management": ["Cards"], "Spending analytics": ["Dashboard"], "Alerts": ["Dashboard", "Transactions"],
}

COMP_RULES = [(re.compile(rx, re.I), txt) for rx, txt in [
 (r"search.*filter|filter.*search", "Search input (debounced 300 ms, clear button, labelled) combined with filter controls; applied filters shown as removable chips with a result count"),
 (r"search", "Search input with debounce, clear button, keyboard submit and an accessible label"),
 (r"filter", "Filter controls with an applied-filter summary, a clear-all action and a result count; state kept in the URL query string"),
 (r"sort", "Sort selector (for example recommended, price, rating) that updates the list without a full reload"),
 (r"card visual", "Presentational payment-card visual (masked number, holder name, expiry) that never shows full card data"),
 (r"time slot", "Time-slot grid with available and unavailable states; exactly one slot selectable; keyboard operable"),
 (r"date picker|calendar|date filter", "Date picker with past dates disabled, clear selected state and keyboard support"),
 (r"stepper|quantity", "Quantity stepper (minimum 1, maximum by stock) that updates totals immediately; buttons have aria-labels"),
 (r"promo|coupon", "Promo code field with an apply button, inline success or error message and recalculated totals"),
 (r"breakdown|summary", "Price or order summary (subtotal, fees, taxes, total) that recalculates when items change"),
 (r"timeline", "Status timeline showing completed, current and upcoming steps; changes announced to screen readers"),
 (r"map", "Map placeholder component behind a clear interface so a real map provider can be plugged in later"),
 (r"call controls", "Call control buttons (mute, camera, end) with labels, toggled states and keyboard shortcuts"),
 (r"video|call area", "Video area with play and pause, seek, captions toggle and playback speed; keyboard accessible"),
 (r"chat|message", "Chat panel with message list and composer, using mock responses"),
 (r"chart|analytics", "Accessible chart with a text or table alternative and clear legends"),
 (r"gallery|image", "Image gallery with thumbnails, zoom and alt text, with a graceful fallback when an image fails"),
 (r"title, price", "Product header showing title, price, rating and stock status"),
 (r"card", "Responsive card with image placeholder, title, key facts and one primary action; keyboard focusable when it navigates"),
 (r"review|rating", "Rating summary and review list with pagination and a clear empty state"),
 (r"otp|biometric|confirm", "Confirmation step requiring OTP or biometric (simulated by the mock) with retry and failure handling"),
 (r"toggle|freeze", "Toggle switch with optimistic update and rollback on failure"),
 (r"upload|download|export|statement", "File action with progress, success and error states (simulated by the mock)"),
 (r"accordion|syllabus", "Accordion with keyboard support and one section open by default"),
 (r"restaurant header|banner|welcome header|hero|course banner|photo and credentials", "Page header block with key facts, using heading levels correctly"),
 (r"navigation|tab bar|footer", "Global navigation component consistent across pages, with the active item indicated"),
 (r"category tabs|\btabs?\b", "Tabs with arrow-key navigation and a visible selected state"),
 (r"selector|dropdown|select", "Selection control (list or dropdown) with a visible label and keyboard support"),
 (r"form|field|details|address|amount", "Form fields with visible labels, correct input types, inline validation and an error summary"),
 (r"button|cta|link", "Action button or link with loading and disabled states; prevents double submission"),
 (r"avatar|logo", "Image or initials placeholder with alt text"),
 (r"privacy|notice|support|help", "Information block in plain language with a link to more detail"),
 (r"notification|alert", "Notification list with read and unread state and clear timestamps"),
 (r"stat", "Statistic cards with label, value and a short description"),
 (r"progress|indicator", "Progress indicator with a text equivalent for screen readers"),
 (r"timer", "Timer that can be paused where allowed and is announced accessibly"),
 (r"question|answer options", "Question card with answer options as a radio group, plus feedback after answering"),
 (r"list|grid|grouped|records", "Responsive list or grid with skeleton loading, empty state and an error state with retry"),
]]
def describe_component(name):
    for rx, txt in COMP_RULES:
        if rx.search(name): return txt
    return "Reusable, accessible component with clear props, and visible loading and empty states"
print("Prompt-generator knowledge loaded:", len(STACKS), "stacks,", len(PAGE_META), "page specs,", len(FEATURE_NOTES), "feature notes")

Prompt-generator knowledge loaded: 5 stacks, 27 page specs, 41 feature notes


In [40]:
#39
import zipfile, tempfile

def _slug(s): return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")
def _yn(x): return "not decided" if x is None else ("yes" if x else "no")
PLATFORM_WORDS = {"mobile": "mobile app", "web": "website / web app", "both": "responsive web app that works well on phones and desktops"}
def product_line(st): return f"A {PLATFORM_WORDS.get(st['platform'], 'app')} for {DOMAINS[st['domain']]['label'].lower()}, built for {st['target_users'] or 'general users'}"
def default_stack(platform): return {"mobile": "react-native", "web": "react", "both": "react"}.get(platform, "react")
def resolve_stack(p):
    s = getattr(p, "build_stack", "auto")
    return s if s in STACKS else default_stack(p.state["platform"])
def project_name(p): return p.state["brand_name"] or (p.name if p.name != "Untitled project" else DOMAINS[p.state["domain"]]["label"] + " app")
def _meta(page):
    return PAGE_META.get(page) or ("/" + _slug(page), f"Provide the '{page}' experience for this product", "Complete the main task on the page", (), ["empty data", "network error"])
def ROLE(stack): return f"You are a senior engineer experienced with {STACKS[stack]['label']}, who writes clean, accessible, well-tested code."

def token_lines(st):
    style = st["style"] or "modern"; pal = get_palette(st); fh, fb = FONT_STACKS[style]
    return [f"--color-primary: {pal['primary']};  /* text on it: {readable_on(pal['primary'])} */", f"--color-accent: {pal['accent']};  /* text on it: {readable_on(pal['accent'])} */",
            f"--color-bg: {pal['bg']};", f"--color-surface: {pal['surface']};", f"--color-text: {pal['text']};",
            f"--font-heading: {fh};", f"--font-body: {fb};", f"--radius: {RADIUS[style]};"]

def recap(p, stack):
    st = p.state; style = st["style"] or "modern"; pal = get_palette(st)
    return "\n".join(["## Context (short recap; the full brief is in the master prompt)",
        f"- Product: {product_line(st)}" + (f", called {st['brand_name']}" if st["brand_name"] else ""),
        f"- Stack: {STACKS[stack]['label']}", f"- Style: {style}; primary {pal['primary']}, accent {pal['accent']}; fonts: {STYLES[style]['fonts']}",
        f"- Login: {_yn(st['auth_needed'])}; payments: {_yn(st['payment_needed'])}", "- Pages: " + ", ".join(p.pages)])

def guidelines_for(p, page):
    st = p.state; ids = ["h_status", "a_contrast", "a_keyboard", "a_labels"] + (["a_targets"] if st["platform"] in ("mobile", "both") else [])
    out, seen = [], set()
    for d in [d for d, _ in retrieve_for_page(p, page, k=4)] + [KB_BY_ID[i] for i in ids]:
        if d["id"] not in seen: seen.add(d["id"]); out.append(d)
    return out[:7]

def _footer(): return ["", "## Reply format", "Start with a 3-line plan and your assumptions, then give complete files (path first, then full contents). End with the commands to run the tests and a list of files created or changed."]
def _build(role, p, stack, title, goal, tasks, constraints, accept, deliver, oos=None, extra=None):
    L = [f"# {title}", "", "## Role", role, "", recap(p, stack), "", "## Goal", goal, ""]
    if extra: L += extra + [""]
    L += ["## Tasks"] + [f"{i}. {t}" for i, t in enumerate(tasks, 1)] + ["", "## Constraints"] + [f"- {c}" for c in constraints]
    L += ["", "## Acceptance criteria"] + [f"- [ ] {a}" for a in accept] + ["", "## Deliverables"] + [f"- {d}" for d in deliver]
    L += ["", "## Out of scope"] + [f"- {o}" for o in (oos or ["Anything not listed in the tasks above; do not start other steps."])]
    return "\n".join(L + _footer())

COMMON_CONSTRAINTS = ["Use design tokens only: no hard-coded colors, font sizes or radii", "Keep components small and typed; no unused dependencies", "Do not invent features beyond the brief"]

def master_prompt(p, stack):
    st, S = p.state, STACKS[stack]; d = DOMAINS[st["domain"]]; style = st["style"] or "modern"
    L = [f"# Master project brief: {project_name(p)}", "> Save this as CLAUDE.md or AGENTS.md (or paste it first in a new chat). Every later step assumes it.", "",
         "## 1. Product", product_line(st) + ".", f"- Login required: {_yn(st['auth_needed'])}", f"- Payments: {_yn(st['payment_needed'])}",
         f"- Accessibility priority: {'high (target WCAG 2.2 AA)' if st['accessibility'] else 'standard (still target WCAG 2.2 AA basics)'}", "",
         "## 2. Users and core journey", f"- Users: {st['target_users'] or 'general users'}", "- Core journey: " + " -> ".join(d["journey"]), "", "## 3. Scope", "Pages:"]
    for pg in p.pages:
        r, purpose, *_ = _meta(pg); L.append(f"- {pg} (`{r}`): {purpose}")
    L += ["", "Features:"] + [f"- {f}: {FEATURE_NOTES.get(f, 'Implement end to end (UI, data, validation, tests).')}" for f in p.features]
    L += ["", "Out of scope: real payment processing, real SMS or email, real maps (use mocks behind interfaces); admin or back-office tools; features not listed above.", "",
          "## 4. Tech stack", f"- Framework: {S['label']}", f"- Routing: {S['router']}", f"- State and data: {S['state']}", f"- Testing: {S['test']}", f"- Mock data: {S['mock']}", f"- Accessibility tooling: {S['a11y']}", "",
          "## 5. Design system", f"- Style: {style}. {STYLES[style]['notes']}", "- Tokens:"] + [f"    {t}" for t in token_lines(st)]
    L += ["- Responsive: " + {"mobile": "design for 360 to 430 px screens, bottom tab bar, thumb-reachable actions.", "web": "breakpoints around 360, 768, 1024 and 1280 px; top navigation; hover and focus states.",
                             "both": "mobile-first; top navigation from 768 px up, bottom tab bar below."}.get(st["platform"], "mobile-first and responsive."), "",
          "## 6. UX and accessibility rules (always apply)"]
    for i in ["h_status", "a_contrast", "a_keyboard", "a_labels", "a_color", "n_cta", "p_states"] + (["a_targets"] if st["platform"] in ("mobile", "both") else []):
        g = KB_BY_ID[i]; L.append(f"- {g['title']}: {g['text']} ({g['source']})")
    L += ["", "## 7. Engineering conventions", "Folder structure:"] + [f"    {ln}" for ln in S["tree"].split("\n")]
    L += ["- Strictest type-checking and lint settings available; format on commit", "- Data only through repositories (never fetch directly in components)", "- Every component handles loading, empty and error states",
          "- Conventional commit messages; one logical change per commit", "", "## 8. How you (the AI assistant) should work",
          "- Work only on the step I give you; do not start other steps", "- State your assumptions briefly; if something is truly ambiguous, ask one question, otherwise choose the simplest option and say so",
          "- Make small, reviewable changes; list files created or changed", "- Write tests with the code and report how to run them", "- Never add dependencies without saying why", "",
          "## 9. Definition of done (every step)", "- Builds with no errors; type-check and lint pass", "- Tests pass; accessibility checks pass for the changed screens", "- Uses design tokens only", "- Short notes added to the README when behavior or setup changes"]
    return "\n".join(L)

def phase_setup(p, stack):
    S = STACKS[stack]
    return _build(ROLE(stack), p, stack, "Phase: Project setup and architecture", "Create the project skeleton so every later step has a stable base.",
        [f"Scaffold the project with the framework's official generator for {S['label']}.", "Enable the strictest type-checking and lint settings available, plus a formatter.",
         f"Set up routing ({S['router']}) with a placeholder screen for each route: " + ", ".join(f"{pg} ({_meta(pg)[0]})" for pg in p.pages) + ".",
         f"Set up state and data fetching ({S['state']}).", f"Set up testing ({S['test']}) with one passing example test.", "Create this folder structure:\n" + "\n".join("   " + ln for ln in S["tree"].split("\n")),
         "Add scripts for dev, build, test, lint and type-check, a `.env.example`, and a README with run instructions."],
        COMMON_CONSTRAINTS + ["Use current stable versions and pin them in the lockfile"],
        ["The app starts and every route shows its placeholder screen", "All scripts run successfully", "The example test passes", "The README explains how to install, run and test"],
        ["Project files", "README.md", ".env.example"])

def phase_design_system(p, stack):
    st = p.state
    nav = {"mobile": "bottom tab bar (first 4 pages plus a More sheet)", "web": "top navigation bar listing the pages", "both": "top navigation from 768 px up and a bottom tab bar below"}.get(st["platform"], "responsive navigation")
    return _build(ROLE(stack), p, stack, "Phase: Design system and app shell", "Build the shared visual language and layout so pages are assembled from consistent parts.",
        ["Implement these design tokens using the stack's theming mechanism:\n" + "\n".join("   " + t for t in token_lines(st)),
         "Build the components: Button (primary, secondary, loading, disabled), Input with label, hint and error, Select, Chip, Card, Badge, Quantity stepper, Tabs, Modal or bottom sheet, Toast, Skeleton, EmptyState, ErrorState with retry.",
         f"Build the app shell with {nav}; the active item must be indicated.", "Add a component gallery screen (or Storybook) that shows every component in every state.", "Add tests: each component renders, is keyboard operable and has no automatic accessibility violations."],
        COMMON_CONSTRAINTS + ["Text on primary and accent backgrounds uses the text colors given in the tokens (WCAG contrast)", "Touch targets at least " + ("48 dp / 44 pt" if st["platform"] in ("mobile", "both") else "40 px")],
        ["Changing a token changes the whole UI", "All components show visible focus and work with the keyboard", "Contrast of text and controls meets WCAG AA", "The shell works at 360, 768 and 1280 px widths"],
        ["Token file or theme", "Components under components/ui (or equivalent)", "App shell", "Component gallery", "Tests"])

def phase_data(p, stack):
    d = p.state["domain"]
    return _build(ROLE(stack), p, stack, "Phase: Data models and mock API", "Create typed data models and a mock data layer so pages can be built before any real backend exists.",
        ["Define typed models for these entities:\n" + "\n".join("   - " + e for e in DOMAIN_ENTITIES[d]), f"Create seed data (6 to 10 realistic records per list) using {STACKS[stack]['mock']}.",
         "Create repository functions (list, get by id, create, update where needed) that return typed results or typed errors; pages must only use these.",
         "Add simulated latency (300 to 800 ms) and a switch to inject random failures (about 10 percent) for testing error states.", "Add pagination and basic filtering helpers.", "Write tests for the repositories, including the error path."],
        COMMON_CONSTRAINTS + ["Replacing the mock with a real API later must only require changes inside the data layer"],
        ["All entities have types and seed data", "Repositories have tests, including failures", "Latency and failure switches work", "No page code fetches data directly"],
        ["Types", "Seed data", "Repositories and mock handlers", "Tests"])

def phase_auth(p, stack):
    d = p.state["domain"]
    extra = {"banking": ["Add automatic logout after inactivity and a biometric or OTP re-check (simulated) before sensitive actions."],
             "healthcare": ["Add a privacy and consent checkbox at sign-up and an inactivity timeout, because records are sensitive."]}.get(d, [])
    return _build(ROLE(stack), p, stack, "Phase: Authentication", "Add sign-up, login and session handling with a mock identity provider.",
        ["Build Sign up, Login and Forgot password flows (forms with inline validation and clear error messages).", "Create an auth store with a mock token, session persistence and logout.",
         "Protect the routes that need a signed-in user and redirect back to the requested page after login.", "Show the user's state in the app shell (signed in or out).", *extra, "Write tests for the success, wrong-password and expired-session cases."],
        COMMON_CONSTRAINTS + ["Never store passwords; the mock provider only issues fake tokens", "Error messages must not reveal whether an email exists"],
        ["A protected route redirects to login and returns after success", "Session survives a reload and ends on logout", "Forms are fully keyboard and screen-reader usable", "Tests cover the main failure cases"],
        ["Auth screens", "Auth store and mock provider", "Route guards", "Tests"])

def phase_payments(p, stack):
    return _build(ROLE(stack), p, stack, "Phase: Payments", "Add a payment layer behind an interface, using a mock provider.",
        ["Define a PaymentService interface (create payment, confirm, refund, list methods) and implement a mock provider with success, failure and delayed outcomes.", "Build the payment method selector and payment result screens (success with receipt, failure with retry).",
         "Use idempotency keys so a double click never creates two payments.", "Map provider errors to friendly messages.", "Write tests for success, failure, retry and double submission."],
        COMMON_CONSTRAINTS + ["Never store or log raw card data; a real integration would use the provider's hosted fields", "Payments run in test mode only"],
        ["The checkout flow can complete, fail and retry", "Double submission creates one payment", "Receipt shows amount, date and reference", "Swapping the mock for a real provider only touches the provider module"],
        ["PaymentService and mock provider", "Payment UI pieces", "Tests"], oos=["Real gateway credentials or live payments"])

def phase_features(p, stack):
    items = [f"{f}: {FEATURE_NOTES.get(f, 'Implement end to end (UI, data, validation, tests).')} (pages: {', '.join(x for x in FEATURE_PAGES.get(f, []) if x in p.pages) or 'see brief'})" for f in p.features]
    return _build(ROLE(stack), p, stack, "Phase: Cross-cutting features", "Make sure every selected feature works end to end across the pages that use it.",
        ["For each feature below, check the implementation across its pages and complete what is missing:\n" + "\n".join("   - " + i for i in items), "Fix inconsistencies (naming, behavior, copy) between pages that share a feature.", "Add or update tests so every feature has at least one test."],
        COMMON_CONSTRAINTS, ["Every listed feature can be demonstrated from the UI", "Shared behavior is consistent across pages", "Each feature has a test"], ["List of changes per feature", "Tests"])

def phase_qa(p, stack):
    d = DOMAINS[p.state["domain"]]
    return _build(ROLE(stack), p, stack, "Phase: Accessibility, QA and performance", "Verify quality across the whole product and fix what you find.",
        [f"Run automated accessibility checks ({STACKS[stack]['a11y']}) on every page and fix all serious violations.", "Walk through the core journey with the keyboard only: " + " -> ".join(d["journey"]) + ".",
         "Check with a screen reader (or the platform's accessibility inspector): names, roles, announcements for errors and status changes.", "Test layouts at 360, 768, 1024 and 1440 px (or common phone and tablet sizes) and with text zoom at 200 percent; respect reduced-motion settings.",
         "Check performance: lazy-load heavy parts, optimize images, avoid layout shift; record a before and after measurement.", "Write an end-to-end test for the core journey and tests for the main error states.", "Write QA_REPORT.md listing what was checked, what failed, what you fixed and what remains."],
        COMMON_CONSTRAINTS, ["No serious automated accessibility violations", "The core journey works with keyboard only", "End-to-end tests pass", "QA_REPORT.md is complete and honest about remaining issues"], ["Fixes", "End-to-end tests", "QA_REPORT.md"])

def phase_deploy(p, stack):
    host = {"react-native": "Expo's build service for test builds and Expo Go for demos", "flutter": "flutter build for the target platform, and a static host for the web build"}.get(stack, "a free static hosting tier such as Vercel, Netlify or GitHub Pages")
    return _build(ROLE(stack), p, stack, "Phase: Deployment and documentation", "Make the project reproducible, deployable and documented.",
        ["Add a CI pipeline that runs lint, type-check, tests and build on every push.", "Document environment variables and add production build settings.", f"Prepare deployment using {host}; write the exact steps.",
         "Complete the README: overview, setup, scripts, folder structure, architecture notes and known limitations.", "Add a short privacy note describing what user data the app handles (mock data only unless changed).", "List next steps for replacing mocks with a real backend."],
        COMMON_CONSTRAINTS, ["A clean clone can install, test, build and run following the README", "CI is green", "Deployment steps are written and tested at least once"], ["CI config", "Deployment instructions", "README.md"])

def page_prompt(p, page, stack, step_id):
    st = p.state; route, purpose, action, nexts, edges = _meta(page)
    nxt = [n for n in nexts if n in p.pages]; comps = components_for(page)
    feats = [f for f in p.features if page in FEATURE_PAGES.get(f, [])]
    ents = ", ".join(e.split("(")[0] for e in DOMAIN_ENTITIES[st["domain"]])
    plat = {"mobile": "Design for 360 to 430 px screens; primary actions within thumb reach; use the bottom tab bar from the app shell.",
            "web": "Breakpoints around 360, 768, 1024 and 1280 px; hover and focus states; full keyboard navigation.",
            "both": "Mobile-first, then scale up: single column on phones, multi-column from 768 px."}.get(st["platform"], "Mobile-first and responsive.")
    guides = guidelines_for(p, page)
    L = [f"# Step {step_id}: Build the {page} page", "", "## Role", ROLE(stack), "", recap(p, stack), "",
         "## Goal", f"Implement the **{page}** page at route `{route}`. Purpose: {purpose}.", "",
         "## User story", f"As a user ({st['target_users'] or 'general user'}), I want to: {action.lower()}, so that I can reach my goal quickly.", "", "## Layout and components"]
    L += [f"{i}. **{c}**: {describe_component(c)}" for i, c in enumerate(comps, 1)]
    L += ["", "## Data and state", f"- Read and write data only through the repositories from the data-layer step. Entities available: {ents}.", "- Keep UI state (selected filters, open panels) local or in the URL; keep server data in the data-fetching layer.", "",
          "## Navigation", (f"- Primary action ({action.lower()}) leads to: {', '.join(nxt)}." if nxt else "- Provide a clear way back to the main page."), "- The page is reachable from the app shell navigation or from the page that links to it.", "",
          "## States to implement", "- Loading: skeletons that match the final layout", "- Empty: friendly message plus a next action", "- Error: clear message with a retry button", "- Success: confirmation where an action completes", "",
          "## Edge cases"] + [f"- {e}" for e in edges]
    L += ["", "## Responsive and platform notes", f"- {plat}", "", "## Accessibility", "- Every interactive element is reachable by keyboard in a logical order, with a visible focus indicator",
          "- Every input has a visible label; errors are announced and linked to the field", "- Text and controls meet WCAG AA contrast using the design tokens", "- Images have alt text; meaning never relies on color alone",
          "", "## UX guidelines to honor (from the project's knowledge base)"] + [f"- {g['title']}: {g['text']} ({g['source']})" for g in guides]
    if feats: L += ["", "## Related features"] + [f"- {f}: {FEATURE_NOTES.get(f, 'Implement end to end.')}" for f in feats]
    accept = [f"The page renders at `{route}` with no console errors in loading, empty, error and success states"]
    accept += [f"{c}: {describe_component(c).split(';')[0].split(',')[0]}" for c in comps[:6]]
    accept += [f"Primary action leads to {', '.join(nxt)}" if nxt else "A clear way back is available"]
    accept += [f"Handles: {e}" for e in edges[:4]]
    accept += ["Usable with keyboard only and with a screen reader", "No horizontal scroll at 360 px; layout adapts at 768 and 1024 px", "Uses design tokens only (no hard-coded colors or sizes)"]
    L += ["", "## Acceptance criteria"] + [f"- [ ] {a}" for a in accept]
    L += ["", "## Tests to write", f"- Component tests for the main components on this page and its empty and error states", f"- One end-to-end test: open `{route}`, {action.lower()}, and verify the result" + (f" (lands on {nxt[0]})" if nxt else ""),
          "", "## Deliverables", "- The page, its sub-components and data hooks", "- Tests", "- A short note in the README if routes or behavior changed", "",
          "## Out of scope", "- Other pages, a real backend, real payments or maps, and any change to the design tokens"]
    return "\n".join(L + _footer())

def build_prompt_pack(p):
    st = p.state
    if not st["domain"]: raise ValueError("Finish the requirements chat first (at least the product type).")
    if not p.pages: p.pages = recommend_pages(st)
    if not p.features: p.features = recommend_features(st)[:5]
    stack = resolve_stack(p)
    seq = [("master", "Master project brief", None, lambda s: master_prompt(p, stack)), ("phase", "Project setup and architecture", None, lambda s: phase_setup(p, stack)),
           ("phase", "Design system and app shell", None, lambda s: phase_design_system(p, stack)), ("phase", "Data models and mock API", None, lambda s: phase_data(p, stack))]
    if st["auth_needed"]: seq.append(("phase", "Authentication", None, lambda s: phase_auth(p, stack)))
    for pg in p.pages: seq.append(("page", f"Page: {pg}", pg, (lambda pg_: (lambda s: page_prompt(p, pg_, stack, s)))(pg)))
    if st["payment_needed"]: seq.append(("phase", "Payments", None, lambda s: phase_payments(p, stack)))
    seq += [("phase", "Cross-cutting features", None, lambda s: phase_features(p, stack)), ("phase", "Accessibility, QA and performance", None, lambda s: phase_qa(p, stack)),
            ("phase", "Deployment and documentation", None, lambda s: phase_deploy(p, stack))]
    pack = []
    for n, (kind, title, page, fn) in enumerate(seq):
        sid = f"{n:02d}"; text = fn(sid)
        pack.append({"id": sid, "kind": kind, "title": title, "page": page, "file": f"{sid}_{kind}_{_slug(title.replace('Page: ', ''))}.md", "text": text, "tokens": int(len(text.split()) * 1.4)})
    return pack

_PACK_CACHE = {}
def get_pack(p):
    sig = json.dumps([p.state, p.pages, p.features, resolve_stack(p)], sort_keys=True, default=str)
    c = _PACK_CACHE.get(p.id)
    if c and c[0] == sig: return c[1]
    pack = build_prompt_pack(p); _PACK_CACHE[p.id] = (sig, pack); return pack

def pack_index_md(p, pack):
    stack = resolve_stack(p)
    L = [f"# Development prompt pack: {project_name(p)}", "", f"Stack: **{STACKS[stack]['label']}**. Generated by the AI UX Design Assistant.", "", "## How to use",
         "1. Start a new session with your coding assistant. Save step 00 as CLAUDE.md or AGENTS.md in the project root, or paste it as the first message.",
         "2. Give the assistant one step at a time, in order. Do not skip steps; later steps rely on earlier ones.", "3. After each step, check its acceptance criteria, run the tests, and commit.",
         "4. If the assistant loses context, paste step 00 again plus the step you are on.", "5. Edit any prompt freely; they are plain Markdown.", "", "| Step | Type | Title | Approx. tokens |", "|---|---|---|---|"]
    return "\n".join(L + [f"| {it['id']} | {it['kind']} | {it['title']} | {it['tokens']} |" for it in pack])

def export_prompt_pack(p):
    pack = get_pack(p); d = tempfile.mkdtemp(prefix="uxa_prompts_")
    for it in pack: open(os.path.join(d, it["file"]), "w", encoding="utf-8").write(it["text"])
    open(os.path.join(d, "README.md"), "w", encoding="utf-8").write(pack_index_md(p, pack))
    open(os.path.join(d, "all_prompts_combined.md"), "w", encoding="utf-8").write("\n\n---\n\n".join(it["text"] for it in pack))
    zpath = os.path.join(tempfile.gettempdir(), f"dev_prompts_{p.id}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(os.listdir(d)): z.write(os.path.join(d, f), f)
    return zpath, pack

def check_prompt_pack(p, pack=None):
    pack = pack or build_prompt_pack(p); st = p.state; rows = []
    def chk(name, ok): rows.append({"check": name, "passed": bool(ok)})
    chk("master brief is step 00", pack[0]["kind"] == "master" and pack[0]["id"] == "00")
    chk("every page has its own prompt", all(any(it["page"] == pg for it in pack) for pg in p.pages))
    chk("every prompt has acceptance criteria", all("Acceptance criteria" in it["text"] or it["kind"] == "master" for it in pack))
    chk("page prompts have at least 8 criteria", all(it["text"].count("- [ ]") >= 8 for it in pack if it["kind"] == "page"))
    chk("no unresolved placeholders", all(not re.search(r"\{[a-z_]+\}|\bNone\b|\bnan\b|undefined", it["text"]) for it in pack))
    chk("no code fences inside prompts (safe to display and copy)", all("```" not in it["text"] for it in pack))
    chk("master contains the design tokens", get_palette(st)["primary"] in pack[0]["text"])
    chk("master names the chosen stack", STACKS[resolve_stack(p)]["label"] in pack[0]["text"])
    chk("auth step exists only if login is needed", any(it["title"] == "Authentication" for it in pack) == bool(st["auth_needed"]))
    chk("payments step exists only if payments are needed", any(it["title"] == "Payments" for it in pack) == bool(st["payment_needed"]))
    chk("every prompt is under 4500 tokens", all(it["tokens"] < 4500 for it in pack))
    chk("step ids are sequential and files unique", [it["id"] for it in pack] == [f"{i:02d}" for i in range(len(pack))] and len({it["file"] for it in pack}) == len(pack))
    return pd.DataFrame(rows)
print("Prompt generator ready")

Prompt generator ready


In [41]:
#43b
needed = {"Cell 2": ["gr", "nlp", "embed", "SAVE_DIR"], "Cell 3": ["DOMAINS", "STYLES", "COLOR_HEX", "PAGE_COMPONENTS", "ALL_PAGES"],
          "Cell 4": ["new_state", "extract_all", "classify_domain", "extract_features", "extract_colors"],
          "Cell 5": ["ask_text", "update_from_answer", "snapshot"], "Cell 6": ["recommend_pages", "get_palette", "match_name", "components_for"],
          "Cell 7": ["Project"], "Cell 12": ["KB_BY_ID"], "Cell 13": ["retrieve_for_page"]}
missing = {c: [n for n in names if n not in globals()] for c, names in needed.items()}
missing = {c: m for c, m in missing.items() if m}
if missing:
    print("Run these cells first, then try again:")
    for c, m in missing.items(): print(f"  {c}: missing {m}")
else:
    print("All prerequisites are loaded. You can run Cells 44 to 48.")

All prerequisites are loaded. You can run Cells 44 to 48.


In [42]:
#44
import re, os, json, datetime, tempfile, sys

YN = "yn"
PAGE_QUESTIONS = {   # page: [(key, question, YN or None)]
 "Home": [("focus", "What is the **single most important thing** a visitor should do on the Home page?", None),
          ("content", "What should be **featured** on Home? (for example popular items, offers, search, categories)", None),
          ("first_time", "What should a first-time visitor understand within 5 seconds?", None)],
 "Login": [("fields", "What should the login page **ask for**? (email, phone, password, OTP, social sign-in)", None),
           ("extras", "Do you need **sign-up, forgot password, or continue as guest** on this page?", None)],
 "Restaurant List": [("filters", "Which **filters and sorting** do users need? (cuisine, price, rating, delivery time, veg only)", None),
                     ("card", "What should each **restaurant card** show?", None)],
 "Menu": [("organise", "How should **dishes be organised**? (categories, photos, dietary tags)", None),
          ("options", "Do dishes have **options** such as size, toppings or add-ons?", YN),
          ("rules", "Any **ordering rules**? (minimum order, one restaurant per order, availability times)", None)],
 "Cart": [("edit", "What can users **change in the cart**? (quantity, notes, remove items, save for later)", None),
          ("promo", "Do you need **promo codes or discounts**? Any fee or minimum-order rules?", None)],
 "Checkout": [("details", "What **details** do you need at checkout? (address, phone, delivery time, instructions)", None),
              ("methods", "Which **payment methods** should checkout support?", None),
              ("guest", "Can guests check out **without an account**?", YN)],
 "Order Tracking": [("statuses", "Which **order statuses** should customers see? (confirmed, preparing, out for delivery, delivered)", None),
                    ("live", "Do you need a **live map**, delivery partner details or a call button?", None)],
 "Profile": [("manage", "What can users **manage** in their profile? (details, addresses, saved items, preferences)", None),
             ("privacy", "Do you need **account deletion, notification settings or privacy controls**?", YN)],
 "Product List": [("filters", "Which **filters and sorting** do shoppers need? (price, brand, size, rating)", None),
                  ("card", "What should each **product card** show?", None)],
 "Product Detail": [("info", "What **information** must a product page show? (photos, variants, specs, reviews, return policy)", None),
                    ("cta", "What is the **main action**? (add to cart, buy now, wishlist, enquire)", None)],
 "Order History": [("show", "What should each past order show, and which **actions** are allowed? (track, reorder, invoice, return)", None)],
 "Wishlist": [("rules", "What can users **do with wishlist items**? (move to cart, share, price alerts)", None)],
 "Course Catalog": [("browse", "How should learners **find courses**? (categories, search, level, price, rating)", None),
                    ("card", "What should each **course card** show?", None)],
 "Course Detail": [("content", "What must a course page show? (syllabus, instructor, duration, reviews, certificate)", None),
                   ("enroll", "How do learners **join**? (free enrolment, payment, approval)", None)],
 "Lesson Player": [("media", "Which **lesson formats** do you need? (video, reading, slides, downloads)", None),
                   ("progress", "How should **progress** be tracked? (automatic completion, notes, bookmarks)", None)],
 "Quiz": [("types", "Which **question types** do you need? (multiple choice, true or false, short answer)", None),
          ("rules", "What are the **rules**? (time limit, retries, passing score, feedback after each question)", None)],
 "Dashboard": [("widgets", "What are the **3 to 5 most important things** a user should see on the dashboard?", None),
               ("actions", "Which **quick actions** should be one tap away?", None)],
 "Find Doctor": [("filters", "How should patients **search**? (specialty, location, availability, fee, language)", None),
                 ("card", "What should each **doctor card** show?", None)],
 "Doctor Profile": [("info", "What must a doctor profile show? (qualifications, experience, fees, languages, reviews)", None),
                    ("mode", "How do patients book: **video, in-clinic, or both**?", None)],
 "Book Appointment": [("slots", "How are **appointment slots** defined? (duration, working hours, buffer time)", None),
                      ("details", "What do you need from the patient when booking? (reason, documents, insurance)", None),
                      ("changes", "Can patients **reschedule or cancel**? Any deadlines or fees?", YN)],
 "Medical Records": [("types", "Which **records** are stored? (reports, prescriptions, scans, vaccination)", None),
                     ("access", "Who can **see or upload** records, and how should consent and privacy work?", None)],
 "Telehealth": [("tools", "What do you need in a **video consultation**? (chat, file sharing, notes, prescription)", None),
                ("fallback", "What should happen if the **video fails**? (chat, phone call, reschedule)", None)],
 "Accounts": [("show", "What should each **account card** show? (balance, number, type, recent activity)", None),
              ("types", "Which **account types** exist? (savings, current, credit, deposits)", None)],
 "Transfer": [("types", "Which **transfers** do you support? (own accounts, other banks, scheduled, international)", None),
              ("security", "How should transfers be **confirmed**? (OTP, biometrics, PIN) Any limits?", None)],
 "Transactions": [("find", "How should users **find transactions**? (search, dates, categories, amount)", None),
                  ("export", "Do users need **statements or export**? (PDF, CSV)", YN)],
 "Cards": [("controls", "Which **card controls** are needed? (freeze, limits, online use, international use, PIN change)", None)],
 "Bills & Payments": [("types", "Which **bill types** are supported? (electricity, mobile, internet, water, rent)", None),
                      ("recurring", "Do you need **saved billers, reminders or recurring payments**?", YN)],
}
def page_questions(page):
    return PAGE_QUESTIONS.get(page) or [("purpose", f"What is the main goal of the **{page}** page?", None),
                                       ("content", f"What information or content must appear on the **{page}** page?", None),
                                       ("actions", f"What can users do on the **{page}** page, and where should they go next?", None)]

def _G(qid, label, text, field, quick=None, when=None): return {"id": qid, "label": label, "text": text, "field": field, "quick": quick, "when": when}
GENERAL = [
 _G("domain", "product type", None, "domain"),
 _G("brand_name", "product name", "What is the **name** of your product, or a working title? (or type **skip**)", "brand_name"),
 _G("goal", "main goal", "What is the **main goal** of your product, or the problem it solves for users? One or two sentences is enough (or **skip**).", "goal"),
 _G("target_users", "target users", None, "target_users"),
 _G("pain_points", "user problems", "What **problems or frustrations** do your users have today that this product should fix? (or **skip**)", "pain_points"),
 _G("platform", "platform", None, "platform", ["mobile app", "website", "both", "skip"]),
 _G("features", "key features", None, "features"),
 _G("auth_needed", "login", None, "auth_needed", ["Yes", "No", "skip"]),
 _G("auth_methods", "sign-in methods", "How should people **sign in**? (email and password, phone and OTP, Google or social login, other)", "auth_methods", None, lambda st: st.get("auth_needed") is True),
 _G("payment_needed", "payments", None, "payment_needed", ["Yes", "No", "skip"]),
 _G("payment_methods", "payment methods", "Which **payment methods** should be supported? (cards, UPI or wallets, cash on delivery, bank transfer, other)", "payment_methods", None, lambda st: st.get("payment_needed") is True),
 _G("style", "visual style", None, "style", list(STYLES) + ["skip"]),
 _G("brand_colors", "brand colors", "Do you have **brand colors**? Name them or give hex codes (for example #1E88E5), or type **none**.", "brand_colors", ["none", "skip"]),
 _G("accessibility_notes", "accessibility needs", "Any **accessibility needs** to design for? (screen readers, large text, color-blind friendly, older users, none)", "accessibility_notes", ["none", "skip"]),
 _G("languages", "languages", "Which **languages** should the product support?", "languages"),
 _G("inspiration", "inspiration", "Name **1 to 3 apps or websites** you like, or that compete with yours (or **skip**).", "inspiration"),
 _G("constraints", "constraints", "Any **constraints**? For example budget, timeline, technology or legal rules. (Type **none** to continue.)", "constraints", ["none", "skip"]),
 _G("success", "success measure", "How will you know the product is **successful** after 3 months? (for example sign-ups, orders, completed lessons)", "success_metric"),
 _G("pages_confirm", "page list", None, "pages_confirmed", ["ok"]),
]
GQ = {g["id"]: g for g in GENERAL}

def _is_page_q(qid): return qid.startswith("pg::")
def _split_pq(qid): _, page, key = qid.split("::", 2); return page, key
def _default_pages(st): return recommend_pages(dict(st, auth_needed=True if st.get("auth_needed") is None else st["auth_needed"]))

def iq_ids(p):
    st = p.state; ids = [g["id"] for g in GENERAL if g["when"] is None or g["when"](st)]
    if st.get("pages_confirmed"):
        for pg in p.pages: ids += [f"pg::{pg}::{k}" for k, _, _ in page_questions(pg)]
    return ids

def iq_filled(p, qid):
    st = p.state
    if qid in st.get("skipped", []): return True
    if _is_page_q(qid):
        page, key = _split_pq(qid); return key in st.get("page_answers", {}).get(page, {})
    if qid == "features": return len(st.get("features", [])) >= 3 or "features" in st.get("asked", [])
    if qid == "brand_colors": return bool(st.get("brand_colors"))
    return st.get(GQ[qid]["field"]) not in (None, "", [])

def iq_next(p):
    for qid in iq_ids(p):
        if not iq_filled(p, qid): return qid
    return None

def iq_counts(p):
    ids = iq_ids(p); gen = [i for i in ids if not _is_page_q(i)]; pgs = [i for i in ids if _is_page_q(i)]
    return sum(iq_filled(p, i) for i in gen), len(gen), sum(iq_filled(p, i) for i in pgs), len(pgs)

def iq_label(p, qid):
    if _is_page_q(qid): return f"{_split_pq(qid)[0]} page"
    return GQ[qid]["label"]

def iq_text(p, qid):
    st = p.state
    if _is_page_q(qid):
        page, key = _split_pq(qid); qs = page_questions(page); j = [k for k, _, _ in qs].index(key)
        i = p.pages.index(page) + 1 if page in p.pages else 1
        return f"**Page {i} of {len(p.pages)}: {page}** (question {j + 1} of {len(qs)}). " + qs[j][1]
    if qid == "domain":
        c = st.get("domain_candidates")
        if c: return f"I can't tell whether this is **{DOMAINS[c[0]]['label']}** or **{DOMAINS[c[1]]['label']}**. Which is closer? (You can also choose another type.)"
        return ask_text(st, "domain")
    if qid == "features":
        lines = [f"{i}. **{n}**: {desc}" for i, (n, desc) in enumerate(DOMAINS[st["domain"]]["features"].items(), 1)]
        return "Which **features** matter most? Reply with numbers (for example 1, 3, 5), feature names, **recommended** (I pick a core set), or **skip**.\n\n" + "\n".join(lines)
    if qid == "pages_confirm":
        if not p.pages: p.pages = _default_pages(st)
        return ("Here are the pages I recommend: " + ", ".join(p.pages) + ".\n\nI will ask a few questions about **each page**. Reply **ok** to continue, or change the list with `add <page>` or `remove <page>`.")
    g = GQ[qid]
    return g["text"] if g["text"] else ask_text(st, qid)

def iq_question(p, qid):
    if _is_page_q(qid): return iq_text(p, qid)
    gen = [i for i in iq_ids(p) if not _is_page_q(i)]
    return f"**Question {gen.index(qid) + 1} of {len(gen)}.** " + iq_text(p, qid)

def iq_quick(p):
    st = p.state; qid = p.pending
    if p.stage_done.get("requirements") and not qid: return ["report", "status", "change style", "change pages"]
    if qid is None: return []
    if _is_page_q(qid):
        page, key = _split_pq(qid); q = next(x for x in page_questions(page) if x[0] == key)
        return (["Yes", "No"] if q[2] == YN else []) + ["skip", "skip page"]
    if qid == "domain":
        c = st.get("domain_candidates") or []
        return [DOMAINS[k]["label"] for k in c] + [v["label"] for k, v in DOMAINS.items() if k not in c]
    if qid == "features": return ["recommended", "all", "skip"]
    return GQ[qid]["quick"] or ["skip"]

def _iq_features(st, text):
    names = list(DOMAINS[st["domain"]]["features"]); low = text.lower(); chosen = []
    if re.search(r"\b(recommended|recommend|you choose|you pick|suggest|core|default)\b", low): chosen = names[:5]
    elif re.search(r"\ball\b", low): chosen = names[:]
    else:
        for n in re.findall(r"\d+", low):
            i = int(n)
            if 1 <= i <= len(names) and names[i - 1] not in chosen: chosen.append(names[i - 1])
        for part in re.split(r",|;|\band\b|\n", text):
            part = part.strip()
            if len(part) >= 3:
                m = match_name(part, names, 0.55)
                if m and m not in chosen: chosen.append(m)
        for f in extract_features(text, st["domain"]):
            if f not in chosen: chosen.append(f)
    if not chosen: return False
    chosen += [f for f in st.get("features", []) if f not in chosen]
    st["features"] = chosen; st.setdefault("asked", []).append("features")
    return True

def _iq_colors(st, t):
    if re.match(r"^(none|no|nothing|nope|n/a|not yet)\b", t.lower()): st["brand_colors"] = "none"; return True
    hexes = re.findall(r"#[0-9a-fA-F]{6}\b", t); names = extract_colors(t)
    if len(t) < 3: return False
    st["brand_colors"] = t
    if hexes: st["brand_hex"] = hexes[0].upper()
    for c in names:
        if c not in st["colors"]: st["colors"].append(c)
    return True

def _iq_pages(p, t):
    st = p.state; low = t.lower().strip()
    if not p.pages: p.pages = _default_pages(st)
    m = re.match(r"^(add|include)\s+(.+)", low)
    if m:
        q = m.group(2).strip(" ."); name = match_name(q, ALL_PAGES, 0.6) or q.title()
        if name not in p.pages: p.pages.append(name)
        return "stay"
    m = re.match(r"^(remove|delete|drop)\s+(.+)", low)
    if m:
        name = match_name(m.group(2).strip(" ."), p.pages)
        if name: p.pages.remove(name)
        if not p.pages: p.pages = _default_pages(st)
        return "stay"
    if re.match(r"^(ok|okay|yes|y|done|continue|go|looks good|sure|fine)\b", low): st["pages_confirmed"] = True; return True
    return False

def iq_update(p, qid, text):
    """True = accepted, False = not understood, 'stay' = accepted but ask the same question again."""
    st = p.state; t = text.strip(); low = t.lower()
    if low in ("skip", "skip it", "pass", "later") and qid not in ("domain", "pages_confirm"):
        st["skipped"].append(qid); return True
    if _is_page_q(qid):
        page, key = _split_pq(qid)
        if len(t) < 2: return False
        st.setdefault("page_answers", {}).setdefault(page, {})[key] = t; return True
    if qid == "pages_confirm": return _iq_pages(p, t)
    if qid == "features": return _iq_features(st, t)
    if qid in ("domain", "target_users", "platform", "auth_needed", "payment_needed", "style"): return update_from_answer(st, qid, t)
    if qid == "brand_colors": return _iq_colors(st, t)
    if qid == "accessibility_notes":
        if re.match(r"^(none|no|nothing|nope|n/a)\b", low): st["accessibility_notes"] = "none"; return True
        if len(t) >= 3: st["accessibility_notes"] = t; st["accessibility"] = True; return True
        return False
    field = GQ[qid]["field"]
    if re.match(r"^(none|no|nothing|nope|n/a)\b", low): st[field] = "none"; return True
    if len(t) >= 3: st[field] = t; return True
    return False

def _product_line(st):
    plat = {"mobile": "mobile app", "web": "website / web app", "both": "responsive web app that works on phones and desktops"}.get(st.get("platform"), "app or website")
    return f"A {plat} for {DOMAINS[st['domain']]['label'].lower()}, built for {st.get('target_users') or 'general users'}"

def _iq_value(st, qid):
    if qid == "domain": return DOMAINS[st["domain"]]["label"] if st.get("domain") else None
    if qid == "features": return ", ".join(st.get("features", [])) or None
    if qid == "pages_confirm": return "confirmed" if st.get("pages_confirmed") else None
    v = st.get(GQ[qid]["field"])
    if qid in ("auth_needed", "payment_needed") and v is not None: return "yes" if v else "no"
    return v

def _iq_status(p):
    st = p.state; gd, gt, pd_, pt = iq_counts(p)
    lines = [f"**Where we are:** general questions {gd}/{gt}" + (f", page questions {pd_}/{pt}" if pt else ""), ""]
    for qid in [i for i in iq_ids(p) if not _is_page_q(i)]:
        val = _iq_value(st, qid); tag = " (skipped)" if qid in st.get("skipped", []) else ""
        lines.append(f"- {GQ[qid]['label'].capitalize()}: {val if val not in (None, '', []) else 'not answered'}{tag}")
    if st.get("pages_confirmed"):
        lines += ["", "**Pages:**"]
        for pg in p.pages:
            n_ok = sum(1 for k, _, _ in page_questions(pg) if k in st.get("page_answers", {}).get(pg, {}))
            lines.append(f"- {pg}: {n_ok}/{len(page_questions(pg))} answered")
    return "\n".join(lines + ["", "Commands: `change platform`, `change Checkout`, `skip page`, `skip general`, `report`."])

IQ_ALIASES = [("product name", "brand_name"), ("name", "brand_name"), ("brand color", "brand_colors"), ("colour", "brand_colors"), ("color", "brand_colors"),
              ("product", "domain"), ("type", "domain"), ("domain", "domain"), ("goal", "goal"), ("problem", "pain_points"), ("pain", "pain_points"),
              ("user", "target_users"), ("audience", "target_users"), ("platform", "platform"), ("feature", "features"), ("sign", "auth_methods"),
              ("login", "auth_needed"), ("account", "auth_needed"), ("payment method", "payment_methods"), ("payment", "payment_needed"),
              ("style", "style"), ("look", "style"), ("accessib", "accessibility_notes"), ("language", "languages"), ("inspiration", "inspiration"),
              ("compet", "inspiration"), ("constraint", "constraints"), ("budget", "constraints"), ("timeline", "constraints"), ("success", "success"),
              ("page list", "pages_confirm"), ("pages", "pages_confirm")]

def _iq_change(p, what):
    st = p.state; w = what.lower()
    page = next((pg for pg in p.pages if pg.lower() in w), None)
    if page and st.get("pages_confirmed"):
        st.get("page_answers", {}).pop(page, None)
        keys = [f"pg::{page}::{k}" for k, _, _ in page_questions(page)]
        st["skipped"] = [s for s in st["skipped"] if s not in keys]
        p.pending = keys[0]; p.stage_done["requirements"] = False
        return f"Okay, let's redo the **{page}** page.\n\n" + iq_question(p, keys[0])
    qid = next((q for key, q in IQ_ALIASES if key in w), None)
    if qid is None: return "Which answer do you want to change? For example: `change platform`, `change style`, `change goal`, or a page name such as `change Checkout`."
    if qid == "features": st["features"] = []; st["asked"] = [a for a in st.get("asked", []) if a != "features"]
    elif qid == "brand_colors": st["brand_colors"] = None; st["colors"] = []; st.pop("brand_hex", None)
    elif qid == "domain":
        st["domain"] = None; st["features"] = []; st["asked"] = []; st["pages_confirmed"] = False; st["page_answers"] = {}; p.pages = []; st.pop("domain_candidates", None)
    elif qid == "pages_confirm": st["pages_confirmed"] = False
    else: st[GQ[qid]["field"]] = None
    st["skipped"] = [s for s in st["skipped"] if s != qid]
    if qid not in iq_ids(p): return "That question does not apply to your current answers."
    p.pending = qid; p.stage_done["requirements"] = False
    return f"Okay, let's change the **{iq_label(p, qid)}**.\n\n" + iq_question(p, qid)

def _iq_finish(p, force=False):
    st = p.state
    if st["domain"] is None:
        p.pending = "domain"; return "I need at least the product type before I can write the report.\n\n" + iq_question(p, "domain")
    still_open = iq_next(p) is not None
    p.pending = None; p.stage_done["requirements"] = True
    gd, gt, pd_, pt = iq_counts(p); n_pages = len(p.pages or _default_pages(st))
    msg = ["✅ " + ("Here is what I have so far." if force and still_open else "All questions answered.") + " **Your detailed report is ready.**", "",
           f"- {_product_line(st)}", (f"- {n_pages} pages with page-by-page requirements ({pd_} of {pt} page questions answered)" if pt else f"- {n_pages} recommended pages"),
           f"- General answers: {gd} of {gt}", "- UX guidance with sources, accessibility checks and a visual design direction"]
    return "\n".join(msg + ["", "Open the **Report** tab to read it and download it as Word or Markdown. To change an answer, type for example `change style` or `change Checkout`, then `report` again."])

def _iq_advance(p, prefix="", prev=None):
    p.pending = None; qid = iq_next(p)
    if qid:
        p.pending = qid
        if prev == "pages_confirm" and _is_page_q(qid):
            prefix += "Great! Now a few questions about **each page**. Type `skip` to skip one question, or `skip page` to skip the rest of a page.\n\n"
        return prefix + iq_question(p, qid)
    return prefix + _iq_finish(p)

def _iq_skip_page(p):
    if not p.pending or not _is_page_q(p.pending): return "`skip page` works while I'm asking about a page. Use `skip` to skip a single question."
    page, _ = _split_pq(p.pending); prev = p.pending
    for k, _, _ in page_questions(page):
        qid = f"pg::{page}::{k}"
        if not iq_filled(p, qid): p.state["skipped"].append(qid)
    return _iq_advance(p, f"Skipped the rest of the **{page}** page.\n\n", prev)

def _iq_skip_general(p):
    for qid in [i for i in iq_ids(p) if not _is_page_q(i) and i not in ("domain", "pages_confirm")]:
        if not iq_filled(p, qid): p.state["skipped"].append(qid)
    return _iq_advance(p, "Skipped the remaining general questions.\n\n", p.pending)

def handle_interview(p, text):
    st = p.state; t = text.strip(); low = t.lower()
    for k, v in (("asked", []), ("skipped", []), ("page_answers", {})): st.setdefault(k, v)
    first_turn = not any(u for u, _ in p.histories.get("requirements", []))
    if not st.get("idea") and len(t) >= 15 and not re.match(r"^(report|status|change|redo|edit|skip)\b", low): st["idea"] = t
    if re.match(r"^(report|finish|skip all|generate( the)? report)\s*$", low): return _iq_finish(p, force=True)
    if re.match(r"^(status|progress|where are we)\s*$", low): return _iq_status(p)
    m = re.match(r"^(change|redo|edit)\s+(.+)", low)
    if m: return _iq_change(p, m.group(2))
    if re.match(r"^skip (this )?page\s*$", low): return _iq_skip_page(p)
    if re.match(r"^skip (the )?(general|section)( questions)?\s*$", low): return _iq_skip_general(p)
    if p.pending not in iq_ids(p): p.pending = None
    if not (p.pending and _is_page_q(p.pending)):            # page answers are never scanned for general slots
        extract_all(t, st)
        if st["domain"] is None and len(t) >= 12:
            d_, _s, sc = classify_domain(t); top = sorted(sc, key=sc.get, reverse=True)[:2]
            if d_ is None and sc[top[0]] >= 0.25: st["domain_candidates"] = top
        if st["domain"] is not None: st.pop("domain_candidates", None)
    prefix = ""; prev = p.pending
    if p.pending:
        qid = p.pending
        res = True if iq_filled(p, qid) else iq_update(p, qid, t)
        if res == "stay": return "Updated.\n\n" + iq_question(p, qid)
        if res: p.retries.pop(qid, None)
        else:
            p.retries[qid] = p.retries.get(qid, 0) + 1
            if qid != "domain" and p.retries[qid] >= 2:
                st["skipped"].append(qid); prefix = "No problem, I'll leave that one open and note it in the report.\n\n"
            else: return "Sorry, I didn't catch that.\n\n" + iq_question(p, qid)
    if first_turn:
        snap = snapshot(st); prefix += (snap + "\n\n") if snap else ""
    return _iq_advance(p, prefix, prev)
print("Interview engine ready:", len(GENERAL), "general questions,", len(PAGE_QUESTIONS), "pages with their own questions")

Interview engine ready: 19 general questions, 27 pages with their own questions


In [43]:
#45
RP_TMP = tempfile.mkdtemp(prefix="uxreport_")

def _lum(hexc):
    h = hexc.lstrip("#"); r, g, b = [int(h[i:i + 2], 16) / 255 for i in (0, 2, 4)]
    f = lambda c: c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * f(r) + 0.7152 * f(g) + 0.0722 * f(b)
def _contrast(a, b):
    la, lb = sorted([_lum(a), _lum(b)], reverse=True); return (la + 0.05) / (lb + 0.05)
def _best_text(bg): return "#FFFFFF" if _contrast("#FFFFFF", bg) >= _contrast("#111111", bg) else "#111111"
def rp_palette(st):
    pal = get_palette(dict(st, colors=[c for c in st.get("colors", []) if c in COLOR_HEX]))
    return dict(pal, primary=st["brand_hex"]) if st.get("brand_hex") else pal

DOMAIN_RISKS = {
 "food_ordering": ["Order accuracy and delivery-time promises: show an ETA and clear cancellation rules", "Payment failures and duplicate orders", "Allergen and dietary information must be accurate and easy to find"],
 "ecommerce": ["Cart abandonment at checkout: show total cost early and offer guest checkout", "Returns and refunds must be clear before purchase", "Stock accuracy and trust in reviews"],
 "education": ["Learners dropping out after the first lessons: use progress, reminders and small wins", "Video accessibility: captions and transcripts", "Fairness and integrity of quizzes and assignments"],
 "healthcare": ["Privacy of health data and explicit consent (check the local regulations)", "The product must not give medical advice unless it is designed and approved for that", "Accessibility for older users and for stressful situations"],
 "banking": ["Security: multi-factor authentication, session timeouts and fraud alerts", "Regulatory requirements (identity checks, local financial rules) must be confirmed with an expert", "Money movement needs clear confirmation, error and receipt flows"],
}
PLATFORM_NOTES = {
 "mobile": ["Bottom tab bar with 3 to 5 destinations", "Touch targets of at least 44 pt (Apple) or 48 dp (Material)", "Design for 360 to 430 px wide screens with thumb-reachable primary actions", "Plan for slow or offline networks"],
 "web": ["Top navigation with visible search and breadcrumbs for deep pages", "Responsive breakpoints around 360, 768, 1024 and 1280 px", "Full keyboard navigation with visible focus", "Plan for page speed and search-engine visibility"],
 "both": ["Mobile-first design that scales up to desktop", "One design system (tokens and components) for both", "Top navigation on large screens and a bottom tab bar on phones", "Test on real phones and on desktop browsers"],
}
def project_title(p): return p.state.get("brand_name") or (p.name if p.name != "Untitled project" else DOMAINS[p.state["domain"]]["label"] + " project")

def build_detailed_report(p):
    st = p.state; d = DOMAINS[st["domain"]]; style = st["style"] or "modern"; pal = rp_palette(st)
    title = project_title(p); B = []
    H = lambda l, t: B.append(("h", l, t)); P = lambda t: B.append(("p", t)); UL = lambda x: B.append(("ul", list(x)))
    OL = lambda x: B.append(("ol", list(x))); T = lambda head, rows: B.append(("table", head, rows))
    num = iter(range(1, 100)); S = lambda t: H(1, f"{next(num)}. {t}")
    skipped = st.get("skipped", []); pa = st.get("page_answers", {})
    eff_auth = True if st["auth_needed"] is None else st["auth_needed"]; eff_pay = True if st["payment_needed"] is None else st["payment_needed"]
    plat = st["platform"] or "both"; pages = list(p.pages) if p.pages else _default_pages(st)
    catalog = d["features"]; chosen = list(st.get("features", [])); feats_assumed = not chosen
    if feats_assumed: chosen = list(catalog)[:5]
    extras = [f for f in catalog if f not in chosen][:4]
    gen_ids = [i for i in iq_ids(p) if not _is_page_q(i)]
    open_gen = [GQ[i]["label"] for i in gen_ids if i in skipped or not iq_filled(p, i)]
    meta = globals().get("PAGE_META", {})
    def row(label, value, fallback="not answered"):
        has = value not in (None, "", [])
        return [label, value if has else fallback, "answered" if has else "open / assumed"]
    yn = lambda x: None if x is None else ("yes" if x else "no")

    S("Summary")
    P(f"**Project:** {title}  |  **Date:** {datetime.date.today():%d %B %Y}  |  **Status:** " + ("complete" if not open_gen else "contains assumptions (see the Assumptions section)"))
    P(_product_line(st) + ".")
    if st.get("idea"): P("**Your idea, in your words:** " + st["idea"])
    P("This report turns your answers into a structured UX analysis: who the product is for, which pages and features it needs, what each page must contain, UX guidance with sources, accessibility checks and a visual design direction.")

    S("Your requirements")
    T(["Topic", "Answer", "Status"], [
        row("Product type", d["label"]), row("Product name", st.get("brand_name"), f"not answered (working title: {title})"), row("Main goal", st.get("goal")),
        row("Target users", st.get("target_users"), "not answered (assumed general users)"), row("User problems", st.get("pain_points")),
        row("Platform", st["platform"], "not answered (assumed both)"), row("Login needed", yn(st["auth_needed"]), f"not answered (assumed {'yes' if eff_auth else 'no'})"),
        row("Sign-in methods", st.get("auth_methods"), "not applicable" if st["auth_needed"] is False else "not answered"),
        row("Payments needed", yn(st["payment_needed"]), f"not answered (assumed {'yes' if eff_pay else 'no'})"),
        row("Payment methods", st.get("payment_methods"), "not applicable" if st["payment_needed"] is False else "not answered"),
        ["Key features", ", ".join(chosen) + (" (recommended set, assumed)" if feats_assumed else ""), "open / assumed" if feats_assumed else "answered"],
        row("Visual style", style if st["style"] else None, "modern (assumed)"), row("Brand colors", st.get("brand_colors")),
        row("Accessibility needs", st.get("accessibility_notes")), row("Languages", st.get("languages"), "not answered (assumed English)"),
        row("Inspiration and competitors", st.get("inspiration")), row("Constraints", st.get("constraints")), row("Success measure", st.get("success_metric"))])

    S("Assumptions and open questions")
    A = []
    if st["platform"] is None: A.append("Platform: assumed **both** (a responsive web app that works on phones and desktops).")
    if not st.get("target_users"): A.append("Target users: assumed general users of this kind of product.")
    if st["auth_needed"] is None: A.append("Login: assumed **yes**, because accounts are normally needed for orders, records or progress.")
    if st["payment_needed"] is None: A.append("Payments: assumed **yes**; confirm whether payments are really in scope.")
    if st["style"] is None: A.append("Visual style: assumed **modern**.")
    if feats_assumed: A.append("Features: a core set of five was recommended; confirm which ones matter most.")
    if not st.get("goal"): A.append("Goal: not provided; write a one-sentence value proposition before design starts.")
    if not st.get("languages"): A.append("Languages: English assumed.")
    if A: UL(A)
    else: P("No assumptions were needed for the core decisions.")
    if open_gen: P("**Questions skipped or left open:** " + ", ".join(open_gen) + ".")
    P("**Questions to confirm before design starts**")
    UL(["Who exactly are your first 10 users, and can you talk to them?", "Which features are must-have for the first release and which can wait?"] + [f"Risk to check: {r}" for r in DOMAIN_RISKS[st["domain"]][:1]])

    S("Users, problems and core journey")
    P(f"**Target users:** {st.get('target_users') or 'general users (assumed)'}.  **Goal:** {st.get('goal') or 'not provided'}.")
    if st.get("pain_points") and st["pain_points"] != "none": P("**Problems to solve:** " + st["pain_points"])
    if st.get("inspiration") and st["inspiration"] != "none": P("**Inspiration and competitors to study:** " + st["inspiration"])
    P("**Core user journey** (design every step so it takes as few taps as possible):")
    OL(d["journey"])

    S("Pages (information architecture)")
    T(["Page", "Purpose", "Key components"], [[pg, (meta.get(pg) or (None, "Supports: " + ", ".join(components_for(pg)[:2])))[1], "; ".join(components_for(pg))] for pg in pages])

    S("Page-by-page requirements")
    P("Your answers for each page. Items marked **open** were skipped or not asked yet.")
    for pg in pages:
        H(2, pg)
        P("**Recommended components:** " + "; ".join(components_for(pg)))
        T(["Question", "Your answer"], [[re.sub(r"\*\*", "", q), pa.get(pg, {}).get(k) or "open (not answered)"] for k, q, _ in page_questions(pg)])

    S("Features")
    T(["Feature", "What it means", "Priority"], [[f, catalog.get(f, ""), "Must have" + (" (assumed)" if feats_assumed else " (you selected it)")] for f in chosen])
    if extras:
        P("**Recommended additions to consider after the first release:**"); UL([f"{f}: {catalog[f]}" for f in extras])

    S("UX guidance with sources")
    P("Each guideline comes from the knowledge base (usability heuristics, WCAG, platform and domain patterns). The last column shows the pages it matters for.")
    agg = {}
    for pg in pages:
        for g, _ in retrieve_for_page(p, pg, k=3): agg.setdefault(g["id"], (g, []))[1].append(pg)
    ranked = sorted(agg.values(), key=lambda x: (-len(x[1]), x[0]["title"]))[:14]
    T(["Guideline", "What to do", "Source", "Applies to"], [[g["title"], g["text"], g["source"], ", ".join(pgs[:4]) + (f" +{len(pgs) - 4}" if len(pgs) > 4 else "")] for g, pgs in ranked])

    S("Accessibility and inclusive design")
    if st.get("accessibility") or (st.get("accessibility_notes") not in (None, "", "none")):
        P("You mentioned accessibility needs" + (f" ({st['accessibility_notes']})" if st.get("accessibility_notes") not in (None, "", "none") else "") + ": treat **WCAG 2.2 level AA** as a requirement, not a goal.")
    else: P("Aim for **WCAG 2.2 level AA**. It is cheaper to design for accessibility from the start than to fix it later.")
    always = ["h_status", "a_contrast", "a_keyboard", "a_labels", "a_color"] + (["a_targets"] if plat in ("mobile", "both") else [])
    UL([f"{KB_BY_ID[i]['title']}: {KB_BY_ID[i]['text']} ({KB_BY_ID[i]['source']})" for i in always])
    P("**Checks to run before launch:** complete the core journey with the keyboard only; test with a screen reader; zoom text to 200 percent; check every color pair in the next section.")

    S("Visual design direction")
    P(f"**Style:** {style}. {STYLES[style]['notes']}  **Fonts:** {STYLES[style]['fonts']}.")
    UL([f"{k.capitalize()}: {v}" for k, v in pal.items()])
    onp, ona = _best_text(pal["primary"]), _best_text(pal["accent"])
    checks = [("Body text on background", pal["text"], pal["bg"], 4.5), ("Body text on cards", pal["text"], pal["surface"], 4.5), ("Button text on primary", onp, pal["primary"], 4.5),
              ("Text on accent", ona, pal["accent"], 4.5), ("Primary color as a control or icon on background", pal["primary"], pal["bg"], 3.0)]
    T(["Color pair", "Colors", "Contrast", "Needed (WCAG AA)", "Result"], [[n_, f"{a} on {b}", f"{_contrast(a, b):.1f}:1", f"{need}:1", "Pass" if _contrast(a, b) >= need else "Check"] for n_, a, b, need in checks])

    S("Platform considerations"); UL(PLATFORM_NOTES[plat])
    ents = globals().get("DOMAIN_ENTITIES", {}).get(st["domain"])
    if ents:
        S("Data to plan for"); P("The information your product will store (a starting point for the database design):"); UL(ents)
    S("Risks and domain priorities"); UL(DOMAIN_RISKS[st["domain"]]); P("**UX priorities for this kind of product:**"); UL(d["tips"])
    S("Next steps")
    OL(["Review this report and confirm the assumptions.", "Talk to 3 to 5 potential users and check that the core journey matches how they work.", "Sketch wireframes for the pages, using the page-by-page answers.",
        "Build a clickable prototype and test the core journey with real people.", "Implement in phases: setup, design system, data, pages, quality checks, deployment.", "Re-check accessibility before launch."])

    H(1, "Appendix A. Conversation record"); rec = []
    for u, b in p.histories.get("requirements", []):
        if u: rec.append("**You:** " + u)
        rec.append("**Assistant:** " + re.sub(r"\s+", " ", re.sub(r"[*`]", "", b))[:160])
    UL(rec)
    H(1, "Appendix B. How this report was produced")
    P("The report was produced with rules and retrieval, without a language model: text understanding with spaCy and sentence embeddings, a zero-shot product-type classifier, a question flow that asks only for missing information (general questions, then questions for each page), and a UX guideline base (Nielsen heuristics, WCAG, platform and domain patterns) with a source for every guideline. Treat it as a well-grounded starting point and have a designer review it.")
    return title, B

def _cell(x): return str(x).replace("|", "/").replace("\n", " ")
def blocks_to_md(title, blocks):
    out = [f"# {title}", "", "*Detailed UX report*", ""]
    for b in blocks:
        k = b[0]
        if k == "h": out += ["#" * (b[1] + 1) + " " + b[2], ""]
        elif k == "p": out += [b[1], ""]
        elif k == "ul": out += [f"- {t}" for t in b[1]] + [""]
        elif k == "ol": out += [f"{i}. {t}" for i, t in enumerate(b[1], 1)] + [""]
        elif k == "table":
            out += ["| " + " | ".join(map(_cell, b[1])) + " |", "|" + "|".join("---" for _ in b[1]) + "|"] + ["| " + " | ".join(map(_cell, r)) + " |" for r in b[2]] + [""]
    return "\n".join(out)

def _runs(par, text):
    for i, seg in enumerate(text.split("**")):
        if seg: par.add_run(seg).bold = (i % 2 == 1)
def blocks_to_docx(title, blocks, path):
    try: from docx import Document
    except ImportError:
        import subprocess; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-docx"]); from docx import Document
    from docx.shared import Pt
    doc = Document(); doc.styles["Normal"].font.name = "Calibri"; doc.styles["Normal"].font.size = Pt(11)
    doc.add_heading(title, 0); doc.add_paragraph("Detailed UX report")
    for b in blocks:
        k = b[0]
        if k == "h": doc.add_heading(b[2], b[1])
        elif k == "p": _runs(doc.add_paragraph(), b[1])
        elif k == "ul":
            for t in b[1]: _runs(doc.add_paragraph(style="List Bullet"), t)
        elif k == "ol":
            for t in b[1]: _runs(doc.add_paragraph(style="List Number"), t)
        elif k == "table":
            t = doc.add_table(rows=1, cols=len(b[1])); t.style = "Light Grid Accent 1"
            for j, c in enumerate(b[1]): t.rows[0].cells[j].text = str(c)
            for r in b[2]:
                cells = t.add_row().cells
                for j, v in enumerate(r): cells[j].text = str(v)
            doc.add_paragraph("")
    doc.save(path)

def make_report_files(p):
    title, blocks = build_detailed_report(p); md = blocks_to_md(title, blocks)
    base = re.sub(r"[^A-Za-z0-9]+", "_", title).strip("_")[:40] or "report"
    md_path = os.path.join(RP_TMP, f"{base}_report.md"); open(md_path, "w", encoding="utf-8").write(md)
    docx_path = os.path.join(RP_TMP, f"{base}_report.docx"); blocks_to_docx(title, blocks, docx_path)
    return md, docx_path, md_path
print("Report builder ready")

Report builder ready


In [44]:
#46
import glob, traceback
RP_PUBLIC = os.environ.get("PUBLIC_MODE", "0") == "1"
RP_INTRO = ("Hi! Tell me about the website or app you want to build, in your own words. I'll ask some **general questions** first, then a few questions about **each page**, "
            "and then write a **detailed report** you can download. I skip anything you already told me.\n\nYou can **pause at any time** with the Pause button and continue later.")

def rp_new_project():
    p = Project(); p.state.setdefault("asked", []); p.histories["requirements"] = [["", RP_INTRO]]; return p
def _rp(p): return p if isinstance(p, Project) else rp_new_project()

def rp_chat(p, text):
    hist = p.histories.setdefault("requirements", [])
    if not hist: hist.append(["", RP_INTRO])
    try: reply = handle_interview(p, text)
    except Exception as e:
        traceback.print_exc()
        reply = (f"⚠️ Something went wrong inside the app: `{type(e).__name__}: {e}`.\n\nYour earlier answers are safe. Please copy this message and send it to me so I can fix it.")
    hist.append([text, reply]); p.stage = "requirements"; p.update_name()
    if not RP_PUBLIC:
        try: p.save()
        except Exception: pass
    return reply

def rp_progress(p):
    gd, gt, pd_, pt = iq_counts(p); done, total = gd + pd_, gt + pt
    filled = int(10 * done / max(total, 1)); bar = "█" * filled + "░" * (10 - filled)
    qid = p.pending or iq_next(p)
    if p.stage_done.get("requirements") and not p.pending: status = "✅ All questions answered. Your report is in the Report tab."
    elif qid: status = f"Next: {iq_label(p, qid)}"
    else: status = "Tell me about your idea"
    return f"**Progress:** {bar}  general {gd}/{gt}" + (f"  ·  pages {pd_}/{pt}" if pt else "  ·  page questions come after the general ones") + f"  ·  {status}"

_RP_MSG = True
def rp_make_chatbot():
    if int(gr.__version__.split(".")[0]) >= 6: return gr.Chatbot(label="Interview", height=470), True
    try: return gr.Chatbot(label="Interview", height=470, type="messages"), True
    except Exception: return gr.Chatbot(label="Interview", height=470), False
def rp_display(hist):
    if _RP_MSG:
        out = []
        for u, b in hist:
            if u: out.append({"role": "user", "content": u})
            out.append({"role": "assistant", "content": b})
        return out
    return [[u or None, b] for u, b in hist]

def rp_state_ui(p):
    labels = iq_quick(p)
    btns = [gr.Button(value=labels[i], visible=True) if i < len(labels) else gr.Button(visible=False) for i in range(7)]
    return (rp_display(p.histories["requirements"]), rp_progress(p), *btns)

def rp_send(msg, p):
    p = _rp(p); t = (msg or "").strip()
    if t: rp_chat(p, t)
    return ("", p, *rp_state_ui(p))
def rp_new():
    p = rp_new_project(); return (p, *rp_state_ui(p))

def rp_report(p):
    if not isinstance(p, Project) or not p.state.get("domain"):
        return "_Answer at least the first question (the product type) in the Interview tab, then come back._", None, None
    try:
        md, docx_path, md_path = make_report_files(p); return md, docx_path, md_path
    except Exception as e:
        traceback.print_exc(); return f"⚠️ The report could not be created: `{type(e).__name__}: {e}`. Please send me this message.", None, None

# ---- pause and resume ----
def rp_pause(p):
    if not isinstance(p, Project) or len(p.histories.get("requirements", [])) < 2: return None, "Nothing to save yet. Answer at least one question first."
    path = os.path.join(RP_TMP, f"{re.sub(r'[^A-Za-z0-9]+', '_', p.name)[:30]}_{p.id}.json")
    p.save(path=path)
    if not RP_PUBLIC:
        try: p.save()
        except Exception: pass
    qid = p.pending or iq_next(p)
    where = "All questions are answered; your report is ready." if (p.stage_done.get("requirements") and not p.pending) or qid is None else f"You stopped at the question about **{iq_label(p, qid)}**."
    return path, ("⏸ **Saved.** " + where + " Download the file below to keep it. To continue later, open the app, go to the **Resume** tab and upload the file"
                  + ("." if RP_PUBLIC else " (or pick it from the list if this notebook is still running)."))

def rp_list_sessions():
    if RP_PUBLIC: return []
    out = []
    for f in sorted(glob.glob(f"{SAVE_DIR}/*.json"), key=os.path.getmtime, reverse=True):
        try:
            d = json.load(open(f)); st = d["state"]
            if not st.get("idea") and not st.get("domain"): continue
            q = Project.from_dict(d); gd, gt, pd_, pt = iq_counts(q)
            flag = "report ready" if d.get("stage_done", {}).get("requirements") else f"{gd}/{gt} general" + (f", {pd_}/{pt} page" if pt else "") + " answered"
            out.append((f"{d['name']} · {flag} · saved {datetime.datetime.fromtimestamp(os.path.getmtime(f)):%d %b %H:%M}", f))
        except Exception: pass
    return out

def rp_load(path, p):
    try: q = Project.load(path)
    except Exception as e: return p, f"❌ Could not open that file: {e}"
    q.stage = "requirements"; q.state.setdefault("asked", []); q.state.setdefault("page_answers", {}); hist = q.histories.setdefault("requirements", [])
    qid = iq_next(q)
    if q.stage_done.get("requirements") or qid is None:
        hist.append(["", "👋 **Welcome back!** All your answers are here. Open the **Report** tab, or type `change style` (or any other answer, or a page name) to edit."])
    else:
        q.pending = qid; hist.append(["", "👋 **Welcome back!** Let's continue where we stopped.\n\n" + iq_question(q, qid)])
    return q, "✅ Session restored. Go to the **Interview** tab to continue."

def rp_resume(path, p):
    p0 = _rp(p)
    if not path: return (p0, "Choose a session first.", *rp_state_ui(p0))
    q, msg = rp_load(path, p0); return (q, msg, *rp_state_ui(q))
def rp_upload(f, p): return rp_resume(f if isinstance(f, str) else getattr(f, "name", None), p)

def build_report_app():
    global _RP_MSG
    try: blocks = gr.Blocks(title="UX Requirements Interview", theme=gr.themes.Soft(primary_hue="indigo"))
    except Exception: blocks = gr.Blocks(title="UX Requirements Interview")
    with blocks as demo:
        gr.Markdown("# UX Requirements Interview\nAnswer general questions and then questions for each page, and download a **detailed report**. You can pause at any time and continue later.")
        proj = gr.State(None); progress = gr.Markdown()
        with gr.Tab("Interview"):
            chatbot, _RP_MSG = rp_make_chatbot()
            with gr.Row(): btns = [gr.Button("", visible=False) for _ in range(7)]
            box = gr.Textbox(label="Your answer", placeholder="Type your answer and press Enter")
            with gr.Row():
                send = gr.Button("Send", variant="primary"); pause = gr.Button("⏸ Pause and save"); new_btn = gr.Button("New session")
            pause_msg = gr.Markdown(); pause_file = gr.File(label="Your saved session (download it to continue later)")
            gr.Examples(examples=[["I want to build a food delivery app for college students"], ["Website for patients to book doctor appointments"],
                                  ["Online store for handmade jewelry with a wishlist and coupons"]], inputs=box, label="Try an example")
        with gr.Tab("Report"):
            gr.Markdown("Your detailed report. Press the button after answering (you can also type `report` in the chat at any time).")
            gen = gr.Button("Generate / refresh report", variant="primary")
            with gr.Row(): docx_file = gr.File(label="Download as Word (.docx)"); md_file = gr.File(label="Download as Markdown (.md)")
            report_md = gr.Markdown()
        with gr.Tab("Resume"):
            gr.Markdown("**Continue an earlier session.**" + (" Upload the session file you saved with *Pause and save*." if RP_PUBLIC else " Pick a session from the list, or upload a saved session file."))
            resume_msg = gr.Markdown(); saved = None
            if not RP_PUBLIC:
                saved = gr.Dropdown(choices=rp_list_sessions(), label="Saved sessions")
                with gr.Row(): refresh_btn = gr.Button("Refresh list"); resume_btn = gr.Button("Resume selected", variant="primary")
            up = gr.File(label="Upload a saved session (.json)", file_types=[".json"])
        outs = [box, proj, chatbot, progress] + btns
        send.click(rp_send, [box, proj], outs); box.submit(rp_send, [box, proj], outs)
        for b in btns: b.click(rp_send, [b, proj], outs)
        new_outs = [proj, chatbot, progress] + btns
        new_btn.click(rp_new, None, new_outs); demo.load(rp_new, None, new_outs)
        pause.click(rp_pause, proj, [pause_file, pause_msg])
        gen.click(rp_report, proj, [report_md, docx_file, md_file])
        res_outs = [proj, resume_msg, chatbot, progress] + btns
        up.upload(rp_upload, [up, proj], res_outs)
        if saved is not None:
            refresh_btn.click(lambda: gr.Dropdown(choices=rp_list_sessions()), None, saved)
            resume_btn.click(rp_resume, [saved, proj], res_outs)
    return demo
print("App ready. Run Cell 47 (tests), then Cell 48 (launch).")

App ready. Run Cell 47 (tests), then Cell 48 (launch).


In [45]:
ANS = {"domain": "food ordering", "brand_name": "CampusBite", "goal": "Help students order cheap meals from campus canteens without queues",
       "target_users": "college students", "pain_points": "long queues and no idea what is available", "platform": "mobile", "features": "1, 4, 5",
       "auth_needed": "yes", "auth_methods": "phone and OTP, Google sign-in", "payment_needed": "yes", "payment_methods": "UPI, cards", "style": "playful",
       "brand_colors": "orange #FF7A00", "accessibility_notes": "large text", "languages": "English and Hindi", "inspiration": "Zomato, Swiggy",
       "constraints": "none", "success": "500 weekly orders"}

tp = rp_new_project(); rp_chat(tp, "I want to build a food delivery app for college students"); used = 0
for _ in range(120):
    qid = tp.pending
    if qid is None: break
    if qid == "pages_confirm": ans = "remove profile" if used == 0 else "ok"; used += 1
    elif qid.startswith("pg::"): ans = "skip page" if "::Cart::" in qid else "Short example answer for " + qid.split("::")[2]
    else: ans = ANS.get(qid, "skip")
    rp_chat(tp, ans)
st = tp.state; gd, gt, pd_, pt = iq_counts(tp)
assert tp.stage_done["requirements"], "the interview did not finish: " + rp_progress(tp)
assert "Profile" not in tp.pages and "Checkout" in tp.pages
assert st["brand_name"] == "CampusBite" and st["platform"] == "mobile" and st["style"] == "playful"
assert st.get("brand_hex") == "#FF7A00", st.get("brand_colors")
assert len(st["features"]) >= 3, st["features"]
assert "details" in st["page_answers"]["Checkout"] and "pg::Cart::edit" in st["skipped"]
print(f"Interview OK: {gd}/{gt} general answers, {pd_}/{pt} page answers")

md, dpath, mpath = make_report_files(tp)
for sec in ("Summary", "Your requirements", "Page-by-page requirements", "Checkout", "UX guidance with sources", "Visual design direction", "Next steps", "Appendix A"):
    assert sec in md, f"missing section: {sec}"
assert "CampusBite" in md and "#FF7A00" in md and os.path.getsize(dpath) > 10000
print("Report OK:", len(md.split()), "words ->", dpath)

t2 = rp_new_project(); rp_chat(t2, "I want to build a food delivery app for college students")
path = os.path.join(RP_TMP, "pause_test.json"); t2.save(path=path)
q, msg = rp_load(path, t2)
assert q.pending == iq_next(q) and "Welcome back" in q.histories["requirements"][-1][1] and q.state["domain"] == t2.state["domain"]
rp_chat(tp, "change checkout"); assert tp.pending.startswith("pg::Checkout")
rp_chat(tp, "change platform"); assert tp.pending == "platform" and tp.state["platform"] is None
print("Pause, resume and change OK")

Interview OK: 19/19 general answers, 17/17 page answers
Report OK: 3422 words -> /tmp/uxreport_otoajn0c/CampusBite_report.docx
Pause, resume and change OK


In [46]:
#48
try: rp_demo.close()
except Exception: pass
rp_demo = build_report_app()
rp_demo.launch(share=True)

Closing server running on port: 7860


/tmp/ipykernel_1332/158335395.py:112: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  try: blocks = gr.Blocks(title="UX Requirements Interview", theme=gr.themes.Soft(primary_hue="indigo"))


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://601550caf27921e26d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [47]:
#49

In [48]:
for need in ("build_prompt_pack", "master_prompt", "page_prompt", "phase_auth", "phase_payments", "check_prompt_pack", "pack_index_md", "STACKS", "get_pack"):
    assert need in globals(), f"Run Cells 38 and 39 first (prompt knowledge and prompt generator): '{need}' is missing."
for need in ("handle_interview", "make_report_files", "build_report_app", "rp_report", "rp_chat", "DOMAIN_RISKS"):
    assert need in globals(), f"Run the updated Cells 44, 45 and 46 first: '{need}' is missing."
import zipfile

# ---- 1. the brand color from the interview drives the design tokens everywhere ----
if "_orig_get_palette" not in globals(): _orig_get_palette = get_palette
def get_palette(st):
    pal = _orig_get_palette(st)
    return dict(pal, primary=st["brand_hex"]) if st.get("brand_hex") else pal

# ---- 2. assumptions and the one-screen summary ----
def _assumptions(st):
    A = []
    if st.get("platform") is None: A.append("platform: both (responsive web app)")
    if not st.get("target_users"): A.append("target users: general users")
    if st.get("auth_needed") is None: A.append("login: yes")
    if st.get("payment_needed") is None: A.append("payments: yes")
    if not st.get("style"): A.append("visual style: modern")
    if not st.get("features"): A.append("features: recommended core set of five")
    if not st.get("goal"): A.append("goal: not provided")
    return A

def build_summary_md(p):
    st = p.state; d = DOMAINS[st["domain"]]; title = project_title(p); gd, gt, pd_, pt = iq_counts(p)
    pages = list(p.pages) or _default_pages(st); feats = list(st.get("features", [])) or list(d["features"])[:5]
    pa = st.get("page_answers", {}); style = st.get("style") or "modern"; pal = rp_palette(st)
    yn = lambda x: "not decided" if x is None else ("yes" if x else "no")
    extra = lambda k: f" ({st[k]})" if st.get(k) not in (None, "", "none") else ""
    L = [f"# Project summary: {title}", "", f"**{_product_line(st)}.**", "", "## Key decisions",
         f"- **Goal:** {st.get('goal') or 'not provided'}", f"- **Users:** {st.get('target_users') or 'general users (assumed)'}", f"- **Platform:** {st.get('platform') or 'both (assumed)'}",
         f"- **Login:** {yn(st.get('auth_needed'))}{extra('auth_methods')}", f"- **Payments:** {yn(st.get('payment_needed'))}{extra('payment_methods')}",
         f"- **Style:** {style}, primary color {pal['primary']}", f"- **Languages:** {st.get('languages') or 'English (assumed)'}", "",
         "## Scope", f"- **{len(pages)} pages:** " + ", ".join(pages), f"- **{len(feats)} features:** " + ", ".join(feats)]
    if pt:
        L += ["", "## Page questions coverage"] + [f"- {pg}: {sum(1 for k, _, _ in page_questions(pg) if k in pa.get(pg, {}))}/{len(page_questions(pg))} answered" for pg in pages]
    L += ["", "## Top UX priorities"] + [f"- {t}" for t in d["tips"][:3]] + ["", "## Watch out for"] + [f"- {r}" for r in DOMAIN_RISKS[st["domain"]][:2]]
    A = _assumptions(st)
    if A: L += ["", "## Assumptions to confirm"] + [f"- {a}" for a in A]
    L += ["", f"**Interview coverage:** general {gd}/{gt}" + (f", pages {pd_}/{pt}" if pt else ""),
          "", "**Next step:** read the detailed report, confirm the assumptions, then give the development prompts to your builder one step at a time."]
    return "\n".join(L)

# ---- 3. interview answers flow into the development prompts ----
def _inject(text, marker, extra):
    if not extra: return text
    return text.replace(marker, extra + marker, 1) if marker in text else text + "\n\n" + extra

def interview_brief(p):
    st = p.state; rows = []
    def add(label, v):
        if v not in (None, "", [], "none"): rows.append(f"- {label}: {v}")
    add("Product name", st.get("brand_name")); add("Main goal", st.get("goal")); add("User problems to solve", st.get("pain_points"))
    if st.get("auth_needed"): add("Sign-in methods", st.get("auth_methods"))
    if st.get("payment_needed"): add("Payment methods", st.get("payment_methods"))
    add("Brand colors", st.get("brand_colors")); add("Accessibility needs", st.get("accessibility_notes")); add("Languages", st.get("languages"))
    add("Inspiration and competitors", st.get("inspiration")); add("Constraints", st.get("constraints")); add("Success measure", st.get("success_metric"))
    if st.get("prompt_assumptions"): rows.append("- Assumptions (confirm with the product owner): " + "; ".join(st["prompt_assumptions"]))
    return ("## 7b. Requirements gathered in the interview\n" + "\n".join(rows) + "\n\n") if rows else ""

if "_orig_master_prompt" not in globals(): _orig_master_prompt = master_prompt
def master_prompt(p, stack):
    return _inject(_orig_master_prompt(p, stack), "## 8. How you (the AI assistant) should work", interview_brief(p))

if "_orig_page_prompt" not in globals(): _orig_page_prompt = page_prompt
def page_prompt(p, page, stack, step_id):
    text = _orig_page_prompt(p, page, stack, step_id)
    pa = p.state.get("page_answers", {}).get(page, {}); qs = page_questions(page)
    items = [(k, q, pa[k]) for k, q, _ in qs if k in pa]; open_keys = [k for k, _, _ in qs if k not in pa]
    if items:
        sec = "\n\n## Your requirements for this page (from the interview)\n" + "\n".join(f"- {re.sub(r'[*]+', '', q)} **Answer:** {a}" for _, q, a in items)
        if open_keys: sec += "\n- Not answered yet (decide before building): " + ", ".join(open_keys)
        crit = "\n".join(f"- [ ] Requirement ({k}): {a}" for k, _, a in items)
        text = text.replace("\n\n## Acceptance criteria", sec + "\n\n## Acceptance criteria", 1) if "\n\n## Acceptance criteria" in text else text + sec
        text = text.replace("\n\n## Tests to write", "\n" + crit + "\n\n## Tests to write", 1) if "\n\n## Tests to write" in text else text + "\n" + crit
    return text

if "_orig_phase_auth" not in globals(): _orig_phase_auth = phase_auth
def phase_auth(p, stack):
    m = p.state.get("auth_methods"); extra = f"## Requirements from the interview\n- Sign-in methods to support: {m}\n\n" if m not in (None, "", "none") else ""
    return _inject(_orig_phase_auth(p, stack), "\n\n## Constraints", extra and "\n\n" + extra.rstrip())
if "_orig_phase_payments" not in globals(): _orig_phase_payments = phase_payments
def phase_payments(p, stack):
    m = p.state.get("payment_methods"); extra = f"## Requirements from the interview\n- Payment methods to support: {m}\n\n" if m not in (None, "", "none") else ""
    return _inject(_orig_phase_payments(p, stack), "\n\n## Constraints", extra and "\n\n" + extra.rstrip())

def sync_for_prompts(p):
    st = p.state
    if not p.pages: p.pages = _default_pages(st)
    p.features = list(st.get("features") or list(DOMAINS[st["domain"]]["features"])[:5])

def rp_build_pack(p, with_checks=False):
    """Builds the prompts. Unanswered login, payments and platform are treated as assumed (same as in the report)."""
    st = p.state; saved = (st.get("auth_needed"), st.get("payment_needed"), st.get("platform")); assumed = _assumptions(st)
    try:
        if st.get("auth_needed") is None: st["auth_needed"] = True
        if st.get("payment_needed") is None: st["payment_needed"] = True
        if st.get("platform") is None: st["platform"] = "both"
        st["prompt_assumptions"] = assumed; sync_for_prompts(p)
        pack = get_pack(p); checks = check_prompt_pack(p, pack) if with_checks else None
    finally:
        st["auth_needed"], st["payment_needed"], st["platform"] = saved; st.pop("prompt_assumptions", None)
    return (pack, checks) if with_checks else pack

def export_rp_prompts(p, out_dir=None):
    pack = rp_build_pack(p); d = out_dir or tempfile.mkdtemp(prefix="uxprompts_"); os.makedirs(d, exist_ok=True)
    for it in pack: open(os.path.join(d, it["file"]), "w", encoding="utf-8").write(it["text"])
    open(os.path.join(d, "README.md"), "w", encoding="utf-8").write(pack_index_md(p, pack))
    open(os.path.join(d, "all_prompts_combined.md"), "w", encoding="utf-8").write("\n\n---\n\n".join(it["text"] for it in pack))
    zpath = d.rstrip("/\\") + ".zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(os.listdir(d)): z.write(os.path.join(d, f), f)
    return zpath, pack

def _rp_pack_summary(p, pack):
    L = [f"Generated **{len(pack)} prompts** for **{STACKS[resolve_stack(p)]['label']}**:", ""] + [f"- **{it['id']}** {it['title']} (about {it['tokens']} tokens)" for it in pack]
    return "\n".join(L + ["", "**How to use:** give your coding assistant step 00 first (or save it as CLAUDE.md or AGENTS.md), then one step at a time, in order. Check each step's acceptance criteria before moving on.",
                          "", "Commands: `show 00`, `show checkout`, `stack flutter`. The **Build prompts** tab has copy buttons and a ZIP download."])

# ---- 4. chat commands, quick replies, report with summary ----
RP_STACK_WORDS = {"react native": "react-native", "react-native": "react-native", "expo": "react-native", "nextjs": "nextjs", "next.js": "nextjs", "next": "nextjs",
                  "flutter": "flutter", "dart": "flutter", "vanilla": "vanilla", "html": "vanilla", "plain": "vanilla", "vite": "react", "react": "react"}
def rp_find_stack(text):
    for k in RP_STACK_WORDS:
        if re.search(r"(?<![a-z])" + re.escape(k) + r"(?![a-z])", text.lower()): return RP_STACK_WORDS[k]
    return None

def rp_prompt_command(p, text):
    low = text.strip().lower(); st = p.state
    if not st.get("domain"): return "Answer the first question (the product type) first; then I can write the summary and the prompts."
    if re.match(r"^summary\s*$", low): return build_summary_md(p)
    m = re.match(r"^stack\s+(.+)", low)
    if m:
        s = rp_find_stack(m.group(1))
        if not s: return "Available stacks: `react`, `nextjs`, `react-native`, `flutter`, `vanilla`."
        p.build_stack = s; return f"Tech stack set to **{STACKS[s]['label']}**. Type `prompts` to rebuild the prompts."
    if re.match(r"^(prompts?|build prompts?|dev(elopment)? prompts?)\s*$", low): return _rp_pack_summary(p, rp_build_pack(p))
    m = re.match(r"^show\s+(.+)", low)
    if m:
        pack = rp_build_pack(p); arg = m.group(1).strip()
        item = next((it for it in pack if it["id"] == arg.zfill(2)), None) if arg.isdigit() else None
        if item is None and "master" in arg: item = pack[0]
        if item is None:
            name = match_name(arg, [it["title"] for it in pack], 0.45); item = next((it for it in pack if it["title"] == name), None)
        return f"**Step {item['id']}: {item['title']}**\n\n```\n{item['text']}\n```" if item else "I couldn't find that step. Type `prompts` to see the list."
    return None

if "_orig_handle_interview" not in globals(): _orig_handle_interview = handle_interview
def handle_interview(p, text):
    done = p.stage_done.get("requirements") and not p.pending; low = text.strip().lower()
    if re.match(r"^(summary|prompts?|build prompts?|dev(elopment)? prompts?)\s*$", low) or (done and re.match(r"^(show|stack)\s+\S", low)):
        r = rp_prompt_command(p, text)
        if r: return r
    return _orig_handle_interview(p, text)

if "_orig_iq_finish" not in globals(): _orig_iq_finish = _iq_finish
def _iq_finish(p, force=False):
    msg = _orig_iq_finish(p, force)
    if p.state.get("domain") and p.stage_done.get("requirements"):
        msg += "\n\nAlso ready: a **project summary** (type `summary`) and **development prompts** for building this product (type `prompts`, or open the **Build prompts** tab)."
    return msg

if "_orig_iq_quick" not in globals(): _orig_iq_quick = iq_quick
def iq_quick(p):
    if p.stage_done.get("requirements") and not p.pending: return ["report", "summary", "prompts", "status", "change style"]
    return _orig_iq_quick(p)

if "_orig_rp_report" not in globals(): _orig_rp_report = rp_report
def rp_report(p):
    md, dx, mp = _orig_rp_report(p)
    if isinstance(p, Project) and p.state.get("domain") and dx: md = build_summary_md(p) + "\n\n---\n\n" + md
    return md, dx, mp

# ---- 5. a "Build prompts" tab in the app ----
if "_orig_build_report_app" not in globals(): _orig_build_report_app = build_report_app
def build_report_app():
    demo = _orig_build_report_app()
    try:
        proj = next(b for b in demo.blocks.values() if isinstance(b, gr.State))
        def _gen(p, stack):
            if not isinstance(p, Project) or not p.state.get("domain"):
                return p, gr.Dropdown(choices=[], value=None), "Answer the first question (the product type) in the Interview tab first.", "", None
            p.build_stack = stack or "auto"
            try: zpath, pack = export_rp_prompts(p)
            except Exception as e:
                traceback.print_exc(); return p, gr.Dropdown(choices=[], value=None), f"⚠️ Could not create the prompts: `{type(e).__name__}: {e}`. Please send me this message.", "", None
            return p, gr.Dropdown(choices=[(f"{it['id']} · {it['title']}", it["id"]) for it in pack], value=pack[0]["id"]), _rp_pack_summary(p, pack), pack[0]["text"], zpath
        def _show(p, sel):
            if not isinstance(p, Project) or not p.state.get("domain") or not sel: return ""
            return next((it["text"] for it in rp_build_pack(p) if it["id"] == sel), "")
        with demo:
            with gr.Tab("Build prompts"):
                gr.Markdown("Ready-to-paste **development prompts** for a coding assistant: a master brief, setup, design system, data layer, one prompt per page (with your answers), QA and deployment.")
                with gr.Row():
                    stack_dd = gr.Dropdown(choices=[("Auto (from platform)", "auto")] + [(v["label"], k) for k, v in STACKS.items()], value="auto", label="Tech stack")
                    gen_btn = gr.Button("Generate development prompts", variant="primary")
                info = gr.Markdown(); sel = gr.Dropdown(choices=[], label="Choose a prompt to view and copy")
                try: view = gr.Code(language="markdown", label="Prompt", lines=22)
                except Exception: view = gr.Textbox(label="Prompt", lines=22)
                zipf = gr.File(label="Download all prompts (.zip)")
            gen_btn.click(_gen, [proj, stack_dd], [proj, sel, info, view, zipf]); sel.change(_show, [proj, sel], view)
    except Exception as e:
        print("Note: could not add the 'Build prompts' tab (the chat commands still work):", e)
    return demo
print("Summary and prompts connected. Re-run Cell 48 later to relaunch the app with the new tab.")

Summary and prompts connected. Re-run Cell 48 later to relaunch the app with the new tab.


In [50]:
if "FONT_STACKS" not in globals():
    FONT_STACKS = {   # (headings, body)
     "minimal":      ("'Inter', system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif",) * 2,
     "modern":       ("'Poppins', 'Inter', system-ui, sans-serif", "'Inter', system-ui, -apple-system, 'Segoe UI', sans-serif"),
     "playful":      ("'Nunito', 'Trebuchet MS', system-ui, sans-serif",) * 2,
     "professional": ("'Source Sans 3', 'Segoe UI', Roboto, sans-serif",) * 2,
     "dark":         ("'Inter', system-ui, -apple-system, 'Segoe UI', sans-serif",) * 2,
     "elegant":      ("'Playfair Display', Georgia, 'Times New Roman', serif", "'Lato', 'Segoe UI', sans-serif"),
    }
if "RADIUS" not in globals():
    RADIUS = {"minimal": "8px", "modern": "14px", "playful": "18px", "professional": "6px", "dark": "12px", "elegant": "4px"}
if "readable_on" not in globals():
    def readable_on(hex_color):
        """Pick white or near-black text, whichever has the higher WCAG contrast on this background."""
        h = hex_color.lstrip("#"); r, g, b = [int(h[i:i + 2], 16) / 255 for i in (0, 2, 4)]
        lin = lambda x: x / 12.92 if x <= 0.03928 else ((x + 0.055) / 1.055) ** 2.4
        L = 0.2126 * lin(r) + 0.7152 * lin(g) + 0.0722 * lin(b)
        return "#FFFFFF" if 1.05 / (L + 0.05) >= (L + 0.05) / 0.05 else "#111111"

# check everything else the prompts and the report need
import pandas as pd
needed = {"Cell 3": ["DOMAINS", "STYLES", "ALL_PAGES", "COLOR_HEX"], "Cell 6": ["recommend_pages", "recommend_features", "get_palette", "match_name", "components_for"],
          "Cell 12": ["KB_BY_ID"], "Cell 13": ["retrieve_for_page"], "Cell 38": ["STACKS", "PAGE_META", "DOMAIN_ENTITIES", "FEATURE_NOTES", "FEATURE_PAGES"],
          "Cell 39": ["build_prompt_pack", "get_pack", "pack_index_md", "check_prompt_pack"]}
missing = {c: [n for n in names if n not in globals()] for c, names in needed.items()}
missing = {c: m for c, m in missing.items() if m}
print("Helpers ready. Missing items:", missing if missing else "none")

Helpers ready. Missing items: none


In [51]:
#50
import pandas as pd, shutil
from IPython.display import display, Markdown

# ================= SETTINGS =================
RUN_CASES = ["food", "ecommerce", "education", "healthcare", "banking", "minimal"]   # which test cases to run
SHOW_DETAILS_FOR = ["food"]        # which cases print their full outputs here (all cases are saved to files)
SHOW_TRANSCRIPT, SHOW_SUMMARY, SHOW_REPORT = True, True, True
SHOW_PROMPTS = "master"            # "none", "list", "master" (list + master + one page prompt) or "all"
OUT_DIR = "/content/test_outputs"
# ============================================

TEST_CASES = {
 "food": {"first": "I want to build a food delivery app for college students",
   "answers": {"brand_name": "CampusBite", "goal": "Help students order cheap meals from campus canteens without queues", "pain_points": "long queues and no idea what is available",
               "target_users": "college students", "platform": "mobile", "features": "1, 4, 5", "auth_needed": "yes", "auth_methods": "phone and OTP, Google sign-in",
               "payment_needed": "yes", "payment_methods": "UPI, cards", "style": "playful", "brand_colors": "orange #FF7A00", "accessibility_notes": "large text",
               "languages": "English and Hindi", "inspiration": "Zomato, Swiggy", "constraints": "none", "success": "500 weekly orders", "pages_confirm": ["remove profile", "ok"]},
   "expect": {"domain": "food_ordering", "platform": "mobile", "pages_contains": ["Menu", "Checkout", "Order Tracking"], "pages_absent": ["Profile"],
              "report_contains": ["CampusBite", "#FF7A00", "Checkout"], "master_contains": ["CampusBite", "phone and OTP", "#FF7A00", "UPI, cards"]}},
 "ecommerce": {"first": "Online store for handmade jewelry with a wishlist and coupons",
   "answers": {"brand_name": "GemCraft", "goal": "Let customers discover and buy handmade jewelry with confidence", "pain_points": "hard to trust quality when buying online",
               "target_users": "gift shoppers and jewelry lovers", "platform": "website", "features": "recommended", "auth_needed": "no", "payment_needed": "yes",
               "payment_methods": "cards, UPI, PayPal", "style": "elegant", "brand_colors": "none", "accessibility_notes": "none", "languages": "English",
               "inspiration": "Etsy", "constraints": "launch in 3 months with a small budget", "success": "50 orders a month", "pages_confirm": "ok"},
   "expect": {"domain": "ecommerce", "platform": "web", "pages_contains": ["Product List", "Cart", "Checkout"], "pages_absent": ["Login"],
              "report_contains": ["GemCraft", "Etsy"], "master_contains": ["GemCraft", "cards, UPI, PayPal"]}},
 "education": {"first": "E-learning platform with video courses and quizzes for school kids",
   "answers": {"brand_name": "BrightLearn", "goal": "Make learning fun and let parents track their child's progress", "pain_points": "kids lose interest and parents cannot see progress",
               "target_users": "school kids and their parents", "platform": "both", "features": "2, 3, 4", "auth_needed": "yes", "auth_methods": "email and parent consent",
               "payment_needed": "no", "style": "playful", "brand_colors": "blue", "accessibility_notes": "captions for all videos, dyslexia-friendly fonts", "languages": "English",
               "inspiration": "Khan Academy", "constraints": "must follow child privacy rules", "success": "70 percent of kids finish their first course", "pages_confirm": "ok"},
   "expect": {"domain": "education", "platform": "both", "pages_contains": ["Course Catalog", "Lesson Player", "Quiz"], "report_contains": ["BrightLearn", "captions"],
              "master_contains": ["BrightLearn", "captions for all videos"]}},
 "healthcare": {"first": "Website for patients to book doctor appointments and consult online",
   "answers": {"brand_name": "CareLink", "goal": "Help patients find a doctor and book a visit in minutes", "pain_points": "long phone queues and no visibility of free slots",
               "target_users": "patients and their families", "platform": "website", "features": "recommended", "auth_needed": "yes", "auth_methods": "phone and OTP",
               "payment_needed": "yes", "payment_methods": "cards, insurance", "style": "professional", "brand_colors": "teal", "accessibility_notes": "older users, screen readers",
               "languages": "English and Spanish", "inspiration": "Practo", "constraints": "must protect patient privacy", "success": "reduce missed appointments by 30 percent", "pages_confirm": "ok"},
   "expect": {"domain": "healthcare", "platform": "web", "pages_contains": ["Find Doctor", "Book Appointment", "Telehealth"], "report_contains": ["CareLink", "screen readers"],
              "master_contains": ["CareLink", "older users, screen readers"]}},
 "banking": {"first": "Mobile banking app for young professionals with biometric login",
   "answers": {"brand_name": "PocketBank", "goal": "Let young professionals manage money, send payments and track spending in one place", "pain_points": "too many apps and unclear spending",
               "target_users": "young professionals", "platform": "mobile", "features": "1, 3, 4", "auth_needed": "yes", "auth_methods": "biometrics and PIN",
               "payment_needed": "yes", "payment_methods": "bank transfer, UPI, cards", "style": "minimal", "brand_colors": "#0B3D91", "accessibility_notes": "none",
               "languages": "English", "inspiration": "Revolut, Monzo", "constraints": "must meet local financial regulations", "success": "40 percent monthly active users", "pages_confirm": "ok"},
   "expect": {"domain": "banking", "platform": "mobile", "pages_contains": ["Transfer", "Transactions", "Cards"], "report_contains": ["PocketBank", "#0B3D91"],
              "master_contains": ["PocketBank", "biometrics and PIN", "#0B3D91"]}},
 "minimal": {"first": "I want to build something for my business",
   "answers": {"domain": "an online shop", "goal": "skip general", "pages_confirm": "ok"}, "skip_pages": True,
   "expect": {"domain": "ecommerce", "pages_contains": ["Checkout"], "report_contains": ["assumed"], "master_contains": ["Assumptions"]}},
}

FALLBACK = {"focus": "Find what they need and start the main task in under a minute", "first_time": "What the product does and how to start", "extras": "Sign-up and forgot password",
            "card": "Photo, name, rating, price and one clear action", "promo": "Yes, promo codes with a visible discount line and no minimum order", "methods": "UPI, cards and cash on delivery",
            "live": "A live map and the delivery partner's phone number", "mode": "Both video and in-clinic", "enroll": "Free enrolment; paid courses go through checkout",
            "access": "Only the patient and approved doctors, with explicit consent", "fallback": "Switch to chat or phone, or reschedule"}
def auto_answer(qtext, yn, key):
    if yn: return "Yes"
    m = re.search(r"\(([^)]+)\)", qtext)
    if m:
        items = [re.sub(r"^or\s+", "", re.sub(r"^(for example|e\.g\.)\s+", "", x.strip(), flags=re.I)) for x in m.group(1).split(",") if x.strip()]
        if items: s = ", ".join(items[:3]); return s[0].upper() + s[1:]
    return FALLBACK.get(key, "Keep it simple and consistent with the rest of the product")

def run_case(name, case):
    p = rp_new_project(); rp_chat(p, case["first"])
    ans = {k: (list(v) if isinstance(v, list) else v) for k, v in case["answers"].items()}
    for _ in range(300):
        qid = p.pending
        if qid is None: break
        if qid in ans:
            v = ans[qid]; a = (v.pop(0) if len(v) > 1 else v[0]) if isinstance(v, list) else v
        elif qid.startswith("pg::"):
            page, key = _split_pq(qid); q = next(x for x in page_questions(page) if x[0] == key)
            a = "skip page" if case.get("skip_pages") else auto_answer(q[1], q[2] == YN, key)
        else: a = "skip"
        rp_chat(p, a)
    summary = build_summary_md(p); md, docx_path, _ = make_report_files(p); pack, pchecks = rp_build_pack(p, True)
    transcript = "\n\n".join((f"**You:** {u}\n\n" if u else "") + f"**Assistant:** {b}" for u, b in p.histories["requirements"])
    cdir = os.path.join(OUT_DIR, name); os.makedirs(cdir, exist_ok=True)
    for fn, txt in (("1_transcript.md", transcript), ("2_summary.md", summary), ("3_report.md", md)): open(os.path.join(cdir, fn), "w", encoding="utf-8").write(txt)
    shutil.copy(docx_path, os.path.join(cdir, "3_report.docx")); export_rp_prompts(p, out_dir=os.path.join(cdir, "4_prompts"))
    return {"p": p, "summary": summary, "report_md": md, "pack": pack, "pchecks": pchecks, "transcript": transcript}

def check_case(case, res):
    p, st, md, pack, e = res["p"], res["p"].state, res["report_md"], res["pack"], case["expect"]; rows = []
    chk = lambda name, ok: rows.append((name, bool(ok)))
    chk("interview finished", p.stage_done.get("requirements") and p.pending is None); chk(f"domain is {e['domain']}", st["domain"] == e["domain"])
    if "platform" in e: chk(f"platform is {e['platform']}", st["platform"] == e["platform"])
    for pg in e.get("pages_contains", []): chk(f"page '{pg}' present", pg in p.pages)
    for pg in e.get("pages_absent", []): chk(f"page '{pg}' removed", pg not in p.pages)
    for s in e.get("report_contains", []): chk(f"report mentions '{s}'", s in md)
    for s in e.get("master_contains", []): chk(f"master prompt mentions '{s}'", s in pack[0]["text"])
    chk("one prompt per page", all(any(it["page"] == pg for it in pack) for pg in p.pages))
    chk("authentication step only if login is needed or assumed", any(it["title"] == "Authentication" for it in pack) == (st["auth_needed"] is not False))
    chk("payments step only if payments are needed or assumed", any(it["title"] == "Payments" for it in pack) == (st["payment_needed"] is not False))
    by_page = {it["page"]: it["text"] for it in pack if it["page"]}
    chk("page answers appear in the page prompts", all(a in by_page[pg] for pg, dct in st.get("page_answers", {}).items() if pg in by_page for a in dct.values()))
    chk("all structural prompt checks pass", bool(res["pchecks"].passed.all()))
    return rows

def show_case(name, case, res):
    display(Markdown(f"---\n# ▶ Test case: {name}\n*First message:* `{case['first']}`"))
    if SHOW_TRANSCRIPT: display(Markdown("## 1. Interview transcript\n\n" + res["transcript"]))
    if SHOW_SUMMARY: display(Markdown("## 2. Project summary\n\n" + res["summary"]))
    if SHOW_REPORT: display(Markdown("## 3. Detailed report\n\n" + res["report_md"]))
    if SHOW_PROMPTS != "none":
        pack = res["pack"]; display(Markdown("## 4. Development prompts\n\n" + "\n".join(f"- **{it['id']}** {it['title']} (about {it['tokens']} tokens)" for it in pack)))
        if SHOW_PROMPTS in ("master", "all"):
            for it in (pack if SHOW_PROMPTS == "all" else [pack[0], next(x for x in pack if x["kind"] == "page")]):
                display(Markdown(f"**Step {it['id']}: {it['title']}**\n\n```\n{it['text']}\n```"))

shutil.rmtree(OUT_DIR, ignore_errors=True); os.makedirs(OUT_DIR)
table, failures = [], []
for name in RUN_CASES:
    case = TEST_CASES[name]; res = run_case(name, case); checks = check_case(case, res)
    if name in SHOW_DETAILS_FOR: show_case(name, case, res)
    p = res["p"]; gd, gt, pd_, pt = iq_counts(p); ok = sum(1 for _, o in checks if o)
    table.append({"case": name, "domain": p.state["domain"], "pages": len(p.pages), "features": len(p.state.get("features", [])), "general answers": f"{gd}/{gt}", "page answers": f"{pd_}/{pt}",
                  "report words": len(res["report_md"].split()), "prompts": len(res["pack"]), "checks": f"{ok}/{len(checks)}", "result": "✅ pass" if ok == len(checks) else "❌ fail"})
    failures += [(name, c) for c, o in checks if not o]
display(Markdown("---\n# Test results"))
df = pd.DataFrame(table); display(df); df.to_csv(os.path.join(OUT_DIR, "results.csv"), index=False)
z = shutil.make_archive("/content/test_outputs", "zip", OUT_DIR)
print("All outputs (transcript, summary, report .md/.docx, prompts) saved in", OUT_DIR, "->", z)
try:
    from google.colab import files; files.download(z)
except Exception:
    pass
if failures:
    print("\n❌ Failed checks:"); [print(f"  - [{n}] {c}") for n, c in failures]
assert not failures, "Some checks failed (listed above). Send me the list and I will fix them."
print("\n✅ All test cases passed.")

---
# ▶ Test case: food
*First message:* `I want to build a food delivery app for college students`

## 1. Interview transcript

**Assistant:** Hi! Tell me about the website or app you want to build, in your own words. I'll ask some **general questions** first, then a few questions about **each page**, and then write a **detailed report** you can download. I skip anything you already told me.

You can **pause at any time** with the Pause button and continue later.

**You:** I want to build a food delivery app for college students

**Assistant:** Noted so far: product: Food ordering, users: college students.

**Question 2 of 17.** What is the **name** of your product, or a working title? (or type **skip**)

**You:** CampusBite

**Assistant:** **Question 3 of 17.** What is the **main goal** of your product, or the problem it solves for users? One or two sentences is enough (or **skip**).

**You:** Help students order cheap meals from campus canteens without queues

**Assistant:** **Question 5 of 17.** What **problems or frustrations** do your users have today that this product should fix? (or **skip**)

**You:** long queues and no idea what is available

**Assistant:** **Question 6 of 17.** Which platform do you want: **mobile app**, **website**, or **both**?

**You:** mobile

**Assistant:** **Question 7 of 17.** Which **features** matter most? Reply with numbers (for example 1, 3, 5), feature names, **recommended** (I pick a core set), or **skip**.

1. **Search & filters**: search restaurants and dishes, filter by cuisine, price and rating
2. **Menu browsing**: browse menus with photos, categories and prices
3. **Cart & customization**: add items to cart, customize toppings, edit quantities
4. **Online payment**: pay online with cards, UPI, wallets or cash on delivery
5. **Live order tracking**: track order status and delivery partner on a live map
6. **Ratings & reviews**: rate and review restaurants and dishes
7. **Saved addresses & favorites**: save delivery addresses and favorite restaurants
8. **Offers & promo codes**: apply coupons, discounts and promotional offers
9. **Reorder**: quickly reorder previous orders

**You:** 1, 4, 5

**Assistant:** **Question 8 of 17.** Will users need to create an account and log in? (yes / no)

**You:** yes

**Assistant:** **Question 9 of 18.** How should people **sign in**? (email and password, phone and OTP, Google or social login, other)

**You:** phone and OTP, Google sign-in

**Assistant:** **Question 10 of 18.** Will the product handle payments? (yes / no)

**You:** yes

**Assistant:** **Question 11 of 19.** Which **payment methods** should be supported? (cards, UPI or wallets, cash on delivery, bank transfer, other)

**You:** UPI, cards

**Assistant:** **Question 12 of 19.** Which visual style fits best: **minimal, modern, playful, professional, dark, or elegant**?

**You:** playful

**Assistant:** **Question 13 of 19.** Do you have **brand colors**? Name them or give hex codes (for example #1E88E5), or type **none**.

**You:** orange #FF7A00

**Assistant:** **Question 14 of 19.** Any **accessibility needs** to design for? (screen readers, large text, color-blind friendly, older users, none)

**You:** large text

**Assistant:** **Question 15 of 19.** Which **languages** should the product support?

**You:** English and Hindi

**Assistant:** **Question 16 of 19.** Name **1 to 3 apps or websites** you like, or that compete with yours (or **skip**).

**You:** Zomato, Swiggy

**Assistant:** **Question 17 of 19.** Any **constraints**? For example budget, timeline, technology or legal rules. (Type **none** to continue.)

**You:** none

**Assistant:** **Question 18 of 19.** How will you know the product is **successful** after 3 months? (for example sign-ups, orders, completed lessons)

**You:** 500 weekly orders

**Assistant:** **Question 19 of 19.** Here are the pages I recommend: Home, Login, Restaurant List, Menu, Cart, Checkout, Order Tracking, Profile.

I will ask a few questions about **each page**. Reply **ok** to continue, or change the list with `add <page>` or `remove <page>`.

**You:** remove profile

**Assistant:** Updated.

**Question 19 of 19.** Here are the pages I recommend: Home, Login, Restaurant List, Menu, Cart, Checkout, Order Tracking.

I will ask a few questions about **each page**. Reply **ok** to continue, or change the list with `add <page>` or `remove <page>`.

**You:** ok

**Assistant:** Great! Now a few questions about **each page**. Type `skip` to skip one question, or `skip page` to skip the rest of a page.

**Page 1 of 7: Home** (question 1 of 3). What is the **single most important thing** a visitor should do on the Home page?

**You:** Find what they need and start the main task in under a minute

**Assistant:** **Page 1 of 7: Home** (question 2 of 3). What should be **featured** on Home? (for example popular items, offers, search, categories)

**You:** Popular items, offers, search

**Assistant:** **Page 1 of 7: Home** (question 3 of 3). What should a first-time visitor understand within 5 seconds?

**You:** What the product does and how to start

**Assistant:** **Page 2 of 7: Login** (question 1 of 2). What should the login page **ask for**? (email, phone, password, OTP, social sign-in)

**You:** Email, phone, password

**Assistant:** **Page 2 of 7: Login** (question 2 of 2). Do you need **sign-up, forgot password, or continue as guest** on this page?

**You:** Sign-up and forgot password

**Assistant:** **Page 3 of 7: Restaurant List** (question 1 of 2). Which **filters and sorting** do users need? (cuisine, price, rating, delivery time, veg only)

**You:** Cuisine, price, rating

**Assistant:** **Page 3 of 7: Restaurant List** (question 2 of 2). What should each **restaurant card** show?

**You:** Photo, name, rating, price and one clear action

**Assistant:** **Page 4 of 7: Menu** (question 1 of 3). How should **dishes be organised**? (categories, photos, dietary tags)

**You:** Categories, photos, dietary tags

**Assistant:** **Page 4 of 7: Menu** (question 2 of 3). Do dishes have **options** such as size, toppings or add-ons?

**You:** Yes

**Assistant:** **Page 4 of 7: Menu** (question 3 of 3). Any **ordering rules**? (minimum order, one restaurant per order, availability times)

**You:** Minimum order, one restaurant per order, availability times

**Assistant:** **Page 5 of 7: Cart** (question 1 of 2). What can users **change in the cart**? (quantity, notes, remove items, save for later)

**You:** Quantity, notes, remove items

**Assistant:** **Page 5 of 7: Cart** (question 2 of 2). Do you need **promo codes or discounts**? Any fee or minimum-order rules?

**You:** Yes, promo codes with a visible discount line and no minimum order

**Assistant:** **Page 6 of 7: Checkout** (question 1 of 3). What **details** do you need at checkout? (address, phone, delivery time, instructions)

**You:** Address, phone, delivery time

**Assistant:** **Page 6 of 7: Checkout** (question 2 of 3). Which **payment methods** should checkout support?

**You:** UPI, cards and cash on delivery

**Assistant:** **Page 6 of 7: Checkout** (question 3 of 3). Can guests check out **without an account**?

**You:** Yes

**Assistant:** **Page 7 of 7: Order Tracking** (question 1 of 2). Which **order statuses** should customers see? (confirmed, preparing, out for delivery, delivered)

**You:** Confirmed, preparing, out for delivery

**Assistant:** **Page 7 of 7: Order Tracking** (question 2 of 2). Do you need a **live map**, delivery partner details or a call button?

**You:** A live map and the delivery partner's phone number

**Assistant:** ✅ All questions answered. **Your detailed report is ready.**

- A mobile app for food ordering, built for college students
- 7 pages with page-by-page requirements (17 of 17 page questions answered)
- General answers: 19 of 19
- UX guidance with sources, accessibility checks and a visual design direction

Open the **Report** tab to read it and download it as Word or Markdown. To change an answer, type for example `change style` or `change Checkout`, then `report` again.

Also ready: a **project summary** (type `summary`) and **development prompts** for building this product (type `prompts`, or open the **Build prompts** tab).

## 2. Project summary

# Project summary: CampusBite

**A mobile app for food ordering, built for college students.**

## Key decisions
- **Goal:** Help students order cheap meals from campus canteens without queues
- **Users:** college students
- **Platform:** mobile
- **Login:** yes (phone and OTP, Google sign-in)
- **Payments:** yes (UPI, cards)
- **Style:** playful, primary color #FF7A00
- **Languages:** English and Hindi

## Scope
- **7 pages:** Home, Login, Restaurant List, Menu, Cart, Checkout, Order Tracking
- **4 features:** Search & filters, Online payment, Live order tracking, Reorder

## Page questions coverage
- Home: 3/3 answered
- Login: 2/2 answered
- Restaurant List: 2/2 answered
- Menu: 3/3 answered
- Cart: 2/2 answered
- Checkout: 3/3 answered
- Order Tracking: 2/2 answered

## Top UX priorities
- Show delivery time and fees early, before the cart.
- Keep the add-to-cart action one tap from the dish card.
- Show real-time order status to reduce anxiety.

## Watch out for
- Order accuracy and delivery-time promises: show an ETA and clear cancellation rules
- Payment failures and duplicate orders

**Interview coverage:** general 19/19, pages 17/17

**Next step:** read the detailed report, confirm the assumptions, then give the development prompts to your builder one step at a time.

## 3. Detailed report

# CampusBite

*Detailed UX report*

## 1. Summary

**Project:** CampusBite  |  **Date:** 21 November 2026  |  **Status:** complete

A mobile app for food ordering, built for college students.

**Your idea, in your words:** I want to build a food delivery app for college students

This report turns your answers into a structured UX analysis: who the product is for, which pages and features it needs, what each page must contain, UX guidance with sources, accessibility checks and a visual design direction.

## 2. Your requirements

| Topic | Answer | Status |
|---|---|---|
| Product type | Food ordering | answered |
| Product name | CampusBite | answered |
| Main goal | Help students order cheap meals from campus canteens without queues | answered |
| Target users | college students | answered |
| User problems | long queues and no idea what is available | answered |
| Platform | mobile | answered |
| Login needed | yes | answered |
| Sign-in methods | phone and OTP, Google sign-in | answered |
| Payments needed | yes | answered |
| Payment methods | UPI, cards | answered |
| Key features | Search & filters, Online payment, Live order tracking, Reorder | answered |
| Visual style | playful | answered |
| Brand colors | orange #FF7A00 | answered |
| Accessibility needs | large text | answered |
| Languages | English and Hindi | answered |
| Inspiration and competitors | Zomato, Swiggy | answered |
| Constraints | none | answered |
| Success measure | 500 weekly orders | answered |

## 3. Assumptions and open questions

No assumptions were needed for the core decisions.

**Questions to confirm before design starts**

- Who exactly are your first 10 users, and can you talk to them?
- Which features are must-have for the first release and which can wait?
- Risk to check: Order accuracy and delivery-time promises: show an ETA and clear cancellation rules

## 4. Users, problems and core journey

**Target users:** college students.  **Goal:** Help students order cheap meals from campus canteens without queues.

**Problems to solve:** long queues and no idea what is available

**Inspiration and competitors to study:** Zomato, Swiggy

**Core user journey** (design every step so it takes as few taps as possible):

1. Open app
2. Find a restaurant or dish
3. View menu
4. Add items to cart
5. Checkout and pay
6. Track delivery
7. Rate the order

## 5. Pages (information architecture)

| Page | Purpose | Key components |
|---|---|---|
| Home | Entry point that gets users to the main task in one or two taps | Top navigation bar; Search bar; Hero / featured section; Category shortcuts; Recommended items; Footer / bottom tab bar |
| Login | Authenticate returning users and link to account creation | Logo; Email / phone field; Password / OTP field; Primary login button; Social login options; Sign-up link |
| Restaurant List | Browse and filter restaurants | Search & filter bar; Restaurant cards (photo, rating, ETA); Sort dropdown; Infinite scroll |
| Menu | Show one restaurant's menu and let users add dishes | Restaurant header (rating, ETA); Category tabs; Dish cards with photo and price; Add-to-cart button; Sticky cart summary |
| Cart | Review and edit the order before paying | Item list with quantity steppers; Promo code field; Price breakdown; Checkout button |
| Checkout | Collect address and payment details, then place the order | Address selector; Payment method selector; Order summary; Place order button |
| Order Tracking | Show live order progress | Order status timeline; Live map; Delivery partner info; Help / support button |

## 6. Page-by-page requirements

Your answers for each page. Items marked **open** were skipped or not asked yet.

### Home

**Recommended components:** Top navigation bar; Search bar; Hero / featured section; Category shortcuts; Recommended items; Footer / bottom tab bar

| Question | Your answer |
|---|---|
| What is the single most important thing a visitor should do on the Home page? | Find what they need and start the main task in under a minute |
| What should be featured on Home? (for example popular items, offers, search, categories) | Popular items, offers, search |
| What should a first-time visitor understand within 5 seconds? | What the product does and how to start |

### Login

**Recommended components:** Logo; Email / phone field; Password / OTP field; Primary login button; Social login options; Sign-up link

| Question | Your answer |
|---|---|
| What should the login page ask for? (email, phone, password, OTP, social sign-in) | Email, phone, password |
| Do you need sign-up, forgot password, or continue as guest on this page? | Sign-up and forgot password |

### Restaurant List

**Recommended components:** Search & filter bar; Restaurant cards (photo, rating, ETA); Sort dropdown; Infinite scroll

| Question | Your answer |
|---|---|
| Which filters and sorting do users need? (cuisine, price, rating, delivery time, veg only) | Cuisine, price, rating |
| What should each restaurant card show? | Photo, name, rating, price and one clear action |

### Menu

**Recommended components:** Restaurant header (rating, ETA); Category tabs; Dish cards with photo and price; Add-to-cart button; Sticky cart summary

| Question | Your answer |
|---|---|
| How should dishes be organised? (categories, photos, dietary tags) | Categories, photos, dietary tags |
| Do dishes have options such as size, toppings or add-ons? | Yes |
| Any ordering rules? (minimum order, one restaurant per order, availability times) | Minimum order, one restaurant per order, availability times |

### Cart

**Recommended components:** Item list with quantity steppers; Promo code field; Price breakdown; Checkout button

| Question | Your answer |
|---|---|
| What can users change in the cart? (quantity, notes, remove items, save for later) | Quantity, notes, remove items |
| Do you need promo codes or discounts? Any fee or minimum-order rules? | Yes, promo codes with a visible discount line and no minimum order |

### Checkout

**Recommended components:** Address selector; Payment method selector; Order summary; Place order button

| Question | Your answer |
|---|---|
| What details do you need at checkout? (address, phone, delivery time, instructions) | Address, phone, delivery time |
| Which payment methods should checkout support? | UPI, cards and cash on delivery |
| Can guests check out without an account? | Yes |

### Order Tracking

**Recommended components:** Order status timeline; Live map; Delivery partner info; Help / support button

| Question | Your answer |
|---|---|
| Which order statuses should customers see? (confirmed, preparing, out for delivery, delivered) | Confirmed, preparing, out for delivery |
| Do you need a live map, delivery partner details or a call button? | A live map and the delivery partner's phone number |

## 7. Features

| Feature | What it means | Priority |
|---|---|---|
| Search & filters | search restaurants and dishes, filter by cuisine, price and rating | Must have (you selected it) |
| Online payment | pay online with cards, UPI, wallets or cash on delivery | Must have (you selected it) |
| Live order tracking | track order status and delivery partner on a live map | Must have (you selected it) |
| Reorder | quickly reorder previous orders | Must have (you selected it) |

**Recommended additions to consider after the first release:**

- Menu browsing: browse menus with photos, categories and prices
- Cart & customization: add items to cart, customize toppings, edit quantities
- Ratings & reviews: rate and review restaurants and dishes
- Saved addresses & favorites: save delivery addresses and favorite restaurants

## 8. UX guidance with sources

Each guideline comes from the knowledge base (usability heuristics, WCAG, platform and domain patterns). The last column shows the pages it matters for.

| Guideline | What to do | Source | Applies to |
|---|---|---|---|
| Cart and fees | Keep a sticky cart summary visible and show delivery fee, taxes and estimated time before checkout to avoid surprises. | Food delivery UX patterns | Menu, Cart, Checkout, Order Tracking |
| Menu and dish cards | Show dish photos, price, dietary tags and ratings on the card, with a quick add button and a customization sheet for options. | Food delivery UX patterns | Login, Restaurant List, Menu, Cart |
| Order tracking | Show a status timeline (confirmed, preparing, on the way, delivered), a live map, ETA updates and a contact or help option. | Food delivery UX patterns | Menu, Checkout, Order Tracking |
| Reorder and saved addresses | Offer one-tap reorder, favorites and saved addresses to make repeat orders fast. | Food delivery UX patterns | Restaurant List, Checkout |
| Shortcuts for experienced users | Offer accelerators such as saved items, search, favorites and quick actions for frequent users. | Nielsen heuristic 7 | Home, Restaurant List |
| Login screen | Offer a show-password toggle, forgot-password link, OTP autofill and optional biometric login, with clear error messages. | Authentication UX practice | Login |
| Mobile navigation | Use a bottom tab bar with 3 to 5 top-level destinations so the main actions sit within thumb reach. | Material Design / Apple HIG | Home |
| Shopping cart | Allow editing quantities and removing items in the cart, keep the cart saved, and show shipping estimates and a promo code field. | E-commerce UX patterns | Cart |
| Show system status | Always tell users what is happening: loading indicators, progress steps, and confirmation messages after actions. | Nielsen heuristic 1 | Order Tracking |
| Sign-up and onboarding | Delay registration until it is needed, keep sign-up short, offer social or OTP login, and explain value before asking for data. | Onboarding practice | Login |
| Web navigation | Use a clear top navigation with a small number of items, a visible search, and breadcrumbs for deep hierarchies. | Common web patterns | Home |

## 9. Accessibility and inclusive design

You mentioned accessibility needs (large text): treat **WCAG 2.2 level AA** as a requirement, not a goal.

- Show system status: Always tell users what is happening: loading indicators, progress steps, and confirmation messages after actions. (Nielsen heuristic 1)
- Text contrast: Meet AA contrast: at least 4.5:1 for normal text and 3:1 for large text and UI components such as buttons and input borders. (WCAG 2.2 (1.4.3, 1.4.11))
- Keyboard access and focus: All functions must work with a keyboard, with a clearly visible focus indicator and a logical tab order. (WCAG 2.2 (2.1.1, 2.4.7))
- Labels and alt text: Give every form field a visible label and every meaningful image a text alternative so screen readers can describe it. (WCAG 2.2 (1.1.1, 3.3.2))
- Do not rely on color alone: Never use color as the only way to show status or errors; add icons, text or patterns as well. (WCAG 2.2 (1.4.1))
- Touch target size: Make tap targets at least 44x44 pt (Apple) or 48x48 dp (Material) with enough spacing between them to avoid mis-taps. (Apple HIG / Material Design)

**Checks to run before launch:** complete the core journey with the keyboard only; test with a screen reader; zoom text to 200 percent; check every color pair in the next section.

## 10. Visual design direction

**Style:** playful. Bright colors, friendly illustrations, rounded shapes.  **Fonts:** Nunito (headings + body).

- Primary: #FF7A00
- Accent: #FFD93D
- Bg: #FFF9F0
- Surface: #FFFFFF
- Text: #2D2A32

| Color pair | Colors | Contrast | Needed (WCAG AA) | Result |
|---|---|---|---|---|
| Body text on background | #2D2A32 on #FFF9F0 | 13.5:1 | 4.5:1 | Pass |
| Body text on cards | #2D2A32 on #FFFFFF | 14.1:1 | 4.5:1 | Pass |
| Button text on primary | #111111 on #FF7A00 | 7.2:1 | 4.5:1 | Pass |
| Text on accent | #111111 on #FFD93D | 13.7:1 | 4.5:1 | Pass |
| Primary color as a control or icon on background | #FF7A00 on #FFF9F0 | 2.5:1 | 3.0:1 | Check |

## 11. Platform considerations

- Bottom tab bar with 3 to 5 destinations
- Touch targets of at least 44 pt (Apple) or 48 dp (Material)
- Design for 360 to 430 px wide screens with thumb-reachable primary actions
- Plan for slow or offline networks

## 12. Data to plan for

The information your product will store (a starting point for the database design):

- User(id, name, email, phone)
- Address(id, user_id, label, line1, city, postal_code)
- Restaurant(id, name, cuisines, rating, eta_minutes, delivery_fee, is_open, image)
- MenuItem(id, restaurant_id, name, description, price, category, dietary_tags, options, available)
- CartItem(menu_item_id, quantity, selected_options)
- Order(id, user_id, restaurant_id, items, status, total, address_id, payment_method, created_at)
- Review(id, order_id, rating, comment)

## 13. Risks and domain priorities

- Order accuracy and delivery-time promises: show an ETA and clear cancellation rules
- Payment failures and duplicate orders
- Allergen and dietary information must be accurate and easy to find

**UX priorities for this kind of product:**

- Show delivery time and fees early, before the cart.
- Keep the add-to-cart action one tap from the dish card.
- Show real-time order status to reduce anxiety.
- Make reordering and saved addresses effortless.

## 14. Next steps

1. Review this report and confirm the assumptions.
2. Talk to 3 to 5 potential users and check that the core journey matches how they work.
3. Sketch wireframes for the pages, using the page-by-page answers.
4. Build a clickable prototype and test the core journey with real people.
5. Implement in phases: setup, design system, data, pages, quality checks, deployment.
6. Re-check accessibility before launch.

## Appendix A. Conversation record

- **Assistant:** Hi! Tell me about the website or app you want to build, in your own words. I'll ask some general questions first, then a few questions about each page, and then
- **You:** I want to build a food delivery app for college students
- **Assistant:** Noted so far: product: Food ordering, users: college students. Question 2 of 17. What is the name of your product, or a working title? (or type skip)
- **You:** CampusBite
- **Assistant:** Question 3 of 17. What is the main goal of your product, or the problem it solves for users? One or two sentences is enough (or skip).
- **You:** Help students order cheap meals from campus canteens without queues
- **Assistant:** Question 5 of 17. What problems or frustrations do your users have today that this product should fix? (or skip)
- **You:** long queues and no idea what is available
- **Assistant:** Question 6 of 17. Which platform do you want: mobile app, website, or both?
- **You:** mobile
- **Assistant:** Question 7 of 17. Which features matter most? Reply with numbers (for example 1, 3, 5), feature names, recommended (I pick a core set), or skip. 1. Search & fil
- **You:** 1, 4, 5
- **Assistant:** Question 8 of 17. Will users need to create an account and log in? (yes / no)
- **You:** yes
- **Assistant:** Question 9 of 18. How should people sign in? (email and password, phone and OTP, Google or social login, other)
- **You:** phone and OTP, Google sign-in
- **Assistant:** Question 10 of 18. Will the product handle payments? (yes / no)
- **You:** yes
- **Assistant:** Question 11 of 19. Which payment methods should be supported? (cards, UPI or wallets, cash on delivery, bank transfer, other)
- **You:** UPI, cards
- **Assistant:** Question 12 of 19. Which visual style fits best: minimal, modern, playful, professional, dark, or elegant?
- **You:** playful
- **Assistant:** Question 13 of 19. Do you have brand colors? Name them or give hex codes (for example #1E88E5), or type none.
- **You:** orange #FF7A00
- **Assistant:** Question 14 of 19. Any accessibility needs to design for? (screen readers, large text, color-blind friendly, older users, none)
- **You:** large text
- **Assistant:** Question 15 of 19. Which languages should the product support?
- **You:** English and Hindi
- **Assistant:** Question 16 of 19. Name 1 to 3 apps or websites you like, or that compete with yours (or skip).
- **You:** Zomato, Swiggy
- **Assistant:** Question 17 of 19. Any constraints? For example budget, timeline, technology or legal rules. (Type none to continue.)
- **You:** none
- **Assistant:** Question 18 of 19. How will you know the product is successful after 3 months? (for example sign-ups, orders, completed lessons)
- **You:** 500 weekly orders
- **Assistant:** Question 19 of 19. Here are the pages I recommend: Home, Login, Restaurant List, Menu, Cart, Checkout, Order Tracking, Profile. I will ask a few questions about
- **You:** remove profile
- **Assistant:** Updated. Question 19 of 19. Here are the pages I recommend: Home, Login, Restaurant List, Menu, Cart, Checkout, Order Tracking. I will ask a few questions about
- **You:** ok
- **Assistant:** Great! Now a few questions about each page. Type skip to skip one question, or skip page to skip the rest of a page. Page 1 of 7: Home (question 1 of 3). What i
- **You:** Find what they need and start the main task in under a minute
- **Assistant:** Page 1 of 7: Home (question 2 of 3). What should be featured on Home? (for example popular items, offers, search, categories)
- **You:** Popular items, offers, search
- **Assistant:** Page 1 of 7: Home (question 3 of 3). What should a first-time visitor understand within 5 seconds?
- **You:** What the product does and how to start
- **Assistant:** Page 2 of 7: Login (question 1 of 2). What should the login page ask for? (email, phone, password, OTP, social sign-in)
- **You:** Email, phone, password
- **Assistant:** Page 2 of 7: Login (question 2 of 2). Do you need sign-up, forgot password, or continue as guest on this page?
- **You:** Sign-up and forgot password
- **Assistant:** Page 3 of 7: Restaurant List (question 1 of 2). Which filters and sorting do users need? (cuisine, price, rating, delivery time, veg only)
- **You:** Cuisine, price, rating
- **Assistant:** Page 3 of 7: Restaurant List (question 2 of 2). What should each restaurant card show?
- **You:** Photo, name, rating, price and one clear action
- **Assistant:** Page 4 of 7: Menu (question 1 of 3). How should dishes be organised? (categories, photos, dietary tags)
- **You:** Categories, photos, dietary tags
- **Assistant:** Page 4 of 7: Menu (question 2 of 3). Do dishes have options such as size, toppings or add-ons?
- **You:** Yes
- **Assistant:** Page 4 of 7: Menu (question 3 of 3). Any ordering rules? (minimum order, one restaurant per order, availability times)
- **You:** Minimum order, one restaurant per order, availability times
- **Assistant:** Page 5 of 7: Cart (question 1 of 2). What can users change in the cart? (quantity, notes, remove items, save for later)
- **You:** Quantity, notes, remove items
- **Assistant:** Page 5 of 7: Cart (question 2 of 2). Do you need promo codes or discounts? Any fee or minimum-order rules?
- **You:** Yes, promo codes with a visible discount line and no minimum order
- **Assistant:** Page 6 of 7: Checkout (question 1 of 3). What details do you need at checkout? (address, phone, delivery time, instructions)
- **You:** Address, phone, delivery time
- **Assistant:** Page 6 of 7: Checkout (question 2 of 3). Which payment methods should checkout support?
- **You:** UPI, cards and cash on delivery
- **Assistant:** Page 6 of 7: Checkout (question 3 of 3). Can guests check out without an account?
- **You:** Yes
- **Assistant:** Page 7 of 7: Order Tracking (question 1 of 2). Which order statuses should customers see? (confirmed, preparing, out for delivery, delivered)
- **You:** Confirmed, preparing, out for delivery
- **Assistant:** Page 7 of 7: Order Tracking (question 2 of 2). Do you need a live map, delivery partner details or a call button?
- **You:** A live map and the delivery partner's phone number
- **Assistant:** ✅ All questions answered. Your detailed report is ready. - A mobile app for food ordering, built for college students - 7 pages with page-by-page requirements (

## Appendix B. How this report was produced

The report was produced with rules and retrieval, without a language model: text understanding with spaCy and sentence embeddings, a zero-shot product-type classifier, a question flow that asks only for missing information (general questions, then questions for each page), and a UX guideline base (Nielsen heuristics, WCAG, platform and domain patterns) with a source for every guideline. Treat it as a well-grounded starting point and have a designer review it.


## 4. Development prompts

- **00** Master project brief (about 1239 tokens)
- **01** Project setup and architecture (about 540 tokens)
- **02** Design system and app shell (about 585 tokens)
- **03** Data models and mock API (about 567 tokens)
- **04** Authentication (about 492 tokens)
- **05** Page: Home (about 1513 tokens)
- **06** Page: Login (about 1418 tokens)
- **07** Page: Restaurant List (about 1345 tokens)
- **08** Page: Menu (about 1436 tokens)
- **09** Page: Cart (about 1370 tokens)
- **10** Page: Checkout (about 1391 tokens)
- **11** Page: Order Tracking (about 1393 tokens)
- **12** Payments (about 474 tokens)
- **13** Cross-cutting features (about 485 tokens)
- **14** Accessibility, QA and performance (about 537 tokens)
- **15** Deployment and documentation (about 438 tokens)

**Step 00: Master project brief**

```
# Master project brief: CampusBite
> Save this as CLAUDE.md or AGENTS.md (or paste it first in a new chat). Every later step assumes it.

## 1. Product
A mobile app for food ordering, built for college students.
- Login required: yes
- Payments: yes
- Accessibility priority: high (target WCAG 2.2 AA)

## 2. Users and core journey
- Users: college students
- Core journey: Open app -> Find a restaurant or dish -> View menu -> Add items to cart -> Checkout and pay -> Track delivery -> Rate the order

## 3. Scope
Pages:
- Home (`/`): Entry point that gets users to the main task in one or two taps
- Login (`/login`): Authenticate returning users and link to account creation
- Restaurant List (`/restaurants`): Browse and filter restaurants
- Menu (`/restaurants/:id`): Show one restaurant's menu and let users add dishes
- Cart (`/cart`): Review and edit the order before paying
- Checkout (`/checkout`): Collect address and payment details, then place the order
- Order Tracking (`/orders/:id`): Show live order progress

Features:
- Search & filters: Debounced search plus filters (cuisine, brand, price, rating), sorting and result counts; filter state kept in the URL
- Online payment: PaymentService interface with a mock provider; success, failure and retry flows
- Live order tracking: Simulated real-time status updates (timers or a WebSocket stub), ETA and a map placeholder
- Reorder: One-tap reorder from order history into the cart

Out of scope: real payment processing, real SMS or email, real maps (use mocks behind interfaces); admin or back-office tools; features not listed above.

## 4. Tech stack
- Framework: React Native (Expo) + TypeScript
- Routing: Expo Router
- State and data: Zustand for client state and TanStack Query for server state
- Testing: Jest + React Native Testing Library, Maestro for end-to-end flows
- Mock data: MSW or a local mock service with seed data
- Accessibility tooling: accessibilityLabel/accessibilityRole props plus manual VoiceOver and TalkBack checks

## 5. Design system
- Style: playful. Bright colors, friendly illustrations, rounded shapes.
- Tokens:
    --color-primary: #FF7A00;  /* text on it: #111111 */
    --color-accent: #FFD93D;  /* text on it: #111111 */
    --color-bg: #FFF9F0;
    --color-surface: #FFFFFF;
    --color-text: #2D2A32;
    --font-heading: 'Nunito', 'Trebuchet MS', system-ui, sans-serif;
    --font-body: 'Nunito', 'Trebuchet MS', system-ui, sans-serif;
    --radius: 18px;
- Responsive: design for 360 to 430 px screens, bottom tab bar, thumb-reachable actions.

## 6. UX and accessibility rules (always apply)
- Show system status: Always tell users what is happening: loading indicators, progress steps, and confirmation messages after actions. (Nielsen heuristic 1)
- Text contrast: Meet AA contrast: at least 4.5:1 for normal text and 3:1 for large text and UI components such as buttons and input borders. (WCAG 2.2 (1.4.3, 1.4.11))
- Keyboard access and focus: All functions must work with a keyboard, with a clearly visible focus indicator and a logical tab order. (WCAG 2.2 (2.1.1, 2.4.7))
- Labels and alt text: Give every form field a visible label and every meaningful image a text alternative so screen readers can describe it. (WCAG 2.2 (1.1.1, 3.3.2))
- Do not rely on color alone: Never use color as the only way to show status or errors; add icons, text or patterns as well. (WCAG 2.2 (1.4.1))
- Primary call to action: Give each screen one clear primary action, with a verb label such as Place order, and style secondary actions less prominently. (Visual hierarchy practice)
- Empty, loading and error states: Design empty, loading and error states deliberately: explain what is happening and offer a next action. Skeleton screens improve perceived speed. (Interaction design practice)
- Touch target size: Make tap targets at least 44x44 pt (Apple) or 48x48 dp (Material) with enough spacing between them to avoid mis-taps. (Apple HIG / Material Design)

## 7. Engineering conventions
Folder structure:
    app/ (Expo Router screens, one per page)
    src/components/ui/ (design-system components)
    src/features/<feature>/
    src/lib/ (api client, utils)
    src/mocks/ (seed data)
    src/theme/ (tokens)
    src/types/
    __tests__/
- Strictest type-checking and lint settings available; format on commit
- Data only through repositories (never fetch directly in components)
- Every component handles loading, empty and error states
- Conventional commit messages; one logical change per commit

## 7b. Requirements gathered in the interview
- Product name: CampusBite
- Main goal: Help students order cheap meals from campus canteens without queues
- User problems to solve: long queues and no idea what is available
- Sign-in methods: phone and OTP, Google sign-in
- Payment methods: UPI, cards
- Brand colors: orange #FF7A00
- Accessibility needs: large text
- Languages: English and Hindi
- Inspiration and competitors: Zomato, Swiggy
- Success measure: 500 weekly orders

## 8. How you (the AI assistant) should work
- Work only on the step I give you; do not start other steps
- State your assumptions briefly; if something is truly ambiguous, ask one question, otherwise choose the simplest option and say so
- Make small, reviewable changes; list files created or changed
- Write tests with the code and report how to run them
- Never add dependencies without saying why

## 9. Definition of done (every step)
- Builds with no errors; type-check and lint pass
- Tests pass; accessibility checks pass for the changed screens
- Uses design tokens only
- Short notes added to the README when behavior or setup changes
```

... (3 outputs left out)

All outputs (transcript, summary, report .md/.docx, prompts) saved in /content/test_outputs -> /content/test_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All test cases passed.
